In [1]:
import __main__
import os
from typing import Dict, List, Literal, Tuple, Optional
import logging
import math
import gc
import pickle
import time
import sys

import json
from datetime import datetime
from pathlib import Path

import category_encoders as ce
from catboost import CatBoostRegressor
from matplotlib import pyplot as plt
from momentfm import MOMENTPipeline
import numpy as np
import pandas as pd
from tqdm import tqdm
from pycatch22 import catch22_all
from scipy.special import softmax
import xarray as xr

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(torch.cuda.memory_reserved(0) / 1e6, "MB reserved")
    print(torch.cuda.memory_allocated(0) / 1e6, "MB allocated")

from methods.forecasting_module import TimeGPTForecaster, SARIMAXForecaster

from utils.io_utils import JSONLogger, Notifiers, read_yaml_params, set_all_seeds
from utils.metrics_utils import AutocorrMetrics, Preds, Losses, DimensionalityEstimator, ForecastUtils, SemiSupLearning
from utils.data_utils import Slicing, Bootstrapping, assign_encoder_weights
from utils.model_utils import Decoder, ProjectionHead, TorchWrapper, profile_epoch

from preprocessing.dataset_preprocessors import DatasetPreprocessor, ECGLoader, GermanyDataset, WeatherDataset, process_argoverse_parquet

from encoders.lstm_network import LSTMModel, LSTMTrainer
import encoders.autoencoders as ae
import encoders.train_autoencoders as train_ae
from encoders.ts2vec_encoder import TS2VecEncoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.info("Starting process...")
logging.warning("Something looks off...")
logging.error("Something failed.")


/home/fouadabiad/miniconda3/envs/venv310/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
2025-11-15 06:30:06.841871: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-15 06:30:06.849513: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-15 06:30:06.858313: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registere

0.0 MB reserved
0.0 MB allocated


INFO:root:Starting process...
ERROR:root:Something failed.


Using device: cuda
Using device: cuda
cuda


In [2]:
"Setting up params"
interim_data_loc = "../interim_data"
public_data_loc  = "../public_datasets"
encoders_folder  = "other_encoders"
ts2vec_params_loc= f"ts2vec_params"

messager_params  = read_yaml_params("param_config/messager.yaml")
WEBHOOK_URL      = messager_params["webhook_url"]

# Default params path
params_path = "param_config/baseline_params.yaml"
params      = read_yaml_params(params_path)

# Override dataset if running from bash
dataset_from_env = os.getenv("DATASET")
if dataset_from_env is not None:
    params["basics"]["dataset"] = dataset_from_env
print("Dataset being used:", params["basics"]["desired_dataset"])
data_params = read_yaml_params("param_config/dataset_params.yaml")

# Optional: detect if running in notebook
running_in_notebook = not hasattr(__main__, "__file__")
print("Running in notebook:", running_in_notebook)
print("Params path:", params_path)

desired_dataset = params["basics"].get("dataset") or params["basics"]["desired_dataset"]
num_runs        = params["basics"]["num_runs"]
predictor_epochs= params["general_params"]["regressor"]["epochs_regressor"]
train_epochs    = data_params["general"]["train_epochs"]
NUM_PAGES_TO_USE= data_params["general"]["num_pages_to_use"]
WINDOWS_PER_PAGE= data_params[desired_dataset]["num_window_splits"]
NUM_ROWS        = data_params[desired_dataset]["num_rows_per_page"]

# dataset_window = data_params[desired_dataset]["window_len"]

if "WINDOW_LEN" in os.environ:
    dataset_window = int(os.environ["WINDOW_LEN"])
else:
    # dataset_window = int(data_params[desired_dataset]["window_len"])
    dataset_window = int(data_params["general"]["window_len"])
print("Using window length:", dataset_window)

# method-specific params
AE_lr       = params["cellsup"]["AE_lr"]
weight_decay= params["cellsup"]["weight_decay"]
dropout     = params["cellsup"]["dropout"]
swav_iters  = params["cellsup"]["swav_iters"]
swav_temp   = params["cellsup"]["swav_temp"]
cluster_min = params["cellsup"]["clustering"]["cluster_min"]
cluster_max = params["cellsup"]["clustering"]["cluster_max"]

layer1_dim  = params["general_params"]["regressor"]["layer1_dim"]
layer2_dim  = params["general_params"]["regressor"]["layer2_dim"]
layer3_dim  = params["general_params"]["regressor"]["layer3_dim"]

regressor_epochs = params["general_params"]["regressor"]["epochs_regressor"]
lr_regressor     = params["general_params"]["regressor"]["lr"]


def make_regression_head(embedding_dim: int, layer1_dim: int, layer2_dim: int, layer3_dim: int,
                         y_train_tensor: torch.Tensor, dropout: float, device: str):
    regression_head = nn.Sequential(
        nn.Linear(embedding_dim, layer1_dim),
        nn.SiLU(),
        nn.Dropout(dropout),
        nn.Linear(layer1_dim, layer2_dim),
        nn.SiLU(),
        nn.Dropout(dropout),
        nn.Linear(layer2_dim, layer3_dim),
        nn.SiLU(),
        nn.Dropout(dropout),
        nn.Linear(layer3_dim, y_train_tensor.shape[1])).to(device)
    return regression_head


def evaluate_regressor(regression_head, z_train_tensor, z_test_tensor, y_train_tensor,
                        y_test_tensor, regressor_epochs, lr_regressor):
    optimizer = torch.optim.AdamW(regression_head.parameters(), lr=lr_regressor)
    criterion = nn.MSELoss()

    if not torch.is_tensor(z_train_tensor):
        z_train_tensor = torch.tensor(z_train_tensor, dtype=torch.float32, device=device)
    if not torch.is_tensor(z_test_tensor):
        z_test_tensor = torch.tensor(z_test_tensor, dtype=torch.float32, device=device)
    if not torch.is_tensor(y_train_tensor):
        y_train_tensor = torch.tensor(y_train_tensor, dtype=torch.float32, device=device)
    if not torch.is_tensor(y_test_tensor):
        y_test_tensor = torch.tensor(y_test_tensor, dtype=torch.float32, device=device)

    for epoch in range(regressor_epochs):
        regression_head.train()
        optimizer.zero_grad()
        preds = regression_head(z_train_tensor)
        loss  = criterion(preds, y_train_tensor)
        loss.backward()
        optimizer.step()
        if epoch % 2 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")
    regression_head.eval()
    with torch.no_grad():
        test_preds = regression_head(z_test_tensor)
        test_loss  = torch.sqrt(criterion(test_preds, y_test_tensor))
    return test_loss.item()


def free_gpu():
    """Drop Python refs to GPU tensors and force CUDA free."""
    gc.collect()
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass


Dataset being used: beijing
Running in notebook: True
Params path: param_config/baseline_params.yaml
Using window length: 1024


In [3]:
"NASA dataset"

def fold_by_engine_unit(df, feature_cols, target_col='RUL', single_target: bool=False, pad_value=0.0):
    """Fold data into 3D array per engine (unit) with padding.
    single_target: If True, return only last RUL per engine; else full sequence.
    Returns:
        X: (num_units, max_seq_len, num_features)
        y: (num_units, max_seq_len) if single_target=False
        (num_units, 1) if single_target=True
        seq_lens: list of original sequence lengths per unit"""
    units        = df['unit'].unique()
    num_features = len(feature_cols)
    seq_lens     = [len(df[df['unit']==u]) for u in units]
    max_len      = max(seq_lens)
    
    X_folded = np.full((len(units), max_len, num_features), pad_value, dtype=np.float32)
    if single_target:
        y = np.zeros((len(units), 1), dtype=np.float32)
    else:
        y = np.full((len(units), max_len), pad_value, dtype=np.float32)
    
    for i, u in enumerate(units):
        unit_df = df[df['unit']==u]
        seq_len = len(unit_df)
        X_folded[i, :seq_len] = unit_df[feature_cols].values
        if single_target:
            y[i, 0] = unit_df[target_col].values[-1]  # last timestep RUL
        else:
            y[i, :seq_len] = unit_df[target_col].values
    return X_folded, y, seq_lens

def pad_X_to_max(X, target_len):
    padded = np.zeros((X.shape[0], target_len, X.shape[2]), dtype=X.dtype)
    padded[:, :X.shape[1], :] = X
    return padded

def pad_y_to_max(y, target_len, pad_value=0.0):
    if y.ndim == 2:  # full sequences
        padded = np.full((y.shape[0], target_len), pad_value, dtype=y.dtype)
        padded[:, :y.shape[1]] = y
    else:
        padded = y
    return padded


if desired_dataset == "nasa":
    # --- Load NASA FD004 dataset ---
    nasa_folder   = "../public_datasets/3D/NASA"
    specific_file = "FD004"
    train_file    = f"train_{specific_file}.txt"
    test_file     = f"test_{specific_file}.txt"
    rul_file      = f"RUL_{specific_file}.txt"

    cols       = ['unit', 'cycle'] + [f'op{i}' for i in range(1, 4)] + [f's{i}' for i in range(1, 22)]
    train_df   = pd.read_csv(f"{nasa_folder}/{train_file}", delim_whitespace=True, header=None, names=cols).dropna(axis=1, how='all')
    test_df    = pd.read_csv(f"{nasa_folder}/{test_file}",  delim_whitespace=True, header=None, names=cols).dropna(axis=1, how='all')
    rul_series = pd.read_csv(f"{nasa_folder}/{rul_file}", header=None).iloc[:,0]

    # compute RUL for train
    train_df['RUL'] = train_df.groupby('unit')['cycle'].transform(lambda x: x.max() - x)

    # compute RUL for test
    last_cycle     = test_df.groupby('unit')['cycle'].max().to_dict()
    test_df['RUL'] = [rul_series[row['unit']-1] + (last_cycle[row['unit']] - row['cycle'])
                    for _, row in test_df.iterrows()]

    # --- Scale features ---
    feature_cols = cols[2:]
    scaler       = StandardScaler()
    train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

    # --- Fold by engine and pad to max length ---
    single_target = False
    X_train, y_train, train_seq_lens = fold_by_engine_unit(train_df, feature_cols, single_target=single_target)
    X_test,  y_test,  test_seq_lens  = fold_by_engine_unit(test_df,  feature_cols, single_target=single_target)

    max_len = max(X_train.shape[1], X_test.shape[1])
    X_train = pad_X_to_max(X_train, max_len)
    X_test  = pad_X_to_max(X_test,  max_len)
    y_train = pad_y_to_max(y_train, max_len)
    y_test  = pad_y_to_max(y_test,  max_len)

    # --- Scale target ---
    y_scaler       = StandardScaler()
    y_train_scaled = y_scaler.fit_transform(y_train)
    y_test_scaled  = y_scaler.transform(y_test)

    # --- Use full sequences directly ---
    X_train_final, y_train_final = X_train, y_train_scaled
    X_test_final,  y_test_final  = X_test,  y_test_scaled

    print("Final NASA dataset (full sequences):")
    print(f"X_train={X_train_final.shape} ({X_train_final.nbytes/1024**2:.1f} MB), y_train={y_train_final.shape} ({y_train_final.nbytes/1024**2:.1f} MB)")
    print(f"X_test={X_test_final.shape} ({X_test_final.nbytes/1024**2:.1f} MB), y_test={y_test_final.shape} ({y_test_final.nbytes/1024**2:.1f} MB)")


In [4]:
"Milling"
import h5py

def load_h5_files_in_1_X_array(folder, y_file):
    """Load all h5 files in folder (except y_file) into a single 3D np array X
    checks if X arrays are of same length, if not, truncate to shortest length"""
    arrays = []
    for file in sorted(os.listdir(folder)):
        if file.endswith(".h5") and file != y_file:
            path = os.path.join(folder, file)
            with h5py.File(path, "r") as f:
                key  = list(f.keys())[0]
                data = f[key][()][:-2]  # remove last 2 rows
                arrays.append(data)
    # unify lengths
    min_len = min(a.shape[0] for a in arrays)
    arrays  = [a[:min_len] for a in arrays]
    X       = np.stack(arrays, axis=0)  # (n_files, timesteps, features)
    return X

if desired_dataset == "milling":
    milling_folder = "../public_datasets/3D/milling/Dataset 1 h5"
    y_file         = "stability_boundary1.h5"

    if not 'X_milling' in locals():
        X_milling = load_h5_files_in_1_X_array(milling_folder, y_file)
        print(f"Loaded X_milling with shape {X_milling.shape}")

    if not 'y_milling' in locals():
        with h5py.File(os.path.join(milling_folder, y_file), "r") as f:
            key = list(f.keys())[0]
            y_milling = f[key][()]
    print(f"{X_milling.shape=}, {y_milling.shape=}")

    print(X_milling[0:10, 0, :])
    print(y_milling[0:10, :])



In [5]:
"""[RUN ME] Preprocess dataset, as class"""
# from dataset_preprocessors import DatasetLoading
from preprocessing.window_folder import WindowFolder
from utils.data_utils import select_top_X_features

class DatasetLoading:

    @staticmethod
    def load_ecg_data():
        ECG_data_path= "../public_datasets/3D/ptb-xl-1.0.3"
        loader       = ECGLoader(ECG_data_path)
        X, y, _, _   = loader.load_dataset(sampling="lr", target="diagnostic_superclass_multi",
                                        segment_duration_sec=200, max_records=2000,
                                        continuous_target=True)
        return X, y

    @staticmethod
    def load_argoverse_data():
        argoverse_data_path = "../public_datasets/3D/argoverse_forecasting"
        folder_name = "00a0ec58-1fb9-4a2b-bfd7-f4e5da7a9eff"
        file_name   = "scenario_00a0ec58-1fb9-4a2b-bfd7-f4e5da7a9eff.parquet"
        return process_argoverse_parquet(f"{argoverse_data_path}/{folder_name}/{file_name}")

    @staticmethod
    def load_asm_data():
        "For info on processing search term 'Fouad intervening', points to a cell in the ASM notebook"
        X_3d  = np.load("../public_datasets/3D/ASM/X_3d.npy")
        y_asm = np.load("../public_datasets/3D/ASM/y_asm.npy")
        return X_3d, y_asm

    @staticmethod
    def load_china_data() -> tuple[np.ndarray, np.ndarray]:
        """Load China weather, split first NUM_PAGES_TO_USE stations into SPLIT_RATIO windows."""
        dataset_location = "../public_datasets/3D/china_weather/weather2k.npy"
        china_data       = np.load(dataset_location, mmap_mode='r').transpose(0, 2, 1)
        print(f"Original China data shape: {china_data.shape}")

        y_indices = [4, 5, 6, 7, 10]
        mask      = np.ones(china_data.shape[2], dtype=bool)
        mask[y_indices] = False
        X_cut  = china_data[:, :, mask]         # (stations, timesteps, n_features)
        y_cut  = china_data[:, :, y_indices]    # (stations, timesteps, n_targets)

        print(f"X_cut: {X_cut.shape}, y_last: {y_cut.shape}")
        return X_cut, y_cut

    @staticmethod
    def load_gas_data() -> tuple[np.ndarray, np.ndarray]:
        """Load gas CSVs and produce X and full y per page, pad pages to max length
        Load gas CSVs and produce X and y windows of uniform length NUM_ROWS.
        Steps:
        1. Read all CSVs in the folder.
        2. Split each CSV into as many full windows of NUM_ROWS as possible.
        Extra rows that don't fit a window are discarded.
        3. For each window, take the last row as y.
        4. Stack all X windows and y rows, preserving order.
        5. If total number of windows > NUM_PAGES_TO_USE, truncate to NUM_PAGES_TO_USE.
        Returns:
            X_windows: (NUM_PAGES_TO_USE, NUM_ROWS, n_features)
            y_windows: (NUM_PAGES_TO_USE, n_targets)"""
        gas_folder = "../public_datasets/3D/gas_emissions"
        csv_files  = sorted([f for f in os.listdir(gas_folder) if f.endswith(".csv")])

        X_pages, y_pages = [], []

        # load all pages first
        for file in csv_files:
            df     = pd.read_csv(os.path.join(gas_folder, file))
            X_file = df.iloc[:, :-2].values      # features (timesteps × features)
            y_file = df.iloc[:, -2:].values      # targets  (timesteps × 2)

            # Outlier clipping
            for i in range(y_file.shape[1]):
                mean, std    = y_file[:, i].mean(), y_file[:, i].std()
                y_file[:, i] = np.clip(y_file[:, i], mean - 3 * std, mean + 3 * std)
            X_pages.append(X_file)
            y_pages.append(y_file)

        # pad pages to max length
        max_len_X = max(x.shape[0] for x in X_pages)
        max_len_y = max(y.shape[0] for y in y_pages)

        X_full = np.stack([np.pad(x, ((0, max_len_X - x.shape[0]), (0,0))) for x in X_pages], axis=0)
        y_full = np.stack([np.pad(y, ((0, max_len_y - y.shape[0]), (0,0))) for y in y_pages], axis=0)

        print(f"Gas dataset (padded): X={X_full.shape}, y={y_full.shape}")
        return X_full, y_full

    @staticmethod
    def load_germany_data() -> tuple[np.ndarray, np.ndarray]:
        """Load CAMELS-DE dataset, split each basin's timeseries into SPLIT_RATIO windows."""
        camels_root_folder = f"{public_data_loc}/3D/camels_de"
        timeseries_folder  = os.path.join(camels_root_folder, "timeseries")
        zarr_path          = os.path.join(camels_root_folder, "camels_de_timeseries.zarr")

        if os.path.exists(zarr_path):
            X = xr.open_zarr(zarr_path)["X"].values  # (basins, timesteps, features)
        else:
            X = GermanyDataset.load_X_from_scratch(timeseries_folder, zarr_path)
        y = GermanyDataset.load_y_from_scratch(camels_root_folder)  # (basins, targets)

        # ---
        total_nan_frac = np.isnan(X).sum() / X.size
        total_nan_frac_y = np.isnan(y).sum() / y.size
        print(f"X missing: {total_nan_frac:.3%}, y missing: {total_nan_frac_y:.3%}")
        # ---
        X, y = np.nan_to_num(X), np.nan_to_num(y)
        return X, y

    @staticmethod
    def load_india_data():
        """Load India catchment dataset, split into windows like WeatherBench."""
        data_path      = "../public_datasets/3D/india_catchments"
        forcing_folder = "catchment_mean_forcings"
        clim_file      = "attributes_csv/camels_ind_clim.csv"

        y_df           = pd.read_csv(f"{data_path}/{clim_file}")
        catchment_ids  = y_df.iloc[:, 0].astype(int).values
        y_all          = y_df.iloc[:, 1:]

        forcing_files    = sorted(os.listdir(f"{data_path}/{forcing_folder}"))
        X_list, file_ids = [], []
        for f in forcing_files:
            df = pd.read_csv(os.path.join(data_path, forcing_folder, f))
            X_list.append(df.drop(columns=['year','month','day','pet(mm/day)']).values)
            file_ids.append(int(f.split('.')[0]))

        X     = np.stack(X_list, axis=0)
        order = [file_ids.index(cid) for cid in catchment_ids]
        X     = X[order]
        y     = y_all.values

        le = LabelEncoder()
        for i in range(y.shape[1]):
            if isinstance(y[0, i], str):
                le.fit(y[:, i])
                y[:, i] = le.transform(y[:, i])
        y = y.astype(float)
        return X, y

    @staticmethod
    def load_nasa_data():
        return X_train.shape, X_test, y_train_scaled, y_test_scaled

    @staticmethod
    def load_panama_data() -> tuple[np.ndarray, np.ndarray]:
        """Load Panama electricity, split first NUM_PAGES_TO_USE stations into SPLIT_RATIO windows."""
        file_name     = "train.csv"
        file_location = f"{public_data_loc}/3D/panama/{file_name}"

        df        = pd.read_csv(file_location)
        df        = df.drop(columns=['datetime'])

        y_indices = [1] # idx of valid y
        X_cut     = df.drop(df.columns[y_indices], axis=1).to_numpy()
        y_cut     = df.iloc[:, y_indices].to_numpy().reshape(-1, 1)  # make 2D

        X_cut = X_cut[None, :, :]
        y_cut = y_cut[None, :, :]
        return X_cut, y_cut

    @staticmethod
    def load_weather_data() -> tuple[np.ndarray, np.ndarray]:
        """Load WeatherBench data, keep the first NUM_PAGES_TO_USE spatial locations,
        split each series into SPLIT_RATIO contiguous windows, and truncate
        each window to NUM_ROWS timesteps. Returns ML-ready (X, y) arrays
        where y is the last timestep of each window.
        Returns:
            X_windows: (NUM_PAGES_TO_USE * SPLIT_RATIO, page_len, n_features)
            y_windows: (NUM_PAGES_TO_USE * SPLIT_RATIO, n_targets)"""
        variables_X    = ["2m_temperature", "10m_u_component_of_wind", "10m_v_component_of_wind"]
        variables_y    = ["mean_sea_level_pressure", "total_precipitation_6hr"]
        dataset_folder = "../public_datasets/3D/weather_bench"
        weather        = WeatherDataset(dataset_folder, variables_X, variables_y)
        X_xarr, y_xarr = weather.load_dataset()

        X_3d    = weather.prepare_X_features(X_xarr, mode="3d")   # (time, channels, lat*lon)
        X_pages = weather.reshape_for_ml(X_3d)

        y_time_series = np.stack([y_xarr[var].values for var in y_xarr.data_vars], axis=-1)
        time_dim, lat_dim, lon_dim, num_targets = y_time_series.shape
        y_long_series = y_time_series.reshape(time_dim, -1, num_targets)
        y_pages       = np.moveaxis(y_long_series, [0, 1, 2], [1, 0, 2])
        print(f"Original Weather shape: {X_pages.shape}, {y_pages.shape}")
        return X_pages, y_pages

    @staticmethod
    def load_beijing_data() -> tuple[np.ndarray, np.ndarray]:
        """Load Beijing Air Quality dataset, encode wind direction, handle missing values.
        - X: (stations, timesteps, features) unwindowed
        - y: (stations, timesteps, 1) unwindowed
        Folding, train/test split, and scaling are handled by load_or_preprocess_dataset."""
        data_path    = "../public_datasets/3D/beijing"
        target_col   = 'PM2.5'
        cols_to_drop = ['No','year','month','day','hour','station']

        def encode_wind_direction_simple(wd_series):
            wd_map = {'N':0,'NNE':22.5,'NE':45,'ENE':67.5,
                      'E':90,'ESE':112.5,'SE':135,'SSE':157.5,
                      'S':180,'SSW':202.5,'SW':225,'WSW':247.5,
                      'W':270,'WNW':292.5,'NW':315,'NNW':337.5}
            angles = wd_series.map(wd_map).fillna(0.0).values
            sin_wd = np.sin(np.deg2rad(angles))
            cos_wd = np.cos(np.deg2rad(angles))
            return np.stack([sin_wd, cos_wd], axis=-1)

        X_list, y_list = [], []
        for file in sorted(os.listdir(data_path)):
            if not file.endswith(".csv"):
                continue
            df = pd.read_csv(f"{data_path}/{file}")
            df = df.drop(columns=cols_to_drop, errors='ignore')

            # Encode wind direction
            if 'wd' in df.columns:
                wd_encoded = encode_wind_direction_simple(df['wd'])
                df         = df.drop(columns=['wd'])
                df[['wd_sin','wd_cos']] = wd_encoded

            # Separate features and target
            feature_cols = [c for c in df.columns if c != target_col]
            X_station    = df[feature_cols].values.astype(np.float32)
            y_station    = df[[target_col]].values.astype(np.float32)  # 2D: (timesteps, 1)
            X_list.append(X_station)
            y_list.append(y_station)

        X_all = np.stack(X_list, axis=0)  # (stations, timesteps, features)
        y_all = np.stack(y_list, axis=0)  # (stations, timesteps, 1)

        # apply per–station masking
        isnan_mask       = ~np.isnan(y_all[..., 0])
        X_clean, y_clean = [], []
        for s in range(X_all.shape[0]):
            m = isnan_mask[s]
            X_clean.append(X_all[s][m])
            y_clean.append(y_all[s][m])

        min_T = min(x.shape[0] for x in X_clean)
        X_all = np.stack([x[:min_T] for x in X_clean], axis=0)
        y_all = np.stack([y[:min_T] for y in y_clean], axis=0)
        print(f"after dropping missing y: X={X_all.shape}, y={y_all.shape}")

        total_nan_frac_X = np.isnan(X_all).sum() / X_all.size
        total_nan_frac_y = np.isnan(y_all).sum() / y_all.size

        print(f"Beijing X missing: {total_nan_frac_X:.3%}, y missing: {total_nan_frac_y:.3%}")
        X_all = np.nan_to_num(X_all)
        y_all = np.nan_to_num(y_all)

        # plt.figure(figsize=(14, 14))
        # # plt.plot(X_all[1, :2_100, 0])
        # plt.scatter(np.arange(X_all.shape[1]), X_all[0, :, 0], s=1)
        # plt.title("Beijing Air Quality - PM2.5")
        # plt.xlabel("Time")
        # plt.ylabel("PM2.5 Concentration")
        # plt.show()
        print(f"Beijing dataset loaded: X={X_all.shape}, y={y_all.shape}")
        return X_all, y_all

def load_or_preprocess_dataset(desired_dataset: str, page_num, do_we_scale_y, random_seed,
                               use_cache: bool = True, dataset_window = None) -> tuple[np.ndarray, ...]:
    """Load cached preprocessed dataset if available, otherwise preprocess and cache it."""
    new_dir_name   = f"{desired_dataset}_{page_num}pages"
    save_dir       = f"{interim_data_loc}/{new_dir_name}"
    X_full, y_full = dataset_loaders_dict[desired_dataset]()
    print(f"X_train: {X_full.shape} ({X_full.nbytes/1024**2:.1f} MB)")
    print(f"y_full: {y_full.shape} ({y_full.nbytes/1024**2:.1f} MB)")

    # if desired_dataset == 'asm':
    #     return X_train, X_test, y_train_scaled, y_test_scaled, window_size

    if dataset_window is not None:
        X_full, y_full, window_size, _ = WindowFolder.auto_fold_timeseries(X_full, y_full, fs=1.0, peak_strength=2.0, fallback_window=NUM_ROWS, 
                    denoise=False, window_size=dataset_window, max_pages=NUM_PAGES_TO_USE, num_pages_to_use=NUM_PAGES_TO_USE)
        print(f"Using predefined dataset window size: {dataset_window}")
    else:
        X_full, y_full, window_size, _ = WindowFolder.auto_fold_timeseries(X_full, y_full, fs=1.0, peak_strength=2.0, fallback_window=NUM_ROWS,
                    denoise=False, window_size=None, max_pages=NUM_PAGES_TO_USE, num_pages_to_use=NUM_PAGES_TO_USE)
        print(f"Auto-detected dataset window size: {window_size}")
    print("Folded the dataset")

    if use_cache and os.path.exists(save_dir): # Load from cache
        print("Path exists, loading from cache:", save_dir)
        X_train        = np.load(f"{save_dir}/X_train.npz")['data']
        X_test         = np.load(f"{save_dir}/X_test.npz")['data']
        y_train_scaled = np.load(f"{save_dir}/y_train.npz")['data']
        y_test_scaled  = np.load(f"{save_dir}/y_test.npz")['data']

    else: # Preprocess fresh
        print("Path does not exist, preprocessing fresh:", save_dir)
        os.makedirs(save_dir, exist_ok=True)
        preprocessor = DatasetPreprocessor(page_num=page_num, test_size=0.2, random_seed=random_seed)
        X_train, X_test, y_train_scaled, y_test_scaled = preprocessor.fit_transform(X_full, y_full, scale_y=do_we_scale_y)

        del X_full, y_full, preprocessor
        gc.collect()

        # Save to cache
        np.savez_compressed(f"{save_dir}/X_train.npz", data=X_train.astype(np.float32))
        np.savez_compressed(f"{save_dir}/X_test.npz", data=X_test.astype(np.float32))
        np.savez_compressed(f"{save_dir}/y_train.npz", data=y_train_scaled.astype(np.float32))
        np.savez_compressed(f"{save_dir}/y_test.npz", data=y_test_scaled.astype(np.float32))
    return X_train, X_test, y_train_scaled, y_test_scaled, window_size


dataset_loaders_dict = {
    "ecg":               DatasetLoading.load_ecg_data,
    "argoverse":         DatasetLoading.load_argoverse_data,
    "weather":           DatasetLoading.load_weather_data,
    "india_catchment":   DatasetLoading.load_india_data,
    "germany_catchment": DatasetLoading.load_germany_data,
    "china_weather":     DatasetLoading.load_china_data,
    "gas":               DatasetLoading.load_gas_data,
    "panama":            DatasetLoading.load_panama_data,
    "beijing":           DatasetLoading.load_beijing_data,
    "asm":               DatasetLoading.load_asm_data}

do_we_scale_y = True
rng  = np.random.default_rng()
seed = rng.integers(42, 70)

if desired_dataset in dataset_loaders_dict:
    X_train, X_test, y_train_scaled, y_test_scaled, window_size = load_or_preprocess_dataset(desired_dataset, NUM_PAGES_TO_USE, 
                                                     do_we_scale_y, dataset_window=dataset_window, random_seed=seed, use_cache=False)
elif desired_dataset == "nasa":
    X_train        = X_train_final
    X_test         = X_test_final
    y_train_scaled = y_train_final
    y_test_scaled  = y_test_final
    window_size = 'N/A'

if X_train.shape[2] > 300:
    top_features_pct         = params["general_params"]["top_features_big_dataset_pct"] # % top features to select
    X_train, X_test, top_idx = select_top_X_features(X_train, y_train_scaled, X_test, top_features_pct)
elif X_train.shape[2] > 30:
    top_features_pct         = params["general_params"]["top_features_pct"] # % of top features to select
    X_train, X_test, top_idx = select_top_X_features(X_train, y_train_scaled, X_test, top_features_pct)

# save the model here

print(f"Random seed: {seed}")
print(f"X_train: {X_train.shape} ({X_train.nbytes/1024**2:.1f} MB), X_test: {X_test.shape} ({X_test.nbytes/1024**2:.1f} MB)")
print(f"y_train: {y_train_scaled.shape} ({y_train_scaled.nbytes/1024**2:.1f} MB), y_test: {y_test_scaled.shape} ({y_test_scaled.nbytes/1024**2:.1f} MB)")
print(f"Mean: {X_train.mean():.2f}, {X_test.mean():.2f}, {y_train_scaled.mean():.2f}, {y_test_scaled.mean():.2f}")
print(f"Y is scaled: {do_we_scale_y}")
print("Min:", np.min(X_train), np.min(X_test))
print("Max:", np.max(X_train), np.max(X_test))


after dropping missing y: X=(12, 34111, 12), y=(12, 34111, 1)
Beijing X missing: 0.729%, y missing: 0.000%
Beijing dataset loaded: X=(12, 34111, 12), y=(12, 34111, 1)
X_train: (12, 34111, 12) (18.7 MB)
y_full: (12, 34111, 1) (1.6 MB)
Shape input to window folder: X=(12, 34111, 12), y=(12, 34111, 1)
[INFO] Selected dominant feature index: 0, window size: 1024
Using predefined dataset window size: 1024
Folded the dataset
Path does not exist, preprocessing fresh: ../interim_data/beijing_10pages
Random seed: 58
X_train: (264, 1024, 12) (12.4 MB), X_test: (66, 1024, 12) (3.1 MB)
y_train: (264, 1) (0.0 MB), y_test: (66, 1) (0.0 MB)
Mean: 0.00, 0.02, 0.00, -0.03
Y is scaled: True
Min: -16.01623 -101.23606
Max: 16.217276 1179.2166


In [6]:
if desired_dataset == "asm":
    from asm_stuff.main_runner import prepare_asm_train_test

    top_idx = [479, 517, 54, 165, 121, 77, 177, 188, 55, 509, 53, 52, 76,
            388, 125, 131, 84, 140, 81, 124, 189, 185, 206, 385, 160, 182,
            178, 75, 145, 144, 100, 306, 102, 205, 0, 149, 117, 163, 312,
            204, 98, 101, 157, 128, 207, 202, 159, 409, 158, 156, 415, 99,
            203, 103, 97, 201, 96, 200, 1]

    asm_folder_loc = "../public_datasets/3D/ASM"

    X_train, X_test, y_train_scaled, y_test_scaled = prepare_asm_train_test(asm_folder_loc, top_idx, keep_frac=0.08, keep='first')
    print(X_train.shape, y_train_scaled.shape)

    rng  = np.random.default_rng()
    seed = rng.integers(42, 70)

    if 'window_size' not in locals():
        window_size = X_train.shape[1]
    if 'label_frac' not in locals():
        label_frac= 1


In [7]:
"[RUN ME] Setup step"
label_frac     = params["basics"]["label_frac"]
data_splitting = params["basics"]["data_splitting"]

rng       = np.random.default_rng(seed)
n_train   = len(X_train)
n_labeled = int(np.ceil(label_frac * n_train))
perm      = rng.permutation(n_train)

if data_splitting == "missing_labels":
    # Keep all of X_train, split y
    X_L = X_train[perm[:n_labeled]]
    y_L = y_train_scaled[perm[:n_labeled]]
    X_U = X_train[perm[n_labeled:]]
    y_U = y_train_scaled[perm[n_labeled:]]
elif data_splitting == "reduced_data":
    # Shrink X_train & y_train by fraction, no unlabeled
    X_L = X_train[perm[:n_labeled]]
    y_L = y_train_scaled[perm[:n_labeled]]
    X_U = np.empty((0, *X_L.shape[1:]), dtype=X_L.dtype)
    y_U = np.empty((0, *y_L.shape[1:]), dtype=y_L.dtype)

# X_train for supervised training is only the labeled portion
X_train        = X_L
y_train_scaled = y_L

# Optional: combine for TimeVAE or other use
X_small = np.concatenate([X_train, X_test], axis=0)
y_small = np.concatenate([y_train_scaled, y_test_scaled], axis=0)

print(f"{data_splitting=}, {label_frac=}")
print(f"X_L: {X_L.shape}, y_L: {y_L.shape}")
print(f"X_U: {X_U.shape}, y_U: {y_U.shape}")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"X_small: {X_small.shape}")

if params["run_console"]["timevae"] == True:
    timevae_file_name = f"{desired_dataset}_frac{label_frac}"
    timevae_folder    = f"{interim_data_loc}/timevae/{timevae_file_name}"
    os.makedirs(timevae_folder, exist_ok=True)
    np.savez_compressed(f"{timevae_folder}.npz", data=np.array(X_small, dtype=np.float32))
    print(f"Saved {timevae_file_name} to TimeVAE repo")

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}, X_small (TimeVAE): {X_small.shape}")
print(f"y_train: {y_train_scaled.shape}, y_test: {y_test_scaled.shape}, y_small (TimeVAE): {y_small.shape}")
print(f"Mean: {np.mean(X_train):.2f}, {np.mean(X_test):.2f}, {y_train_scaled.mean():.2f}, {y_test_scaled.mean():.2f}")


data_splitting='reduced_data', label_frac=1
X_L: (264, 1024, 12), y_L: (264, 1)
X_U: (0, 1024, 12), y_U: (0, 1)
X_train: (264, 1024, 12), X_test: (66, 1024, 12)
X_small: (330, 1024, 12)
Saved beijing_frac1 to TimeVAE repo
X_train: (264, 1024, 12), X_test: (66, 1024, 12), X_small (TimeVAE): (330, 1024, 12)
y_train: (264, 1), y_test: (66, 1), y_small (TimeVAE): (330, 1)
Mean: -0.00, 0.02, 0.00, -0.03


In [8]:
"[RUN ME] functions for Headsup/cellsup"

def make_augmentations(X: torch.Tensor,  augment_type: str, device, scale_factor: float = 0.1) -> torch.Tensor:
    """Apply a chosen augmentation to a 3D time series batch.
    - X: Input array of shape (batch, time, channels).
    - augment_type: Type of augmentation to apply.
    - scale_factor: Magnitude factor controlling strength of augmentation.
    returns augmented array with the same shape as X (except cropping)"""
    batches, timesteps, cols = X.shape
    X = X.to(device)
    if augment_type == "jitter":
        noise = torch.randn_like(X) * float(scale_factor)
        return X + noise
    if augment_type == "scaling":
        # per-sample, per-channel scaling factor
        factor = torch.randn(batches, 1, cols, device=device) * scale_factor + 1.0
        return X * factor
    if augment_type == "mag_warp":
        mag = torch.empty(batches, 1, 1, device=device).uniform_(1 - scale_factor, 1 + scale_factor)
        return X * mag
    if augment_type == "time_warp":
        B, T, C = X.shape
        # random smooth warp along time
        tt = torch.linspace(-1, 1, T, device=device).unsqueeze(0).repeat(B, 1)  # (B,T)
        warp = tt + scale_factor * torch.randn(B, T, device=device)  # jittered time coords
        warp = warp.clamp(-1, 1)

        # make grid: need 2 coords (x,y), here y is dummy zero
        grid = torch.stack([warp, torch.zeros_like(warp)], dim=-1)  # (B,T,2)
        grid = grid.unsqueeze(2)  # (B,T,1,2)

        X_reshaped = X.permute(0, 2, 1).unsqueeze(-1)  # (B,C,T,1)
        X_warped   = F.grid_sample(X_reshaped, grid, mode="bilinear",
                                padding_mode="border", align_corners=True)
        return X_warped.squeeze(-1).permute(0, 2, 1)  # back to (B,T,C)
    if augment_type == "permutation":
        X_aug = torch.empty_like(X)
        # per-sample random segmentation + permutation
        for b in range(batches):
            n_segs   = int(torch.randint(2, 5, (1,)).item())
            pts      = np.linspace(0, timesteps, n_segs + 1, dtype=int)
            perm     = np.random.permutation(n_segs)
            pieces   = [X[b, pts[i]:pts[i+1], :] for i in perm]
            X_aug[b] = torch.cat(pieces, dim=0)
        return X_aug
    if augment_type == "cropping":
        keep    = int(timesteps * (1 - scale_factor))
        start   = int(torch.randint(0, timesteps - keep + 1, (1,)).item())
        cropped = X[:, start:start + keep, :]
        # pad or trim to keep shape (B, T, C)
        if cropped.shape[1] < timesteps:
            pad = torch.zeros(batches, timesteps - cropped.shape[1], cols, device=device)
            return torch.cat([cropped, pad], dim=1)
        else:
            return cropped
    if augment_type == "masking":
        mask  = (torch.rand(batches, timesteps, cols, device=device) < scale_factor)
        X_aug = X.clone()
        X_aug[mask] = 0.0
        return X_aug
    if augment_type == "drift":
        drift = torch.linspace(0, float(scale_factor), timesteps, device=device).view(1, timesteps, 1)
        sign  = 1.0 if torch.rand(1, device=device) < 0.5 else -1.0
        return X + sign * drift
    raise ValueError(f"Unknown augment type {augment_type}")

def _interpolate_to_length(x: torch.Tensor, target_len: int) -> torch.Tensor:
    """Interpolate along time dimension to match target length."""
    # b, t, c = x.shape
    x = x.permute(0, 2, 1).unsqueeze(-1)  # (B, C, T, 1)
    x = F.interpolate(x, size=(target_len, 1), mode="linear", align_corners=True)
    return x.squeeze(-1).permute(0, 2, 1)  # (B, target_len, C)

def make_two_views_augmentation(X: torch.Tensor, device, scale: float = 0.1):
    aug_types = ["jitter", "scaling", "masking", "cropping", "time_warp"]
    a1, a2 = np.random.choice(aug_types, 2, replace=False)
    v1, v2 = make_augmentations(X, a1, device, scale), make_augmentations(X, a2, device, scale)
    # if cropping shortened, interpolate back
    if v1.shape[1] != X.shape[1]:
        v1 = _interpolate_to_length(v1, X.shape[1])
    if v2.shape[1] != X.shape[1]:
        v2 = _interpolate_to_length(v2, X.shape[1])
    return v1, v2

# to remove (maybe)
def update_target_encoding_ema(target_encoder: torch.nn.Module, online_encoder: torch.nn.Module, decay: float):
    """In-place EMA update of target params: target = decay*target + (1-decay)*online"""
    # return decay * target_encoding + (1 - decay) * new_values

    with torch.no_grad():
        for t_param, o_param in zip(target_encoder.parameters(), online_encoder.parameters()):
            t_param.data.mul_(decay).add_(o_param.data * (1.0 - decay))

def get_loss_weights(step, warmup_steps, max_steps): #can have schedule as linear OR cosine
    if step < warmup_steps:
        return dict(recon=1.0, contrast=0.0, pred=0.0)
    else:
        t = (step - warmup_steps) / (max_steps - warmup_steps)
        return dict(
            recon=max(0.1, 1.0 - t),   # decay recon to 0.1
            contrast=min(0.5, t),      # grow contrast to 0.5
            pred=min(1.0, t))           # grow pred to 1.0

def schedule_learning_rate(step, max_steps, lr_0=1e-3, lr_end=1e-5, schedule_type="linear"):
    """Compute learning rate at given step with linear or cosine decay from lr0 to lr_end."""
    if schedule_type == "linear":
        return lr_0 - (lr_0 - lr_end) * (step / max_steps)
    elif schedule_type == "cosine":
        cosine_decay = 0.5 * (1 + math.cos(math.pi * step / max_steps))
        return lr_end + (lr_0 - lr_end) * cosine_decay
    else:
        raise ValueError(f"Unknown schedule type: {schedule_type}")

def nt_xent_loss(z1, z2, temperature=0.5):
    """Normalized temperature-scaled cross entropy loss"""
    B   = z1.size(0)   # dynamically set batch size
    z1  = F.normalize(z1, dim=1)
    z2  = F.normalize(z2, dim=1)
    z   = torch.cat([z1, z2], dim=0)  # (2B, dim)

    sim = torch.matmul(z, z.T) / temperature
    mask= torch.eye(2*B, device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(mask, -9e15)

    labels = torch.cat([torch.arange(B) + B, torch.arange(B)], dim=0).to(z.device)
    return F.cross_entropy(sim, labels)




In [ ]:
"""TimeVAE"""

@torch.no_grad()
def encode_timevae_in_batches(model: torch.nn.Module, X: np.ndarray, batch_size: int = 32, return_mean=False) -> np.ndarray:
    """Encode X to latent z in batches to avoid OOM."""
    zs = []
    device = next(model.parameters()).device
    for i in range(0, len(X), batch_size):
        xb = torch.from_numpy(X[i:i+batch_size]).float().to(device)
        # zs.append(z.cpu().numpy())
        z_mean, z_log_var, z_sample = model.encoder(xb)
        zs.append(z_mean.cpu().numpy() if return_mean else z_sample.cpu().numpy())
    return np.concatenate(zs, 0)

def run_timevae(X_train, X_test, y_train_scaled, y_test_scaled, *,
                timevae_file_name, dataset_name, device, batch_size=64):
    """Train (or load) TimeVAE, encode X in batches, evaluate predictors, and return metrics.
    Returns: losses, r2, profiling_metrics"""
    sys.path.append("./timevae_torch/src")
    from vae_pipeline import run_vae_pipeline
    from vae.timevae import TimeVAE

    # Paths
    model_dir = f"{interim_data_loc}/timevae/{timevae_file_name}"
    os.makedirs(model_dir, exist_ok=True)

    z_train, z_test, timevae_recon_loss, profiling_metrics = run_vae_pipeline(timevae_file_name, dataset_name, "timeVAE")
    timevae_model = TimeVAE.load(model_dir).to(device).eval()
    timevae_model._print_model_param_summary()

    z_train = encode_timevae_in_batches(timevae_model, X_train, batch_size=batch_size)
    z_test  = encode_timevae_in_batches(timevae_model, X_test, batch_size=batch_size)

    # z_mean_train = encode_timevae_in_batches(timevae_model, X_train, batch_size=64, return_mean=True)
    # print("z_train stats: ", np.nanmin(z_train), np.nanmax(z_train), np.isnan(z_train).any(), np.isinf(z_train).any())
    # print("z_test stats: ", np.nanmin(z_test), np.nanmax(z_test), np.isnan(z_test).any(), np.isinf(z_test).any())    

    # fondue_latent_dim = DimensionalityEstimator.estimate_latent_dim_using_fondue(z_mean_train, z_train, verbose=True)
    # active_dims_mask  = DimensionalityEstimator.prune_latent_dims(z_train, threshold_frac=0.05)
    # z_train = z_train[:, active_dims_mask]
    # z_test  = z_test[:, active_dims_mask]

    # Clip extreme values
    z_train = np.clip(z_train, -1e3, 1e3)
    z_test  = np.clip(z_test, -1e3, 1e3)

    # Evaluate predictors
    losses, rf_model = Preds().evaluate_models_on_dataset(z_train, y_train_scaled, z_test, y_test_scaled)
    r2 = rf_model.score(z_test, y_test_scaled)

    # # Convert to tensors
    # z_train_tensor = torch.tensor(z_train, dtype=torch.float32).to(device)
    # z_test_tensor  = torch.tensor(z_test,  dtype=torch.float32).to(device)
    # y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32).to(device)
    # y_test_tensor  = torch.tensor(y_test_scaled,  dtype=torch.float32).to(device)
    # Train + evaluate neural regressor
    # embedding_dim   = z_train_tensor.shape[1]
    # regression_head = make_regression_head(embedding_dim, layer1_dim, layer2_dim, layer3_dim,
    #                                        y_train_tensor, dropout, device)
    # nn_loss = evaluate_regressor(regression_head, z_train_tensor, z_test_tensor,
    #                              y_train_tensor, y_test_tensor, regressor_epochs, lr_regressor)
    # timevae_losses.append(nn_loss)

    return losses, r2, profiling_metrics, timevae_recon_loss, z_train, z_test

def log_timevae_results(dataset_name, losses, r2, profiling_metrics, recon_loss, filename="results/hyperparam_search.txt"):
    """Log TimeVAE results in a file and stdout."""
    rmse, linreg, catboost = losses[:3]

    with open(filename, 'a') as f:
        f.write(f"timevae/{dataset_name}\n")
        f.write(f"& Z (timevae) & {rmse:.4f} & {linreg:.4f} & {catboost:.4f}\n")
        f.write(f"R²: {r2:.3f}, L_recons: {recon_loss:.3f}\n")
        f.write("time & params & flops & memory\n")
        f.write(f"{profiling_metrics['runtime_s']:.3f} & {profiling_metrics['num_params_M']:.3f} & "
                f"{profiling_metrics['flops_M']:.3f} & {profiling_metrics['peak_memory_MB']:.3f}\n\n")

    print(f"timevae/{dataset_name}")
    print(f"& Z (timevae) & {rmse:.4f} & {linreg:.4f} & {catboost:.4f}")
    print(f"R²: {r2:.3f}, L_recons: {recon_loss:.3f}")
    print("time & params & flops & memory")
    print(f"{profiling_metrics['runtime_s']:.3f} & {profiling_metrics['num_params_M']:.3f} & "
          f"{profiling_metrics['flops_M']:.3f} & {profiling_metrics['peak_memory_MB']:.3f}\n")


if params["run_console"]["timevae"]:
    losses, r2, metrics, recon_loss, z_train, z_test = run_timevae(
        X_train, X_test, y_train_scaled, y_test_scaled,
        timevae_file_name=timevae_file_name,
        dataset_name=desired_dataset,
        device=device,
        batch_size=64)

    log_timevae_results(desired_dataset, losses, r2, metrics, recon_loss)

    print(f"dataset: {desired_dataset}, method: timevae")
    print("    RMSE      | LinReg | CatBoost | RForest")
    print(f"& Z (timevae) & {losses[0]:.4f} & {losses[1]:.4f} & {losses[2]:.4f}")
    print(f"R² (TimeVAE): {r2:.3f}")
    print(f"L_recons: {recon_loss:.3f}")
    


In [ ]:
hidden_dims_list = [8, 12, 16, 20]  # [8, 16, 32]
latent_dims_list = [6, 8, 12, 16]  # [8, 16, 32]
depth_list       = [3, 4]
lr_list          = [0.001, 0.05]
batch_size_list  = [256]

window_size = 64 
repeats = 2
counter = 0

for hd in hidden_dims_list:
    for ld in latent_dims_list:
        for d in depth_list:
            for lr in lr_list:
                for bs in batch_size_list:
                    rmse_accum, linreg_accum, catboost_accum, r2_accum = 0, 0, 0, 0
                    runtime_accum, params_accum, flops_accum, mem_accum = 0, 0, 0, 0

                    for _ in range(repeats):
                        model_cfg = {
                            "z_pooling_method": z_pooling_method,
                            "hidden_dims": hd,
                            "latent_dims": ld,
                            "depth": d,}
                        train_cfg = {
                            "lr": lr,
                            "patience": patience,
                            "epochs": ts2vec_epochs,
                            "batch_size": bs,
                            "window_size": window_size,}
                        losses, r2, metrics = run_ts2vec(
                            X_train, X_test, y_train_scaled, y_test_scaled,
                            model_cfg=model_cfg, train_cfg=train_cfg, device=device)
                        rmse_accum    += losses[0]
                        linreg_accum  += losses[1]
                        catboost_accum += losses[2]
                        r2_accum      += r2
                        runtime_accum += metrics['runtime_s']
                        params_accum  += metrics['num_params_M']
                        flops_accum   += metrics['flops_M']
                        mem_accum     += metrics['peak_memory_MB']

                    # average over repeats
                    losses_avg = [rmse_accum / repeats, linreg_accum / repeats, catboost_accum / repeats]
                    r2_avg     = r2_accum / repeats
                    metrics_avg = {
                        'runtime_s': runtime_accum / repeats,
                        'num_params_M': params_accum / repeats,
                        'flops_M': flops_accum / repeats,
                        'peak_memory_MB': mem_accum / repeats}

                    log_ts2vec_results(desired_dataset, losses_avg, r2_avg, metrics_avg, model_cfg, train_cfg)
                    print(f"done with config #{counter}")
                    counter += 1


In [ ]:
"dimensionality"
# intrinsic_dim_est = DimensionalityEstimator.estimate_intrinsic_dim_mle(X_train)
# print("Levina-Bickel MLE Intrinsic dim estimate:", intrinsic_dim_est)

# latent_dim_est = DimensionalityEstimator.estimate_intrinsic_dim_skdim(X_train, method="mle", K=15)
# print(f"Skdim estim. intrinsic dim: {latent_dim_est:.2f}")

# avg_dimensionality = (intrinsic_dim_est + latent_dim_est) / 2
# latent_dim = int(np.ceil(avg_dimensionality * 1.3))
# print(f"Chosen latent dim (1.3x avg): {latent_dim}")

# pca_latent_dim = DimensionalityEstimator.count_active_latents(timevae_model, X_train.reshape(X_train.shape[0], -1), kl_threshold=0.99)
# print(f"PCA-based latent dim for 95% energy: {pca_latent_dim}")


In [ ]:
class ARDSeqVAE(nn.Module):
    """
    Sequence VAE with ARD-style latent dimension selection
    X shape: (batch, seq_len, features)
    """
    def __init__(self, input_dim: int, latent_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder
        self.encoder_rnn = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        # ARD parameters
        self.log_alpha = nn.Parameter(torch.zeros(latent_dim))

        # Decoder
        self.fc_dec = nn.Linear(latent_dim, hidden_dim)
        self.decoder_rnn = nn.LSTM(hidden_dim, input_dim, batch_first=True)

    def encode(self, x):
        _, (h_n, _) = self.encoder_rnn(x)  # h_n: (1, batch, hidden_dim)
        h_n = h_n.squeeze(0)
        mu = self.fc_mu(h_n)
        logvar = self.fc_logvar(h_n)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, seq_len):
        h = F.relu(self.fc_dec(z)).unsqueeze(1).repeat(1, seq_len, 1)
        out, _ = self.decoder_rnn(h)
        return out

    def forward(self, x):
        seq_len = x.size(1)
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z, seq_len)
        return x_hat, mu, logvar, z

def ard_seq_vae_loss(x, x_hat, mu, logvar, log_alpha):
    recon_loss = F.mse_loss(x_hat, x, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - log_alpha - (mu**2 + torch.exp(logvar)) / torch.exp(log_alpha))
    return recon_loss + kl


latent_dim = 40
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)

model = ARDSeqVAE(
    input_dim=X_train_tensor.shape[2],
    latent_dim=latent_dim).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(200):
    x_hat, mu, logvar, z = model(X_train_tensor)
    loss = ard_seq_vae_loss(X_train_tensor, x_hat, mu, logvar, model.log_alpha)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if epoch % 50 == 0:
        print(epoch, loss.item())

# inspect
log_alpha = model.log_alpha.detach().cpu()
print("active dims:", (log_alpha < 8).sum().item())


In [14]:
"""TS2Vec"""

def run_ts2vec(X_train, X_test, y_train_scaled, y_test_scaled, *,
               model_cfg: dict, train_cfg: dict, device, save_ts2vec_encoder=False,):
    """Train TS2Vec with config dicts. Returns (losses, profiling_metrics)."""

    ts2vec = TS2VecEncoder(
        z_pooling=model_cfg["z_pooling_method"],
        lr=train_cfg["lr"],
        device=device,
        patience=train_cfg["patience"],
        max_train_length=train_cfg["window_size"],)

    metrics = ts2vec.fit_ts2vec(
        X_train,
        hidden_dims=model_cfg["hidden_dims"],
        output_dims=model_cfg["latent_dims"],
        depth=model_cfg["depth"],
        batch_size=train_cfg["batch_size"],
        n_epochs=train_cfg["epochs"],)

    z_train = ts2vec.encode(X_train, pooling=None)
    z_test  = ts2vec.encode(X_test,  pooling=None)

    z_train_flat = z_train.reshape(len(z_train), -1)
    z_test_flat  = z_test.reshape(len(z_test),  -1)

    losses, rf_model = Preds().evaluate_models_on_dataset(z_train_flat, y_train_scaled, z_test_flat,  y_test_scaled)
    r2 = rf_model.score(z_test_flat, y_test_scaled)
    # ====== plain predictors ======
    # z_train_tensor = torch.tensor(z_train_flat, dtype=torch.float32, device=device)
    # z_test_tensor  = torch.tensor(z_test_flat, dtype=torch.float32, device=device)
    # y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32, device=device)
    # y_test_tensor  = torch.tensor(y_test_scaled, dtype=torch.float32, device=device)

    # embedding_dim   = z_train_tensor.shape[1]
    # regression_head = make_regression_head(embedding_dim, layer1_dim, layer2_dim, layer3_dim,
    #                                        y_train_tensor, dropout, device)
    # test_loss = evaluate_regressor(regression_head, z_train_tensor, z_test_tensor,
    #                                y_train_tensor, y_test_tensor, regressor_epochs, lr_regressor)
    # ts2vec_losses.append(test_loss)
    return losses, r2, metrics

def log_ts2vec_results(dataset_name, losses, r2, metrics, model_cfg, train_cfg, filename="results/hyperparam_search.txt"):
    """Log TS2Vec results in the desired format, both to file and stdout."""
    rmse, linreg, catboost = losses[:3]  # only 3 models

    with open(filename, 'a') as f:
        f.write(f"ts2vec/{dataset_name}: hidden_dims={model_cfg['hidden_dims']} "
                f"latent_dims={model_cfg['latent_dims']} depth={model_cfg['depth']} "
                f"batch_size={train_cfg['batch_size']} lr={train_cfg['lr']} "
                f"window_size={train_cfg['window_size']}\n")
        f.write(f"& Z (ts2vec) & {rmse:.4f} & {linreg:.4f}  & {catboost:.4f}\n")
        f.write(f"R² (TS2Vec): {r2:.3f}\n")
        f.write("time & params & flops & memory \n")
        f.write(f"{metrics['runtime_s']:.3f} & {metrics['num_params_M']:.3f} & "
                f"{metrics['flops_M']:.3f} & {metrics['peak_memory_MB']:.3f}\n\n")

    print(f"ts2vec/{dataset_name}: hidden_dims={model_cfg['hidden_dims']} "
          f"latent_dims={model_cfg['latent_dims']} depth={model_cfg['depth']} "
          f"batch_size={train_cfg['batch_size']} lr={train_cfg['lr']} "
          f"window_size={train_cfg['window_size']}")
    print(f"& Z (ts2vec) & {rmse:.4f} & {linreg:.4f}  & {catboost:.4f}")
    print(f"R² (TS2Vec): {r2:.3f}")
    print("time & params & flops & memory")
    print(f"{metrics['runtime_s']:.3f} & {metrics['num_params_M']:.3f} & "
          f"{metrics['flops_M']:.3f} & {metrics['peak_memory_MB']:.3f}\n")


save_ts2vec_encoder = False #True

if params["run_console"]["ts2vec"] == True:
    z_pooling_method   = params["ts2vec"]["z_pooling_method"]
    # ts2vec_hidden_dims = params["ts2vec"]["ts2vec_hidden_dims"]
    # ts2vec_depth       = params["ts2vec"]["ts2vec_depth"]
    # patience           = params["ts2vec"]["patience"]
    # ts2vec_lr          = params["ts2vec"]["lr_encoder"]
    # ts2vec_latent_dims = params["ts2vec"]["ts2vec_latent_dims"]
    # ts2vec_epochs      = params["ts2vec"]["ts2vec_epochs"]
    # ts2vec_batch_size  = params["ts2vec"]["ts2vec_batch_size"]
    patience           = 25
    ts2vec_epochs      = 100
    ts2vec_batch_size  = 16
    ts2vec_latent_dims = 16 # latent dim

    ts2vec_lr          = 0.05
    ts2vec_hidden_dims = 16 # units in each layer (> than latent dim)
    ts2vec_depth       = 2  # num layers

    model_cfg = {
        "z_pooling_method": z_pooling_method,
        "hidden_dims": ts2vec_hidden_dims,
        "latent_dims": ts2vec_latent_dims,
        "depth": ts2vec_depth,}

    train_config = {
        "lr": ts2vec_lr,
        "patience": patience,
        "epochs": ts2vec_epochs,
        "batch_size": ts2vec_batch_size,
        "window_size": window_size,}

    losses, r2, metrics = run_ts2vec(X_train, X_test, y_train_scaled, y_test_scaled,
                                     model_cfg=model_cfg, train_cfg=train_config, device=device,)
    log_ts2vec_results(desired_dataset, losses, r2, metrics, model_cfg, train_config, filename="results/hyperparam_search.txt")



Epoch #0: loss=1.874788


KeyboardInterrupt: 

In [16]:
hidden_dims_list = [8, 12, 16, 20]  # [8, 16, 32]
latent_dims_list = [6, 8, 12, 16]  # [8, 16, 32]
depth_list       = [3, 4]
lr_list          = [0.001, 0.05]
batch_size_list  = [256]

window_size = 64 
repeats = 2
counter = 0

for hd in hidden_dims_list:
    for ld in latent_dims_list:
        for d in depth_list:
            for lr in lr_list:
                for bs in batch_size_list:
                    rmse_accum, linreg_accum, catboost_accum, r2_accum = 0, 0, 0, 0
                    runtime_accum, params_accum, flops_accum, mem_accum = 0, 0, 0, 0

                    for _ in range(repeats):
                        model_cfg = {
                            "z_pooling_method": z_pooling_method,
                            "hidden_dims": hd,
                            "latent_dims": ld,
                            "depth": d,}
                        train_cfg = {
                            "lr": lr,
                            "patience": patience,
                            "epochs": ts2vec_epochs,
                            "batch_size": bs,
                            "window_size": window_size,}
                        losses, r2, metrics = run_ts2vec(
                            X_train, X_test, y_train_scaled, y_test_scaled,
                            model_cfg=model_cfg, train_cfg=train_cfg, device=device)
                        rmse_accum    += losses[0]
                        linreg_accum  += losses[1]
                        catboost_accum += losses[2]
                        r2_accum      += r2
                        runtime_accum += metrics['runtime_s']
                        params_accum  += metrics['num_params_M']
                        flops_accum   += metrics['flops_M']
                        mem_accum     += metrics['peak_memory_MB']

                    # average over repeats
                    losses_avg = [rmse_accum / repeats, linreg_accum / repeats, catboost_accum / repeats]
                    r2_avg     = r2_accum / repeats
                    metrics_avg = {
                        'runtime_s': runtime_accum / repeats,
                        'num_params_M': params_accum / repeats,
                        'flops_M': flops_accum / repeats,
                        'peak_memory_MB': mem_accum / repeats}

                    log_ts2vec_results(desired_dataset, losses_avg, r2_avg, metrics_avg, model_cfg, train_cfg)
                    print(f"done with config #{counter}")
                    counter += 1


Epoch #0: loss=3.702925
Epoch #2: loss=3.565047
Epoch #4: loss=3.449259
Epoch #6: loss=3.343083
Epoch #8: loss=3.296654
Epoch #10: loss=3.206711
Epoch #12: loss=3.138403
Epoch #14: loss=3.078215
Epoch #16: loss=3.007391
Epoch #18: loss=2.966416
Epoch #20: loss=2.921290
Epoch #22: loss=2.862582
Epoch #24: loss=2.833378
Epoch #26: loss=2.768583
Epoch #28: loss=2.703507
Epoch #30: loss=2.684271
Epoch #32: loss=2.644313
Epoch #34: loss=2.642765
Epoch #36: loss=2.581351
Epoch #38: loss=2.608061
Epoch #40: loss=2.579443
Epoch #42: loss=2.568339
Epoch #44: loss=2.544028
Epoch #46: loss=2.549999
Epoch #48: loss=2.551280
Epoch #50: loss=2.476260
Epoch #52: loss=2.504344
Epoch #54: loss=2.476313
Epoch #56: loss=2.485783
Epoch #58: loss=2.463478
Epoch #60: loss=2.441316
Epoch #62: loss=2.412307
Epoch #64: loss=2.442372
Epoch #66: loss=2.416392
Epoch #68: loss=2.417678
Epoch #70: loss=2.408098
Epoch #72: loss=2.402137
Epoch #74: loss=2.375600
Epoch #76: loss=2.365916
Epoch #78: loss=2.387139
Epoch

Epoch #98: loss=2.310179
===== Profiling =====
Avg Runtime [s]: 0.02
Peak Memory [MB]: 196.67
# Params [x10^6]: 0.00
FLOPs [x10^6]: 1.59
Using device: cuda (GPU)
Linear Regression done
0:	learn: 0.9493929	total: 11.7ms	remaining: 2.34s
100:	learn: 0.1199776	total: 985ms	remaining: 965ms
199:	learn: 0.0377382	total: 1.95s	remaining: 0us
CatBoost done
Random Forest done
Epoch #0: loss=3.901453
Epoch #2: loss=3.608295
Epoch #4: loss=3.527911
Epoch #6: loss=3.509454
Epoch #8: loss=3.458918
Epoch #10: loss=3.317311
Epoch #12: loss=3.319819
Epoch #14: loss=3.253498
Epoch #16: loss=3.151561
Epoch #18: loss=3.083209
Epoch #20: loss=2.987021
Epoch #22: loss=2.950751
Epoch #24: loss=2.888335
Epoch #26: loss=2.845979
Epoch #28: loss=2.801536
Epoch #30: loss=2.735274
Epoch #32: loss=2.713766
Epoch #34: loss=2.683415
Epoch #36: loss=2.648589
Epoch #38: loss=2.611156
Epoch #40: loss=2.624016
Epoch #42: loss=2.579407
Epoch #44: loss=2.594240
Epoch #46: loss=2.549968
Epoch #48: loss=2.540974
Epoch #50

Epoch #98: loss=2.342817
===== Profiling =====
Avg Runtime [s]: 0.02
Peak Memory [MB]: 196.67
# Params [x10^6]: 0.00
FLOPs [x10^6]: 1.59
Using device: cuda (GPU)
Linear Regression done
0:	learn: 0.9439412	total: 10.8ms	remaining: 2.15s
100:	learn: 0.1338611	total: 984ms	remaining: 964ms
199:	learn: 0.0444193	total: 1.94s	remaining: 0us
CatBoost done
Random Forest done
ts2vec/beijing: hidden_dims=8 latent_dims=6 depth=3 batch_size=256 lr=0.001 window_size=64
& Z (ts2vec) & 0.7993 & 0.5333  & 0.6372
R² (TS2Vec): 0.463
time & params & flops & memory
0.020 & 0.002 & 1.585 & 196.665

done with config #0
Epoch #0: loss=3.743378
Epoch #2: loss=3.178956
Epoch #4: loss=2.847588
Epoch #6: loss=2.695305
Epoch #8: loss=2.586809
Epoch #10: loss=2.483612
Epoch #12: loss=2.506799
Epoch #14: loss=2.463005
Epoch #16: loss=2.384556
Epoch #18: loss=2.374901
Epoch #20: loss=2.385952
Epoch #22: loss=2.377196
Epoch #24: loss=2.506028
Epoch #26: loss=2.410264
Epoch #28: loss=2.406441
Epoch #30: loss=2.314054

Epoch #66: loss=2.299098
Early stopping at epoch 68
===== Profiling =====
Avg Runtime [s]: 0.02
Peak Memory [MB]: 196.67
# Params [x10^6]: 0.00
FLOPs [x10^6]: 1.59
Using device: cuda (GPU)
Linear Regression done
0:	learn: 0.9476608	total: 10.8ms	remaining: 2.15s
100:	learn: 0.1380534	total: 979ms	remaining: 960ms
199:	learn: 0.0436442	total: 1.94s	remaining: 0us
CatBoost done
Random Forest done
Epoch #0: loss=3.691147
Epoch #2: loss=3.224862
Epoch #4: loss=2.857880
Epoch #6: loss=2.716127
Epoch #8: loss=2.567101
Epoch #10: loss=2.524542
Epoch #12: loss=2.433150
Epoch #14: loss=2.438168
Epoch #16: loss=2.411511
Epoch #18: loss=2.383342
Epoch #20: loss=2.349912
Epoch #22: loss=2.356578
Epoch #24: loss=2.320189
Epoch #26: loss=2.339703
Epoch #28: loss=2.236287
Epoch #30: loss=2.315227
Epoch #32: loss=2.264839
Epoch #34: loss=2.324509
Epoch #36: loss=2.296197
Epoch #38: loss=2.267850
Epoch #40: loss=2.260768
Epoch #42: loss=2.256342
Epoch #44: loss=2.218234
Epoch #46: loss=2.214744
Epoch #

Epoch #92: loss=2.218647
Early stopping at epoch 93
===== Profiling =====
Avg Runtime [s]: 0.02
Peak Memory [MB]: 196.67
# Params [x10^6]: 0.00
FLOPs [x10^6]: 1.59
Using device: cuda (GPU)
Linear Regression done
0:	learn: 0.9384240	total: 10.7ms	remaining: 2.13s
100:	learn: 0.1219427	total: 972ms	remaining: 953ms
199:	learn: 0.0415832	total: 1.93s	remaining: 0us
CatBoost done
Random Forest done
ts2vec/beijing: hidden_dims=8 latent_dims=6 depth=3 batch_size=256 lr=0.05 window_size=64
& Z (ts2vec) & 0.7198 & 0.5222  & 0.6414
R² (TS2Vec): 0.455
time & params & flops & memory
0.019 & 0.002 & 1.585 & 196.665

done with config #1
Epoch #0: loss=3.807900
Epoch #2: loss=3.626902
Epoch #4: loss=3.558571
Epoch #6: loss=3.488424
Epoch #8: loss=3.416336
Epoch #10: loss=3.304827
Epoch #12: loss=3.221009
Epoch #14: loss=3.080509
Epoch #16: loss=3.054631
Epoch #18: loss=2.929740
Epoch #20: loss=2.899508
Epoch #22: loss=2.862409
Epoch #24: loss=2.824907
Epoch #26: loss=2.770594
Epoch #28: loss=2.73607

...

Epoch #60: loss=24.626660
Early stopping at epoch 62
===== Profiling =====
Avg Runtime [s]: 0.01
Peak Memory [MB]: 413.25
# Params [x10^6]: 0.01
FLOPs [x10^6]: 9.72
Using device: cuda (GPU)
Linear Regression done
0:	learn: 0.9486358	total: 27.8ms	remaining: 5.53s
100:	learn: 0.1025049	total: 2.58s	remaining: 2.53s
199:	learn: 0.0264187	total: 5.11s	remaining: 0us
CatBoost done
Random Forest done
ts2vec/beijing: hidden_dims=20 latent_dims=16 depth=3 batch_size=256 lr=0.05 window_size=64
& Z (ts2vec) & 0.7211 & 0.4288  & 0.6284
R² (TS2Vec): 0.477
time & params & flops & memory
0.011 & 0.010 & 9.716 & 413.253

done with config #61
Epoch #0: loss=3.572597
Epoch #2: loss=3.190257
Epoch #4: loss=3.029184
Epoch #6: loss=2.831539
Epoch #8: loss=2.635695
Epoch #10: loss=2.414484
Epoch #12: loss=2.211521
Epoch #14: loss=2.041302
Epoch #16: loss=1.938732
Epoch #18: loss=1.775131
Epoch #20: loss=1.754013
Epoch #22: loss=1.659479
Epoch #24: loss=1.673674
Epoch #26: loss=1.594485
Epoch #28: loss=1.4

Epoch #98: loss=1.048633
===== Profiling =====
Avg Runtime [s]: 0.01
Peak Memory [MB]: 497.16
# Params [x10^6]: 0.01
FLOPs [x10^6]: 12.17
Using device: cuda (GPU)
Linear Regression done
0:	learn: 0.9402908	total: 27.5ms	remaining: 5.46s
100:	learn: 0.1052586	total: 2.58s	remaining: 2.53s
199:	learn: 0.0275147	total: 5.12s	remaining: 0us
CatBoost done
Random Forest done
Epoch #0: loss=3.617322
Epoch #2: loss=3.245899
Epoch #4: loss=3.064437
Epoch #6: loss=2.830930
Epoch #8: loss=2.664634
Epoch #10: loss=2.394138
Epoch #12: loss=2.180368
Epoch #14: loss=2.140065
Epoch #16: loss=2.049748
Epoch #18: loss=1.853408
Epoch #20: loss=1.781180
Epoch #22: loss=1.679897
Epoch #24: loss=1.837269
Epoch #26: loss=1.693907
Epoch #28: loss=1.562884
Epoch #30: loss=1.517037
Epoch #32: loss=1.539211
Epoch #34: loss=1.441320
Epoch #36: loss=1.437266
Epoch #38: loss=1.223353
Epoch #40: loss=1.461944
Epoch #42: loss=1.281684
Epoch #44: loss=1.361937
Epoch #46: loss=1.254174
Epoch #48: loss=1.330626
Epoch #5

Epoch #98: loss=0.983744
===== Profiling =====
Avg Runtime [s]: 0.01
Peak Memory [MB]: 497.16
# Params [x10^6]: 0.01
FLOPs [x10^6]: 12.17
Using device: cuda (GPU)
Linear Regression done
0:	learn: 0.9410471	total: 27.7ms	remaining: 5.52s
100:	learn: 0.1095609	total: 2.58s	remaining: 2.53s
199:	learn: 0.0277729	total: 5.11s	remaining: 0us
CatBoost done
Random Forest done
ts2vec/beijing: hidden_dims=20 latent_dims=16 depth=4 batch_size=256 lr=0.001 window_size=64
& Z (ts2vec) & 0.7786 & 0.5597  & 0.7140
R² (TS2Vec): 0.326
time & params & flops & memory
0.013 & 0.012 & 12.173 & 497.162

done with config #62
Epoch #0: loss=4.669849
Epoch #2: loss=3.436117
Epoch #4: loss=3.052797
Epoch #6: loss=2.674230
Epoch #8: loss=2.364352
Epoch #10: loss=2.142747
Epoch #12: loss=2.148772
Epoch #14: loss=1.890402
Epoch #16: loss=1.667594
Epoch #18: loss=1.626163
Epoch #20: loss=1.687967
Epoch #22: loss=1.608829
Epoch #24: loss=1.341812
Epoch #26: loss=1.377147
Epoch #28: loss=1.395690
Epoch #30: loss=1.2

Epoch #74: loss=1669.092079
Early stopping at epoch 76
===== Profiling =====
Avg Runtime [s]: 0.01
Peak Memory [MB]: 497.16
# Params [x10^6]: 0.01
FLOPs [x10^6]: 12.17
Using device: cuda (GPU)
Linear Regression done
0:	learn: 0.9529358	total: 32.8ms	remaining: 6.52s
100:	learn: 0.1237824	total: 2.58s	remaining: 2.53s
199:	learn: 0.0308197	total: 5.11s	remaining: 0us
CatBoost done
Random Forest done
Epoch #0: loss=5.229075
Epoch #2: loss=3.269433
Epoch #4: loss=2.676325
Epoch #6: loss=2.430811
Epoch #8: loss=2.302345
Epoch #10: loss=1.895547
Epoch #12: loss=1.799964
Epoch #14: loss=1.970935
Epoch #16: loss=1.780265
Epoch #18: loss=1.521180
Epoch #20: loss=1.445796
Epoch #22: loss=1.320437
Epoch #24: loss=1.545403
Epoch #26: loss=1.273895
Epoch #28: loss=1.419075
Epoch #30: loss=1.186249
Epoch #32: loss=1.145360
Epoch #34: loss=1.045617
Epoch #36: loss=1.192146
Epoch #38: loss=1.286639
Epoch #40: loss=1.286940
Epoch #42: loss=1.170744
Epoch #44: loss=1.146710
Epoch #46: loss=1.031631
Epo

Epoch #84: loss=7.146659
Early stopping at epoch 85
===== Profiling =====
Avg Runtime [s]: 0.01
Peak Memory [MB]: 497.16
# Params [x10^6]: 0.01
FLOPs [x10^6]: 12.17
Using device: cuda (GPU)
Linear Regression done
0:	learn: 0.9505776	total: 32.3ms	remaining: 6.42s
100:	learn: 0.1060601	total: 2.58s	remaining: 2.53s
199:	learn: 0.0256268	total: 5.11s	remaining: 0us
CatBoost done
Random Forest done
ts2vec/beijing: hidden_dims=20 latent_dims=16 depth=4 batch_size=256 lr=0.05 window_size=64
& Z (ts2vec) & 0.7289 & 0.5213  & 0.6548
R² (TS2Vec): 0.426
time & params & flops & memory
0.013 & 0.012 & 12.173 & 497.162

done with config #63


In [ ]:
"""TS2Vec (fed)"""

def encode_in_batches(encoder, X, batch_size=64):
    out = []
    for i in range(0, X.shape[0], batch_size):
        z = encoder.encode(X[i:i+batch_size].cpu().numpy(), pooling=None)
        out.append(z)
    z = np.concatenate(out, axis=0)
    return torch.tensor(z.reshape(z.shape[0], -1), dtype=torch.float32)

if params["run_console"]["ts2vec_fed"] == True:
    z_pooling_method   = params["ts2vec"]["z_pooling_method"]
    ts2vec_hidden_dims = params["ts2vec"]["ts2vec_hidden_dims"]
    ts2vec_depth       = params["ts2vec"]["ts2vec_depth"]
    patience           = params["ts2vec"]["patience"]
    ts2vec_lr          = params["ts2vec"]["predictor_lr"]
    ts2vec_latent_dims = params["ts2vec"]["ts2vec_latent_dims"]
    ts2vec_epochs      = params["ts2vec"]["ts2vec_epochs"]
    ts2vec_batch_size  = params["ts2vec"]["ts2vec_batch_size"]

    layer1_dim       = params["general_params"]["regressor"]["layer1_dim"]
    layer2_dim       = params["general_params"]["regressor"]["layer2_dim"]
    layer3_dim       = params["general_params"]["regressor"].get("layer3_dim", None)
    dropout          = params["ts2vec"]["predictor_dropout"]
    regressor_epochs = params.get("regressor_epochs", 50)
    lr_regressor     = params.get("lr_regressor", 1e-3)

    num_fed_splits = params["ts2vec_fed"]["num_splits"]
    dim_splitting  = params["ts2vec_fed"]["dim_splitting"] # 0=pages,1=rows,2=features
    dim_concat     = 0 if dim_splitting == 0 else 1

    # ====== PREPARE DATA ======
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor  = torch.tensor(X_test,  dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
    y_test_tensor  = torch.tensor(y_test_scaled,  dtype=torch.float32)

    max_splits = X_train_tensor.shape[dim_splitting]
    if num_fed_splits > max_splits:
        print(f"⚠️ 'WINDOWS_PER_PAGE' ({num_fed_splits}) > size of dim_splitting ({max_splits}) → adjusting.")
        num_fed_splits = max_splits

    X_train_splits = torch.tensor_split(X_train_tensor, num_fed_splits, dim=dim_splitting)
    X_test_splits  = torch.tensor_split(X_test_tensor,  num_fed_splits, dim=dim_splitting)

    if dim_splitting == 0:
        y_train_splits = torch.tensor_split(y_train_tensor, num_fed_splits, dim=0)
        y_test_splits  = torch.tensor_split(y_test_tensor,  num_fed_splits, dim=0)
    else:
        y_train_splits = [y_train_tensor] * num_fed_splits
        y_test_splits  = [y_test_tensor] * num_fed_splits

    # ====== TRAIN LOCAL TS2VEC ENCODERS ======
    ts2vec_encoders = []
    for i in range(num_fed_splits):
        print(f"🧩 Training TS2Vec encoder {i+1}/{num_fed_splits} on split {X_train_splits[i].shape}")
        encoder = TS2VecEncoder(z_pooling=z_pooling_method, lr=ts2vec_lr, device=device, patience=patience)
        encoder.fit(
            X_train_splits[i].cpu().numpy(),
            hidden_dims=ts2vec_hidden_dims,
            output_dims=ts2vec_latent_dims,
            depth=ts2vec_depth,
            batch_size=ts2vec_batch_size,
            n_epochs=ts2vec_epochs)
        ts2vec_encoders.append(encoder)

    # ====== ENCODE TO LATENTS ======
    print("Encoding (X→z)...")
    latents_train = [encode_in_batches(ts2vec_encoders[i], X_train_splits[i]) for i in range(num_fed_splits)]
    latents_test  = [encode_in_batches(ts2vec_encoders[i], X_test_splits[i])  for i in range(num_fed_splits)]

    # ====== TRAIN REGRESSORS PER ENCODER ======
    regression_heads   = []
    test_rmse_per_head = []
    for i in range(num_fed_splits):
        z_train = latents_train[i].to(device)
        z_test  = latents_test[i].to(device)
        y_train = y_train_splits[i].to(device)
        y_test  = y_test_splits[i].to(device)

        embedding_dim = z_train.shape[1]
        head          = make_regression_head(embedding_dim, layer1_dim, layer2_dim, layer3_dim, y_train, dropout, device)
        regression_heads.append(head)

        test_loss = evaluate_regressor(head, z_train, z_test, y_train, y_test, regressor_epochs, lr_regressor)
        test_rmse_per_head.append(float(test_loss))
        print(f"✅ Split {i+1} RMSE: {test_loss:.4f}")

    # ====== CONCATENATE LATENTS ======
    Z_train = torch.cat([z.cpu() for z in latents_train], dim=dim_concat).numpy()
    Z_test  = torch.cat([z.cpu() for z in latents_test],  dim=dim_concat).numpy()

    # ====== EVALUATE LATENTS ======
    ts2vec_fed_losses, rf_model = Preds().evaluate_models_on_dataset(Z_train, y_train_scaled, Z_test, y_test_scaled)
    ts2vec_fed_losses.append(sum(test_rmse_per_head) / len(test_rmse_per_head))
    print(f"dataset: {desired_dataset}, method: ts2vec_fed")
    print("    RMSE   | LinReg | CatBoost | RForest | NN")
    print(f"& {ts2vec_fed_losses[0]:.4f} & {ts2vec_fed_losses[1]:.4f} & {ts2vec_fed_losses[2]:.4f} & {ts2vec_fed_losses[3]:.4f} \\\\ ")
    print(f"R²: {rf_model.score(Z_test, y_test_scaled):.3f}")

In [ ]:
"Test to see why fed-ts2vec does better"

if params["run_console"]["ts2vec_fed"] == True:
    # Single encoder, same total training as federated
    total_epochs = ts2vec_epochs * num_fed_splits  # match total updates
    encoder      = TS2VecEncoder(z_pooling=z_pooling_method, lr=ts2vec_lr, device=device, patience=patience)
    encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
                depth=ts2vec_depth, batch_size=ts2vec_batch_size,
                n_epochs=total_epochs)
    # Encode
    z_train = encoder.encode(X_train, pooling=None).reshape(X_train.shape[0], -1)
    z_test  = encoder.encode(X_test,  pooling=None).reshape(X_test.shape[0], -1)
    z_train_tensor = torch.tensor(z_train, dtype=torch.float32, device=device)
    z_test_tensor  = torch.tensor(z_test, dtype=torch.float32, device=device)
    y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32, device=device)
    y_test_tensor  = torch.tensor(y_test_scaled, dtype=torch.float32, device=device)

    # Create regression head
    embedding_dim   = z_train_tensor.shape[1]
    regression_head = make_regression_head(embedding_dim, layer1_dim, layer2_dim, layer3_dim,
                                        y_train_tensor, dropout, device)
    # Train + Evaluate
    test_loss = evaluate_regressor(regression_head, z_train_tensor, z_test_tensor,
                                   y_train_tensor, y_test_tensor, regressor_epochs, lr_regressor)
    print(f"Single encoder, extended epochs RMSE: {test_loss:.4f}")
    # ============
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # # Move all latents to same device
    # latents_train = [z.to(device) for z in latents_train]
    # latents_test  = [z.to(device) for z in latents_test]
    # y_train_tensor = y_train_tensor.to(device)
    # y_test_tensor  = y_test_tensor.to(device)

    # # Average latents across encoders
    # Z_train_avg = torch.stack(latents_train, dim=0).mean(dim=0)
    # Z_test_avg  = torch.stack(latents_test,  dim=0).mean(dim=0)

    # # Regression head
    # embedding_dim = Z_train_avg.shape[1]
    # regression_head_avg = make_regression_head(
    #     embedding_dim, layer1_dim, layer2_dim, layer3_dim, y_train_tensor, dropout, device)

    # # Train + Evaluate
    # test_loss_avg = evaluate_regressor(
    #     regression_head_avg, Z_train_avg, Z_test_avg,
    #     y_train_tensor, y_test_tensor, regressor_epochs, lr_regressor)
    # print(f"Multiencoder average-latent RMSE: {test_loss_avg:.4f}")


In [ ]:
"""MOMENT (centralized)"""
"there is no regression task in MOMENT (see 'moment_model.task_name'), so we make our own head"

def pad_to_moment_patch_size(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    "MOMENT has a patch embedding layer, so need to pad the input to be a multiple of patch_size"
    pad_len     = (patch_size - x.shape[1] % patch_size) % patch_size
    if pad_len == 0: return x
    zero_padding= torch.zeros(x.shape[0], pad_len, x.shape[2], device=x.device)
    return torch.cat([x, zero_padding], dim=1)

def encode_x_to_z_in_batches(model, X, batch_size=32):
    """Direct encoding X to z (using MOMENT) is too heavy causing OOM, so do in batches"""
    from torch.cuda.amp import autocast
    model.eval()
    all_embeds = []
    use_amp = device == "cuda"  # autocast only on GPU
    with torch.no_grad():
        for i in range(0, X.size(0), batch_size):
            batch = X[i:i+batch_size].to(device)
            if use_amp:
                with autocast(device_type='cuda'):
                    z = model.embed(x_enc=batch).embeddings
            else:
                z = model.embed(x_enc=batch).embeddings#.detach().cpu()
            all_embeds.append(z.detach().cpu())  # move batch to CPU to relieve GPU memory
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    return torch.cat(all_embeds, dim=0)

def train_moment_encoders(encoders: list, heads: list, X_splits: list, y_data,
                          batch_size: int, epochs: int, lr_encoder: float, lr_head: float,
                          unfreeze_last_n: int, fine_tune: bool, device: str):
    """Finetunes the Moment model encoder and prediction head. Trains the pre-trained Moment backbone
    along with a new randomly-initialized prediction head on the task-specific data.
    model: The Moment model instance containing the pre-trained encoder backbone."""
    if isinstance(y_data, torch.Tensor):
        y_splits = [y_data] * len(X_splits)
    else:
        y_splits = y_data

    for i, (encoder, head, X_split, y_split) in enumerate(zip(encoders, heads, X_splits, y_splits)):
    # for i, (encoder, head) in enumerate(zip(encoders, heads)):
        if fine_tune:
            # Unfreeze last N blocks
            num_blocks = len(encoder.encoder.block)
            for j in range(num_blocks - unfreeze_last_n, num_blocks):
                block_name = f"encoder.block.{j}"
                for name, param in encoder.named_parameters():
                    if block_name in name:
                        param.requires_grad = True
            # Unfreeze final_layer_norm
            for name, param in encoder.named_parameters():
                if "final_layer_norm" in name:
                    param.requires_grad = True

        optimizer = torch.optim.AdamW([
            {'params': [p for p in encoder.parameters() if p.requires_grad], 'lr': lr_encoder},
            {'params': head.parameters(), 'lr': lr_head}])

        encoder.train()
        head.train()
        X_split, y_split = X_split.to(device), y_split.to(device)

        for epoch in range(epochs):
            permutation = torch.randperm(X_split.size(0))
            epoch_loss  = 0.0
            for idx in range(0, X_split.size(0), batch_size):
                batch_idx = permutation[idx:idx+batch_size]
                batch_X   = X_split[batch_idx]
                # batch_y   = y_tensor[batch_idx].to(device)
                batch_y = y_split[batch_idx]
                batch_z   = encoder.embed(x_enc=batch_X).embeddings
                preds     = head(batch_z)
                optimizer.zero_grad()
                loss      = criterion(preds, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item() * batch_X.size(0)
            epoch_loss /= X_split.size(0)
            if epoch % 2 == 0:
                print(f"Encoder {i+1}, Epoch {epoch}, Loss: {epoch_loss:.4f}")
        encoder.eval()
        head.eval()

# def forward_last_block(model, x):
#     # run patch embedding + all but last block on CPU
#     x = x.to("cpu")
#     with torch.no_grad():
#         for blk in model.encoder.block[:-1]:
#             x = blk(x)
#     # move to GPU for last block
#     x = x.to(device)
#     x = model.encoder.block[-1](x)
#     return x


if params["run_console"]["moment"] == True:

    # model_type     = params["moment"]["model_type"] # options: classification (= regression), repres_learning
    model_name       = params["moment"]["model_name"]
    reload_model     = params["moment"]["reload_model"]
    epochs           = params["moment"]["epochs_finetune"] # small dataset: 5-20 usually
    lr_moment_head   = params["moment"]["lr_head"]
    lr_moment_encoder= params["moment"]["lr_encoder"]
    batch_size       = params["moment"]["batch_size"]
    unfreeze_last_n  = params["moment"]["unfreeze_last_n"]
    fine_tune        = params["moment"]["fine_tune"]
    dropout          = params["moment"]["predictor_dropout"]

    # ====== PREPARE TENSORS ======
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
    y_test_tensor  = torch.tensor(y_test_scaled, dtype=torch.float32)

    # ====== LOAD MOMENT ======
    moment_model = MOMENTPipeline.from_pretrained(
        f"AutonLab/{model_name}",
        model_kwargs={'task_name': 'regression', 'n_channels': X_train.shape[2]})
    moment_model.to(device)

    # ====== PAD TO PATCH SIZE ======
    patch_size = getattr(moment_model.tokenizer, "patch_size", None) or getattr(moment_model.tokenizer, "patch_len", None)
    if patch_size is None:
        raise ValueError("Cannot find patch size from MOMENT tokenizer")

    # print(f"❕ Split shape before padding: {X_train_tensor.shape}")
    X_train_tensor = pad_to_moment_patch_size(X_train_tensor, patch_size).permute(0, 2, 1)
    # print(f"❕ Split shape after padding: {X_train_tensor.shape}")
    X_test_tensor  = pad_to_moment_patch_size(X_test_tensor, patch_size).permute(0, 2, 1)
    # print(f"After padding: {X_train_tensor.shape=},  {X_test_tensor.shape=}")

    # ====== FREEZE & PARTIAL UNFREEZE ======
    for p in moment_model.parameters():
        p.requires_grad = False

    if fine_tune: #unfreeze
        num_blocks = len(moment_model.encoder.block)
        for i in range(num_blocks - unfreeze_last_n, num_blocks):
            block_name = f"encoder.block.{i}"
            for name, param in moment_model.named_parameters():
                if block_name in name:
                    param.requires_grad = True
        for name, param in moment_model.named_parameters(): # unfreeze final_layer_norm
            if "final_layer_norm" in name:
                param.requires_grad = True
    moment_model.train() if fine_tune else moment_model.eval()

    with torch.no_grad():
        z_sample = moment_model.embed(x_enc=X_train_tensor[:1].to(device)).embeddings

    criterion = nn.MSELoss()
    print(f"🧩 shape before preds: {X_train_tensor.shape}")

    embedding_dim = z_sample.shape[1]

    # ===== Load ====
    # checkpoint_path = f"{interim_data_loc}/moment_finetuned_{desired_dataset}.pth"
    # if load_model and os.path.exists(checkpoint_path):
    #     checkpoint = torch.load(checkpoint_path, map_location=device)
    #     moment_model.load_state_dict(checkpoint['moment_state_dict'])
    #     regressor.load_state_dict(checkpoint['regressor_state_dict'])
    #     print(f"Loaded MOMENT + head from {checkpoint_path}")

    # ====== REGRESSION HEAD ======
    regressor = make_regression_head(embedding_dim, layer1_dim, layer2_dim, layer3_dim,
                                     y_train_tensor, dropout, device)
    train_moment_encoders([moment_model], [regressor], [X_train_tensor], y_train_tensor,
                          batch_size, epochs, lr_moment_encoder, lr_moment_head,
                          unfreeze_last_n, fine_tune, device)
    # regressor.to(device)
    # free_gpu()
    # ====== EVALUATION ======
    moment_model.eval()
    regressor.eval()
    batch_size_embed = 8 if device.type=="cuda" else 2
    with torch.no_grad():
        z_train_final = encode_x_to_z_in_batches(moment_model, X_train_tensor, batch_size=batch_size_embed)
        z_test_final  = encode_x_to_z_in_batches(moment_model, X_test_tensor, batch_size=batch_size_embed)
        train_preds   = regressor(z_train_final.to(device))
        test_preds    = regressor(z_test_final.to(device))
        train_loss    = criterion(train_preds, y_train_tensor.to(device))
        test_loss     = criterion(test_preds, y_test_tensor.to(device))
        train_rmse    = torch.sqrt(train_loss)
        test_rmse     = torch.sqrt(test_loss)
        print(f"Test RMSE: {test_rmse.item():.4f}")

    moment_losses, rf_model = Preds().evaluate_models_on_dataset(z_train_final.cpu().numpy(), y_train_scaled,
                                                                 z_test_final.cpu().numpy(), y_test_scaled)
    moment_losses.append(test_rmse.item())
    print(f"dataset: {desired_dataset}, method: moment")
    print( "    RMSE   | LinReg   |   CatBoost  |   RForest |   NN")
    print(f"& {moment_losses[0]:.4f} & {moment_losses[1]:.4f} & {moment_losses[2]:.4f} & {moment_losses[3]:.4f}")
    print(f"R² (TS2Vec): {rf_model.score(z_test_final.cpu().numpy(), y_test_scaled):.3f}")

    # # Save
    # torch.save({'moment_state_dict':    moment_model.state_dict(),
    #             'regressor_state_dict': regressor.state_dict()},
    #             f"{interim_data_loc}/moment_finetuned_{desired_dataset}.pth")


In [ ]:
# """MOMENT (light + finetune last block only)"""

# def pad_to_moment_patch_size(x: torch.Tensor, patch_size: int) -> torch.Tensor:
#     """Pad input to multiple of patch_size for MOMENT"""
#     pad_len = (patch_size - x.shape[1] % patch_size) % patch_size
#     if pad_len == 0:
#         return x
#     zero_padding = torch.zeros(x.shape[0], pad_len, x.shape[2], device=x.device)
#     return torch.cat([x, zero_padding], dim=1)

# def encode_x_to_z_in_batches(model, X, batch_size=8):
#     """Encode X to z in batches (CPU/GPU safe)"""
#     model.eval()
#     all_embeds = []
#     use_amp = device.type == "cuda"
#     from torch.cuda.amp import autocast
#     with torch.no_grad():
#         for i in range(0, X.size(0), batch_size):
#             batch = X[i:i+batch_size].to(device)
#             if use_amp:
#                 with autocast(device_type='cuda'):
#                     z = model.embed(x_enc=batch).embeddings
#             else:
#                 z = model.embed(x_enc=batch).embeddings
#             all_embeds.append(z.detach().cpu())
#             torch.cuda.empty_cache()
#             torch.cuda.ipc_collect()
#     return torch.cat(all_embeds, dim=0)

# # ===== LOAD MOMENT =====
# moment_model = MOMENTPipeline.from_pretrained(
#     f"AutonLab/{params['moment']['model_name']}",
#     model_kwargs={'task_name': 'regression', 'n_channels': X_train.shape[2]})

# # ===== PAD INPUTS =====
# patch_size = getattr(moment_model.tokenizer, "patch_size", None) or getattr(moment_model.tokenizer, "patch_len", None)
# X_train_tensor = pad_to_moment_patch_size(torch.tensor(X_train, dtype=torch.float32), patch_size).permute(0, 2, 1)
# X_test_tensor  = pad_to_moment_patch_size(torch.tensor(X_test, dtype=torch.float32), patch_size).permute(0, 2, 1)
# y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
# y_test_tensor  = torch.tensor(y_test_scaled, dtype=torch.float32)

# # ===== FREEZE ALL BUT LAST BLOCK =====
# for p in moment_model.parameters():
#     p.requires_grad = False
# for p in moment_model.encoder.block[-1].parameters():
#     p.requires_grad = True
# for name, param in moment_model.named_parameters():
#     if "final_layer_norm" in name:
#         param.requires_grad = True

# # ===== MOVE MODEL TO GPU =====
# moment_model.to(device)

# # ===== CREATE HEAD =====
# with torch.no_grad():
#     emb_dim = moment_model.embed(x_enc=X_train_tensor[:1].to(device)).embeddings.shape[-1]

# regressor = make_regression_head(
#     embedding_dim=emb_dim,
#     layer1_dim=layer1_dim, layer2_dim=layer2_dim, layer3_dim=layer3_dim,
#     y_train_tensor=y_train_tensor,
#     dropout=params['moment']['predictor_dropout'],
#     device=device)

# criterion = nn.MSELoss()

# # ===== TRAIN FUNCTION =====
# def train_last_block_batchwise(model, head, X, y, batch_size, epochs, lr_encoder, lr_head, device):
#     optimizer = torch.optim.AdamW([
#         {'params': [p for p in model.encoder.block[-1].parameters() if p.requires_grad], 'lr': lr_encoder},
#         {'params': head.parameters(), 'lr': lr_head}
#     ])
#     model.train()
#     head.train()
#     for epoch in range(epochs):
#         perm = torch.randperm(X.size(0))
#         epoch_loss = 0.0
#         for idx in range(0, X.size(0), batch_size):
#             batch_idx = perm[idx:idx+batch_size]
#             batch_X = X[batch_idx].to(device)
#             batch_y = y[batch_idx].to(device)
#             # Forward (keyword argument fixed!)
#             batch_z = model.embed(x_enc=batch_X).embeddings
#             preds = head(batch_z)
#             # Backward
#             optimizer.zero_grad()
#             loss = criterion(preds, batch_y)
#             loss.backward()
#             optimizer.step()
#             epoch_loss += loss.item() * batch_X.size(0)
#         epoch_loss /= X.size(0)
#         print(f"Epoch {epoch}, Loss: {epoch_loss:.4f}")

# # ===== TRAIN =====
# train_last_block_batchwise(
#     moment_model, regressor,
#     X_train_tensor, y_train_tensor,
#     batch_size=4,  # adjust if GPU memory is limited
#     epochs=params['moment']['epochs_finetune'],
#     lr_encoder=params['moment']['lr_encoder'],
#     lr_head=params['moment']['lr_head'],
#     device=device)

# # ===== EVALUATION =====
# moment_model.eval()
# regressor.eval()
# with torch.no_grad():
#     z_train_final = encode_x_to_z_in_batches(moment_model, X_train_tensor, batch_size=8)
#     z_test_final  = encode_x_to_z_in_batches(moment_model, X_test_tensor, batch_size=8)
#     train_preds = regressor(z_train_final.to(device))
#     test_preds  = regressor(z_test_final.to(device))
#     train_rmse = torch.sqrt(criterion(train_preds, y_train_tensor.to(device)))
#     test_rmse  = torch.sqrt(criterion(test_preds, y_test_tensor.to(device)))
#     print(f"Test RMSE: {test_rmse.item():.4f}")


In [ ]:
"""MOMENT (federated)"""
if params["run_console"]["moment_fed"] == True:
    model_name       = params["moment"]["model_name"]
    reload_model     = params["moment"]["reload_model"]
    epochs           = params["moment"]["epochs_finetune"] # small dataset: 5-20 usually
    lr_moment_head   = params["moment"]["lr_head"]
    lr_moment_encoder= params["moment"]["lr_encoder"]
    batch_size       = params["moment"]["batch_size"]
    layer1_dim       = params["moment"]["layer1_dim"]
    layer2_dim       = params["moment"]["layer2_dim"]
    unfreeze_last_n  = params["moment"]["unfreeze_last_n"]
    dropout          = params["moment"]["predictor_dropout"]
    WINDOWS_PER_PAGE = params["moment_fed"]["WINDOWS_PER_PAGE"]
    dim_splitting    = params["moment_fed"]["dim_splitting"] # across: 0=pages, 1=rows, 2=features
    dim_concat       = 0 if dim_splitting==0 else 1
    fine_tune        = True
    # load_model       = False

    # ====== PREPARE TENSORS ======
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)#.permute(0, 2, 1)
    X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)#.permute(0, 2, 1)
    y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
    y_test_tensor  = torch.tensor(y_test_scaled, dtype=torch.float32)

    max_splits = X_train_tensor.shape[dim_splitting]
    if WINDOWS_PER_PAGE > max_splits:
        print(f"⚠️: 'WINDOWS_PER_PAGE' ({WINDOWS_PER_PAGE}) > size of dim_splitting ({max_splits} at dim={dim_splitting}). Adjusting WINDOWS_PER_PAGE.")
        WINDOWS_PER_PAGE = max_splits

    X_train_splits_unprocessed = torch.tensor_split(X_train_tensor, WINDOWS_PER_PAGE, dim=dim_splitting)
    X_test_splits_unprocessed  = torch.tensor_split(X_test_tensor, WINDOWS_PER_PAGE, dim=dim_splitting)
    # y_train_splits = torch.tensor_split(y_train_tensor, WINDOWS_PER_PAGE, dim=dim_splitting)

    if dim_splitting == 0:
        y_train_splits = torch.tensor_split(y_train_tensor, WINDOWS_PER_PAGE, dim=0)
    else:
        y_train_splits = [y_train_tensor] * WINDOWS_PER_PAGE

    temp_model = MOMENTPipeline.from_pretrained(f"AutonLab/{model_name}")
    patch_size = getattr(temp_model.tokenizer, "patch_size", None) or getattr(temp_model.tokenizer, "patch_len", None)
    if patch_size is None:
        raise ValueError("Cannot find patch size from MOMENT tokenizer")
    del temp_model # Free up memory

    X_train_splits = []
    X_test_splits  = []
    for s in X_train_splits_unprocessed:
        print(f"❗️ Split shape before padding: {s.shape}")
        padded_s = pad_to_moment_patch_size(s, patch_size) # Pads along Time (dim=1)
        print(f"❗️ Split shape after padding: {padded_s.shape}")
        X_train_splits.append(padded_s.permute(0, 2, 1)) # Permutes to (B, C, T)
    for s in X_test_splits_unprocessed:
        padded_s = pad_to_moment_patch_size(s, patch_size) # Pads along Time (dim=1)
        X_test_splits.append(padded_s.permute(0, 2, 1)) # Permutes to (B, C, T)

    # ---- Initialize MOMENT encoders ----
    moment_encoders  = []
    regression_heads = []
    for i in range(WINDOWS_PER_PAGE):
        n_channels_for_encoder = X_train_splits[i].shape[1]

        print(f"Loading encoder {i+1}/{WINDOWS_PER_PAGE} ...")
        encoder = MOMENTPipeline.from_pretrained(
            f"AutonLab/{model_name}",
            model_kwargs={'task_name': 'regression', 'n_channels': n_channels_for_encoder}).to(device)
        for p in encoder.parameters(): # Freeze all by default
            p.requires_grad = False
        
        print("Shape before embed:", X_train_splits[i][:1].shape)
        embedding_dim   = encoder.embed(x_enc=X_train_splits[i][:1].to(device)).embeddings.shape[1]
        # regression_head = make_regression_head(embedding_dim, layer1_dim, layer2_dim, layer3_dim, y_train_tensor, dropout, device)
        y_for_head      = y_train_splits[i] if dim_splitting == 0 else y_train_tensor
        regression_head = make_regression_head(embedding_dim, layer1_dim, layer2_dim, layer3_dim,
                                               y_for_head, dropout, device)
        regression_heads.append(regression_head)
        moment_encoders.append(encoder)
    criterion = nn.MSELoss()

    for s in X_train_splits:
        print(f"🧩 shape before preds: {s.shape}")

    # train_moment_encoders(moment_encoders, regression_heads, X_train_splits, y_train_tensor,
    #                       batch_size, epochs, lr_moment_encoder, lr_moment_head,
    #                       unfreeze_last_n, fine_tune, device)
    train_moment_encoders(moment_encoders, regression_heads, X_train_splits, y_train_splits,
                          batch_size, epochs, lr_moment_encoder, lr_moment_head,
                          unfreeze_last_n, fine_tune, device)

    # ---- Encode latents ----
    print("Encoding data to latents (X>z)...")
    # latents_train, latents_test = [], []
    # with torch.no_grad():
    #     for i, (encoder, head) in enumerate(zip(moment_encoders, regression_heads)):
    #         z_train = encoder.embed(x_enc=X_train_splits[i].to(device)).embeddings.cpu()
    #         z_test  = encoder.embed(x_enc=X_test_splits[i].to(device)).embeddings.cpu()
    #         latents_train.append(z_train)
    #         latents_test.append(z_test)
    batch_size_embed = 128 if device.type=="cuda" else 32
    latents_train = [encode_x_to_z_in_batches(moment_encoders[i], X_train_splits[i], batch_size=batch_size_embed) 
                     for i in range(WINDOWS_PER_PAGE)]
    latents_test = [encode_x_to_z_in_batches(moment_encoders[i], X_test_splits[i], batch_size=batch_size_embed) 
                    for i in range(WINDOWS_PER_PAGE)]

    # ---- Concatenate latent spaces ----
    Z_train = torch.cat([t.cpu() for t in latents_train], dim=dim_concat).numpy()
    Z_test  = torch.cat([t.cpu() for t in latents_test], dim=dim_concat).numpy()

    # ====== EVALUATION (federated) ======
    # Feed each split into its own regression head, then combine predictions
    for head in regression_heads:
        head.eval()
    for encoder in moment_encoders:
        encoder.eval()
    with torch.no_grad():
        train_preds_list = []
        test_preds_list  = []
        for i in range(WINDOWS_PER_PAGE):
            z_train = latents_train[i].to(device)
            z_test  = latents_test[i].to(device)
            train_preds_list.append(regression_heads[i](z_train))
            test_preds_list.append(regression_heads[i](z_test))
        # Average predictions across splits
        # train_preds = torch.mean(torch.stack(train_preds_list), dim=0)
        # test_preds  = torch.mean(torch.stack(test_preds_list), dim=0)

        if dim_splitting == 0:
            # splits across samples -> concatenate predictions back in sample-order
            train_preds = torch.cat(train_preds_list, dim=0)
            test_preds  = torch.cat(test_preds_list,  dim=0)
        else:
            # splits across features/time -> each head predicts full-sample; average (ensemble)
            train_preds = torch.mean(torch.stack(train_preds_list), dim=0)
            test_preds  = torch.mean(torch.stack(test_preds_list),  dim=0)

        train_loss = criterion(train_preds, y_train_tensor.to(device))
        test_loss  = criterion(test_preds,  y_test_tensor.to(device))
        train_rmse = torch.sqrt(train_loss)
        test_rmse  = torch.sqrt(test_loss)
        print(f"Test RMSE: {test_rmse.item():.4f}")

    # ==== eval latents (z>y)===
    moment_fed_losses, rf_model = Preds().evaluate_models_on_dataset(Z_train, y_train_scaled,
                                                                     Z_test, y_test_scaled)
    moment_fed_losses.append(test_rmse.item())
    print(f"dataset: {desired_dataset}, method: moment")
    print( "    RMSE   | LinReg | CatBoost | RForest | NN")
    print(f"& {moment_fed_losses[0]:.4f} & {moment_fed_losses[1]:.4f} & {moment_fed_losses[2]:.4f} & {moment_fed_losses[3]:.4f} \\\\ ")
    print(f"R²: {rf_model.score(Z_test, y_test_scaled):.3f}")

    # # Save
    # torch.save({'moment_state_dict':    moment_model.state_dict(),
    #             'regressor_state_dict': regressor.state_dict()},
    #             f"{interim_data_loc}/moment_finetuned_{desired_dataset}.pth")


In [ ]:
# testing moment stuff

# print(moment_model.moment.encoder.blocks)  # list of transformer blocks
# for name, param in moment_model.moment.named_parameters():
#     print(name, param.shape)
# print(moment_model.model.encoder.blocks)
# print(moment_model.named_parameters())

# print(len(list(moment_model.encoder.parameters())))
# for name, param in moment_model.encoder.named_parameters():
#     print(name)
# sum(p.requires_grad for p in moment_model.parameters())

# print(patch_size)
# print(moment_model.task_name)
# print(X_train.shape, X_train.shape)
# print(y_train_tensor.shape, y_train_scaled.shape,y_test_tensor.shape, y_test_scaled.shape)

# embeddings = moment_model.embed(x_enc=X_train_tensor).embeddings
# print(embeddings.shape)  # likely [batch, feature_dim] or [batch, seq_len]

# send_discord_message(WEBHOOK_URL, "Moment finished")

# from pprint import pprint

# model_name   = "MOMENT-1-large" #MOMENT-1-base, MOMENT-1-large, 
# moment_model = MOMENTPipeline.from_pretrained(
#     f"AutonLab/{model_name}",
#     model_kwargs={'task_name': 'regression', 'n_channels': X_train.shape[2]})

# print(moment_model.task_name)

# # 2️⃣ Inspect the forward methods / supported tasks
# print("Supported tasks in the model:")
# pprint([attr for attr in dir(moment_model) if callable(getattr(moment_model, attr)) and not attr.startswith("_")])

# Save
# torch.save({'moment_state_dict':    moment_model.state_dict(),
#             'regressor_state_dict': regressor.state_dict()},
#             f"{interim_data_loc}/moment_finetuned_{desired_dataset}.pth")
# # Load
# checkpoint = torch.load(f"{interim_data_loc}/moment_finetuned_{desired_dataset}.pth", map_location=device)
# moment_model.load_state_dict(checkpoint['moment_state_dict'])
# regressor.load_state_dict(checkpoint['regressor_state_dict'])
# moment_model.to(device)
# regressor.to(device)


In [ ]:
"params for running Headsup"
# ===== TS2Vec params ======
z_pooling_method   = "mean"
ts2vec_hidden_dims = 16 # units in each layer (> than latent dim)
ts2vec_latent_dims = 8 # latent dim
ts2vec_depth       = 3 # num layers
ts2vec_batch_size  = 32
ts2vec_epochs      = 5 #20
ts2vec_patience    = 25

# predictor_lr           = 0.009
# predictor_epochs       = 50
predictor_dropout      = 0.05
predictor_hidden_sizes = [32, 48, 64] # latent to y output

# ===== Headsup pretrain/train params =====
set_all_seeds(42)
batch_size_pretrain  = 16
batch_size_train     = 16

decoder_hidden_dims  = [16, 64, 128]
projection_dim       = 16
lr_pretrain          = 1e-3
lr_train             = 1e-4 #1e-3

patience_pretrain    = 15
patience_train       = 6

train_epochs_pretrain= 8
train_epochs_finetune= 3 # aim for 30-50
warmup_frac_pretrain = 0.05 # 5-10% of pretrain steps
warmup_frac_train    = 0.05 # 5-10% of train steps

weights_pretrain     = {"recon":0.1,"contrast":1.0}
weights_train_100    = {"pred": 1.0, "recon": 0.5, "contrast": 0.5} # 100 is the label_fraction
weights_train_50     = {"pred": 1.0, "recon": 0.1, "contrast": 0.1} # 50 is the label_fraction
weights_train_other  = {"pred": 2.0, "recon": 0.0, "contrast": 0.0}

label_fractions = [1.0, 0.5, 0.25, 0.1]

# === augmentations ===
aug1          = "jitter"
aug1_strength = 0.2
aug2          = "mag_warp"
aug2_strength = 0.1

# === files and names ===
TS2VEC_ENCODER_NAME   = f"ts2vec_encoder_{desired_dataset}_{ts2vec_hidden_dims}hiddendims_{ts2vec_depth}layers_{ts2vec_latent_dims}dims_{ts2vec_batch_size}batch_{ts2vec_epochs}epoch.pkl"
TS2VEC_ENCODER_FILE   = os.path.join(interim_data_loc, "ts2vec_encoders", TS2VEC_ENCODER_NAME)

PRETRAIN_ENCODER_NAME = (f'pretrained_encoder_{desired_dataset}_lr{lr_pretrain}_epochs{train_epochs_pretrain}_batch{batch_size_pretrain}'
                         f'_enc{ts2vec_latent_dims}dims_{ts2vec_depth}layers_dec{decoder_hidden_dims}_proj{projection_dim}'
                         f'_warmup{warmup_frac_pretrain}_frac{"_".join(map(str, label_fractions))}.pth')
PRETRAIN_ENCODER_FILE = os.path.join(interim_data_loc, "pretrained_encoders", PRETRAIN_ENCODER_NAME)

EMBEDDING_FILE_NAME   = (f"cached_embeddings_{desired_dataset}_{desired_dataset}_lr{lr_train}_epochs{train_epochs_finetune}_batch{batch_size_train}"
                         f'_enc{ts2vec_latent_dims}dims_{ts2vec_depth}layers_dec{decoder_hidden_dims}_proj{projection_dim}'
                         f'_warmup{warmup_frac_train}.pth')
EMBEDDING_CACHE_FILE  = os.path.join(interim_data_loc, "trained_encoders", EMBEDDING_FILE_NAME)


In [ ]:
"Cellsup: Clustering (L+U), predictor (L), eval (test)"
from methods.cellsup import Cellsup, DeepClusterAndSwav
from encoders.sup_heads import SupHead, MLPHead, _get_orthogonality_penalty, train_sup_head_per_encoder, train_sup_heads_joint
from encoders.latents import Latents

if params["run_console"]["cellsup"] == True:
    # ===== params + prepare data =====
    num_epochs  = train_epochs #params["cellsup"]["num_epochs"]
    hidden_dim  = X_train.shape[2]//2
    AE_lr       = params["cellsup"]["AE_lr"]
    # ===== pretrain section =====
    # Pretraining step: each encoder learns X > z > X_recon. After pretraining, encoder is frozen for downstream tasks

    # ===== pretrain AE variants =====
    ae_encoders = {}
    encoders_dims_list = params["cellsup"]["encoders_dims_list"]
    # for i, latent_dim in enumerate(encoders_dims_list):
    #     ae_model = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, latent_dim], pred_dim=0).to(device)
    #     ae_model = Bootstrapping.train_ae_with_bootstraps(ae_model, X_train, num_epochs=num_epochs, lr=AE_lr,
    #                                         sample_frac=0.8, weight_decay=weight_decay, device=device)
    #     ae_encoders[f"AE_{latent_dim}"] = ae_model

    "even slicing"
    # num_slices = len(encoders_dims_list)
    # for i, latent_dim in enumerate(encoders_dims_list):
    #     X_train_slice   = get_sliced_data(X_train, num_slices, i)
    #     input_dim_slice = X_train_slice.shape[1] * X_train_slice.shape[2]
    #     ae_model = ae.FlexibleAutoencoder(layer_dims=[input_dim_slice, 64, latent_dim], pred_dim=0).to(device)
    #     ae_model = Bootstrapping.train_ae_with_bootstraps(ae_model, X_train_slice, num_epochs=num_epochs,lr=AE_lr,
    #                                         sample_frac=0.8, weight_decay=weight_decay, device=device)
    #     ae_encoders[f"AE_slice{i}_dim{latent_dim}"] = ae_model

    start = time.time()

    "weighted slicing"
    weights  = np.array(encoders_dims_list) / np.sum(encoders_dims_list)
    # X_slices = Slicing.get_weighted_slices(X_train, weights)
    X_slices = Slicing.get_weighted_slices_sqrt(X_train, encoders_dims_list)
    for i, (latent_dim, X_train_slice) in enumerate(zip(encoders_dims_list, X_slices)):
        input_dim_slice = X_train_slice.shape[1] * X_train_slice.shape[2]
        ae_model        = ae.FlexibleAutoencoder(layer_dims=[input_dim_slice, hidden_dim, latent_dim], pred_dim=0).to(device)
        ae_model = Bootstrapping.train_ae_with_bootstraps(ae_model,X_train_slice,num_epochs=train_epochs,lr=AE_lr,
                                                          sample_frac=0.8,weight_decay=weight_decay,device=device)
        ae_encoders[f"AE_slice{i}_dim{latent_dim}"] = ae_model

    # ===== pretrain Denoising AE =====
    # denoise_ae    = ae.DenoisingAE(input_size=input_dim, hidden_dims=[64,16], latent_dim=8,
    #                                dropout_prob=dropout, noise_std=0.1).to(device)
    # optimizer_dae = torch.optim.AdamW(denoise_ae.parameters(), lr=AE_lr, weight_decay=weight_decay)
    # for epoch in range(num_epochs):
    #     optimizer_dae.zero_grad()
    #     X_recon = denoise_ae(X_tensor)
    #     loss    = F.mse_loss(X_recon, X_tensor)
    #     loss.backward()
    #     optimizer_dae.step()

    # ===== assemble encoders =====
    encoders_dict = {**ae_encoders,
                    #  "denoiseAE": denoise_ae,
                    }

    # Add a suphead for each encoder
    # sup_head_rmse = {}
    # for name, encoder in encoders_dict.items():
    #     rmse = train_sup_head_per_encoder(encoder, X_L, y_L, X_test, y_test_scaled, dropout,
    #                                       all_encoders=encoders_dict, reg_ortho=1e-3,   # tune this
    #                                       train_encoder=True, device=device, epochs=num_epochs)
    #     sup_head_rmse[name] = rmse
    #     print(f"{name}: RMSE = {rmse:.4f}")

    # ===== Train sup-heads (optionally finetune encoders) =====
    sup_head_rmse = train_sup_heads_joint(encoders_dict, X_L, y_L, X_test, y_test_scaled,
                                          hidden_sizes=[64,32], lr=AE_lr, epochs=train_epochs,
                                          device=device, train_encoders=True, reg_ortho=0e-3)
    for name, rmse in sup_head_rmse.items():
        print(f"{name}: RMSE = {rmse:.4f}")

    # xxxxxxxxxx Per-encoder evaluation xxxxxxxxxx
    print("Per-encoder CatBoost RMSE:")
    for name, encoder in encoders_dict.items():
        z_train = Latents.get_latent_tensor(encoder, X_L, train_encoder=False, device=device).cpu().numpy()
        z_test  = Latents.get_latent_tensor(encoder, X_test, train_encoder=False, device=device).cpu().numpy()
        _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_train, y_L, z_test, y_test_scaled)
        print(f"   {name}: {rmse:.4f}")

    # ===== Encoder weights =====
    weight_encoding_method = "inverse_rmse"  # "uniform", "inverse_rmse", "softmax"
    encoder_weights = assign_encoder_weights(encoders_dict, sup_head_rmse, weight_encoding_method)

    # ===== ensemble clustering =====
    n_clusters = 8
    ensemble_clusters = Cellsup(encoders_dict=encoders_dict, n_clusters=n_clusters,
                                device=device, cluster_assignment="soft", cluster_metric="ch")

    # """§0 BASELINE: Pure supervised on latents (no clustering, no pseudo-labels)"""
    # z_train_concat = Latents.get_weighted_latents(encoders_dict, X_L, encoder_weights, device=device)
    # z_test_concat  = Latents.get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)
    # _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_train_concat, y_L, z_test_concat, y_test_scaled)

    # print(f"  >> §0 latent (no cluster z>y) CatBoost RMSE: {rmse:.4f}")
    # linreg_loss0, catboost_loss0, unsupervised_rmse0, rf_rmse0 = \
    #     Preds.evaluate_models_on_dataset(z_train_concat, y_L, z_test_concat, y_test_scaled)

    """§1 Clustering (clusters > pseudo-labels > RMSE)"""
    ensemble_clusters.encoders_dict = encoders_dict
    print("Encoders used for pseudo-labels:", list(ensemble_clusters.encoders_dict.keys()))
    X_all_aug    = np.concatenate([X_L, X_U], axis=0)
    z_all_concat = Latents.get_weighted_latents(encoders_dict, X_all_aug, encoder_weights, device=device)
    z_test_concat= Latents.get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)

    ensemble_clusters.fit_kmeans_on_encoder_latents(X_L, encoder_weights=encoder_weights, cluster_range=(cluster_min, cluster_max))
    end   = time.time()
    print(f"Training time for {train_epochs} epochs: {(end - start):.2f} s")

    y_U_pseudo   = ensemble_clusters.assign_pseudo_labels(X_L, y_L, X_U, confidence_thresh=0)
    y_all_aug    = np.concatenate([y_L, y_U_pseudo], axis=0)
    _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    print(f"  >> §1 Semi-supervised latent+cluster CatBoost RMSE: {rmse:.4f}")
    cellsup_losses, _ = Preds().evaluate_models_on_dataset(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    """§2 DeepCluster (z > clusters > rmse)"""
    # ensemble_clusters.cluster_prob_matrix = None
    # multiview_bool = True
    # # if len(encoders_dict) == 1:
    # #     multiview_bool = False

    # if multiview_bool:
    #     X_U_torch    = torch.tensor(X_U, dtype=torch.float32, device=device)
    #     X_U_view1, X_U_view2 = make_two_views_augmentation(X_U_torch, device, scale=0.1)
    #     X_U_aug      = torch.cat([X_U_view1, X_U_view2], dim=0).cpu().numpy()
    #     ensemble_clusters.deepcluster_step_swav(X_U_aug, n_iters=swav_iters, cluster_range=(4, 16),
    #                                             temperature=swav_temp, refine_encoder=False)
    # else:
    #     ensemble_clusters.deepcluster_step_swav(X_U, n_iters=swav_iters, cluster_range=(4,16),
    #                                             temperature=swav_temp, refine_encoder=False)

    # swav_feats_U          = ensemble_clusters.cluster_prob_matrix  # now shape (N_unlabeled, sum_k)
    # encoder_cluster_sizes = [ensemble_clusters.clusterers[name].n_clusters for name in encoders_dict]
    # start = 0
    # per_encoder_means = []
    # for k in encoder_cluster_sizes:
    #     per_encoder_means.append(np.mean(swav_feats_U[:, start:start+k], axis=1, keepdims=True))
    #     start += k
    # y_dim      = y_L.shape[1]
    # y_U_pseudo = np.mean(np.concatenate(per_encoder_means, axis=1), axis=1, keepdims=True)  # (N_unlabeled, 1)
    # y_U_pseudo = y_U_pseudo[:len(X_U)]  

    # y_U_pseudo_full = np.tile(y_U_pseudo, (1, y_dim))  # (N_unlabeled, y_dim)
    # X_all_aug       = np.concatenate([X_L, X_U], axis=0)
    # y_all_aug       = np.concatenate([y_L, y_U_pseudo_full], axis=0)
    # z_all_concat    = get_weighted_latents(encoders_dict, X_all_aug, encoder_weights, device=device)
    # z_test_concat   = get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)

    # _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    # print(f"  >> §2 DeepCluster latent+cluster CatBoost RMSE: {rmse:.4f}")
    # swav_losses = Preds.evaluate_models_on_dataset(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    # """§3 Barlow Twins (SSL consistency regularizer on unlabeled data)"""
    # print(">> Running §3 Barlow Twins consistency step")
    # z_view1_concat = get_weighted_latents(encoders_dict, X_U_view1.cpu().numpy(), encoder_weights, device=device)
    # z_view2_concat = get_weighted_latents(encoders_dict, X_U_view2.cpu().numpy(), encoder_weights, device=device)

    # # compute BT loss (as regularization indicator, not for training)
    # loss_BT = barlow_twins_loss(
    #     torch.tensor(z_view1_concat, device=device, dtype=torch.float32),
    #     torch.tensor(z_view2_concat, device=device, dtype=torch.float32),)
    # print(f"  >> §3 Barlow Twins unsupervised loss: {loss_BT.item():.4f}")

    # # optionally, use BT consistency as pseudo-supervision
    # z_all_concat  = get_weighted_latents(encoders_dict, np.concatenate([X_L, X_U]), encoder_weights, device=device)
    # z_test_concat = get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)

    # # make pseudo-targets = avg 2 BT views’ means (simple consistency trick)
    # y_U_pseudo      = (z_view1_concat.mean(axis=1, keepdims=True) + z_view2_concat.mean(axis=1, keepdims=True))/2
    # y_U_pseudo_full = np.tile(y_U_pseudo, (1, y_L.shape[1]))
    # y_all_aug       = np.concatenate([y_L, y_U_pseudo_full], axis=0)

    # _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)
    # print(f"  >> §3 Barlow Twins latent+consistency CatBoost RMSE: {rmse:.4f}")
    # linreg_loss3, catboost_loss3, unsupervised_rmse3, rf_rmse3 = \
    #     Preds.evaluate_models_on_dataset(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    print(f"Results for dataset: {desired_dataset}, {label_frac=}")
    print("    RMSE       | LinReg | CatBoost | Cluster | RForest")
    # print(f"& Z (concat)   & {linreg_loss0:.4f} & {catboost_loss0:.4f}   & {unsupervised_rmse0:.4f}  & {rf_rmse0:.4f} \\\\")
    print(f"& Z (pseudo)   & {cellsup_losses[0]:.4f} & {cellsup_losses[1]:.4f}   & {cellsup_losses[2]:.4f} \\\\")
    # print(f"& Z (swav)     & {swav_losses[0]:.4f} & {swav_losses[1]:.4f}   & {swav_losses[2]:.4f}  \\\\")
    # print(f"& Z (Barlow)   & {linreg_loss3:.4f} & {catboost_loss3:.4f}   & {unsupervised_rmse3:.4f} & {rf_rmse3:.4f} \\\\")


In [ ]:
"Load ts2vec latents for clustering"
clustering = False

if clustering == True:
    z_train = np.load(os.path.join(ts2vec_params_loc, z_train_file_name))
    z_test  = np.load(os.path.join(ts2vec_params_loc, z_test_file_name))
    if label_frac < 1 and X_U.shape[0] > 0:
        z_U = np.load(os.path.join(ts2vec_params_loc, z_U_file_name))

    print(f"z_train: {z_train.shape}, z_test: {z_test.shape}, z_U: {z_U.shape}")
    print(f"y_train: {y_train_scaled.shape}, y_test: {y_test_scaled.shape}")

    # z_train_flat = z_train.reshape(z_train.shape[0], -1)
    # z_test_flat  = z_test.reshape(z_test.shape[0], -1)
    # ========================
    # Flatten over time
    def flat(z: np.ndarray) -> np.ndarray:
        # return z.mean(axis=1)  # (N,D)
        return z.reshape(z.shape[0], -1)

    ZL = flat(z_train)        # labeled latents
    ZU = flat(z_U)            # unlabeled latents
    YL = y_train_scaled.squeeze()

    # 1️⃣ Fit clusters on labeled data only
    k  = int(np.sqrt(len(np.unique(YL))))  # tune as needed
    km = KMeans(n_clusters=k, random_state=0).fit(ZL)

    # 2️⃣ Assign clusters to labeled data
    cL = km.predict(ZL)

    # 3️⃣ Map each cluster to its median y
    cluster2y = {c: np.median(YL[cL==c]) for c in range(k)}

    # 4️⃣ Soft pseudo-labels for unlabeled data
    cU   = km.predict(ZU)
    dist = km.transform(ZU)                        # distance to each cluster
    from scipy.special import softmax
    prob = softmax(-dist / dist.std(), axis=1)     # closer clusters = higher weight
    cluster_values = np.array([cluster2y[c] for c in range(k)])
    YU_soft = prob @ cluster_values                # weighted pseudo-labels

    # Optional: confidence mask
    min_dist = dist.min(axis=1)
    conf_mask = min_dist < np.percentile(min_dist, 50)
    ZU_filtered = ZU[conf_mask]
    YU_filtered = YU_soft[conf_mask]

    # 5️⃣ Augment labeled + pseudo-labeled data
    Z_aug = np.concatenate([ZL, ZU_filtered], axis=0)
    Y_aug = np.concatenate([YL, YU_filtered], axis=0).reshape(-1,1)

    # 6️⃣ Predict on test set
    z_test_concat = flat(z_test)
    _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(
        Z_aug, Y_aug, z_test_concat, y_test_scaled)
    print(f"TS2Vec + KMeans soft pseudo-label RMSE: {rmse:.4f}")


In [ ]:
"BARLOW (CNN)"

# class CnnAutoencoder(nn.Module):
#     """Flexible CNN autoencoder for 1D time-series.
#     Allows variable number of Conv+Pool layers."""
#     def __init__(
#         self,
#         n_features: int,
#         n_timesteps: int,
#         latent_dim: int,
#         channels: list[int] = [64, 128],
#         kernel_size: int = 3,
#         pool_kernel: int = 2,):
#         super().__init__()
#         self.n_features  = n_features
#         self.n_timesteps = n_timesteps
#         self.kernel_size = kernel_size
#         self.pool_kernel = pool_kernel
#         self.channels    = channels
#         self.latent_dim  = latent_dim

#         # --- ENCODER ---
#         convs, pools = [], []
#         in_ch = n_features
#         for out_ch in channels:
#             convs.append(nn.Conv1d(in_ch, out_ch, kernel_size))
#             pools.append(nn.MaxPool1d(pool_kernel))
#             in_ch = out_ch
#         self.convs = nn.ModuleList(convs)
#         self.pools = nn.ModuleList(pools)

#         # --- Compute flat size dynamically ---
#         with torch.no_grad():
#             x = torch.zeros(1, n_timesteps, n_features).permute(0, 2, 1)
#             for conv, pool in zip(self.convs, self.pools):
#                 x = pool(F.relu(conv(x)))
#             self.flat_size = x.numel()
#             self.final_channels = x.shape[1]
#             self.final_time = x.shape[2]

#         self.enc_linear = nn.Linear(self.flat_size, latent_dim)
#         self.dec_linear = nn.Linear(latent_dim, self.flat_size)

#         # --- DECODER ---
#         deconvs = []
#         rev_channels = channels[::-1]
#         for i in range(len(rev_channels) - 1):
#             deconvs.append(nn.ConvTranspose1d(rev_channels[i], rev_channels[i + 1], kernel_size))
#         deconvs.append(nn.ConvTranspose1d(rev_channels[-1], n_features, kernel_size))
#         self.deconvs = nn.ModuleList(deconvs)

#     def encode(self, x: torch.Tensor) -> torch.Tensor:
#         x = x.permute(0, 2, 1)  # (B, F, T)
#         for conv, pool in zip(self.convs, self.pools):
#             x = pool(F.relu(conv(x)))
#         z = self.enc_linear(x.reshape(x.size(0), -1))
#         return z

#     def decode(self, z: torch.Tensor) -> torch.Tensor:
#         x = F.relu(self.dec_linear(z))
#         x = x.view(-1, self.final_channels, self.final_time)
#         for deconv in self.deconvs[:-1]:
#             x = F.interpolate(x, scale_factor=self.pool_kernel, mode="nearest")
#             x = F.relu(deconv(x))
#         x = F.interpolate(x, scale_factor=self.pool_kernel, mode="nearest")
#         x = self.deconvs[-1](x)
#         diff = self.n_timesteps - x.shape[2]
#         if diff > 0:
#             x = F.pad(x, (0, diff))
#         elif diff < 0:
#             x = x[:, :, :self.n_timesteps]
#         return x.permute(0, 2, 1)

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         return self.decode(self.encode(x))

from encoders.cnn import CnnAutoencoder

def encode_in_batches(model, X_tensor, batch_size=64):
    model.eval()
    embeddings = []
    with torch.no_grad():
        for i in range(0, X_tensor.size(0), batch_size):
            batch = X_tensor[i:i+batch_size].to(device)
            embeddings.append(model.encode(batch).detach().cpu())
    return torch.cat(embeddings, dim=0)

SSL_LAMBDA  = params["barlow"]["cnn"]["SSL_LAMBDA"] # value from BT paper
SSL_WEIGHT  = params["barlow"]["cnn"]["SSL_WEIGHT"] # Weight for recon vs. SSL
CHANNELS_1  = params["barlow"]["cnn"]["CHANNELS_1"]
CHANNELS_2  = params["barlow"]["cnn"]["CHANNELS_2"]
KERNEL_SIZE = params["barlow"]["cnn"]["KERNEL_SIZE"]
POOL_KERNEL = params["barlow"]["cnn"]["POOL_KERNEL"]
latent_dim  = params["barlow"]["cnn"]["latent_dim"]
augment_const_cnn = params["barlow"]["cnn"]["augment_const"]

if params["run_console"]["barlow_cnn"] == True:
    n_timesteps = X_train.shape[1]
    n_features  = X_train.shape[2]

    # cnn_ae    = CnnAutoencoder(n_features, n_timesteps, latent_dim,
    #                            CHANNELS_1, CHANNELS_2, KERNEL_SIZE, POOL_KERNEL).to(device)
    cnn_ae    = CnnAutoencoder(n_features, n_timesteps, latent_dim, channels=[CHANNELS_1, CHANNELS_2], 
                               kernel_size=KERNEL_SIZE, pool_kernel=POOL_KERNEL).to(device)

    optimizer = torch.optim.AdamW(cnn_ae.parameters(), lr=AE_lr)
    # X_tensor must now be the 3D sequence data (B, T, F)
    X_tensor  = torch.tensor(X_train, dtype=torch.float32, device=device) # NO .reshape(len(X_train), -1)
    cnn_ae.train()
    start     = time.time()

    # ===== CNN ENCODER + Barlow Twins SSL =====
    for epoch in range(train_epochs):
        optimizer.zero_grad()

        v1, v2 = make_two_views_augmentation(X_tensor, device, augment_const_cnn)
        z1 = cnn_ae.encode(v1) # Z1 shape: (B, latent_dim)
        z2 = cnn_ae.encode(v2) # Z2 shape: (B, latent_dim)
        
        # 3. Barlow Twins Loss
        B, D     = z1.shape
        z1_norm  = (z1 - z1.mean(0)) / (z1.std(0) + 1e-12)
        z2_norm  = (z2 - z2.mean(0)) / (z2.std(0) + 1e-12)
        xcorr    = (z1_norm.T @ z2_norm) / B
        on_diag  = torch.diagonal(xcorr).add_(-1).pow(2).sum()
        off_diag = (xcorr - torch.diag(torch.diagonal(xcorr))).pow(2).sum()
        ssl_loss = on_diag + SSL_LAMBDA * off_diag

        # 4. Final Loss
        # X_recon    = cnn_ae(X_tensor) 
        # recon_loss = F.mse_loss(X_recon, X_tensor) # Now shapes match: (B, T, F) vs (B, T, F)
        loss_total = SSL_WEIGHT * ssl_loss #+ recon_loss
        loss_total.backward()
        optimizer.step()
        if (epoch + 1) % 10 == 0 or epoch == train_epochs-1:
            print(f"Epoch {epoch+1}/{train_epochs} - Loss: {loss_total.item():.4f}")
    cnn_ae.eval()
    end   = time.time()
    print(f"Training time for {train_epochs} epochs: {(end - start):.2f} s")

    # ===== GET LATENT VECTORS FOR CATBOOST =====
    X_L_tensor    = torch.tensor(X_L, dtype=torch.float32, device=device)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32, device=device)

    with torch.no_grad():
        z_train = cnn_ae.encode(X_L_tensor).cpu().numpy()
        z_test  = cnn_ae.encode(X_test_tensor).cpu().numpy()
    # z_train = encode_in_batches(cnn_ae, X_L_tensor)
    # z_test  = encode_in_batches(cnn_ae, X_test_tensor)
    # z_train_np = z_train.numpy() if isinstance(z_train, torch.Tensor) else z_train
    # z_test_np  = z_test.numpy()  if isinstance(z_test, torch.Tensor)  else z_test

    # ===== EVALUATE =====
    barlow_cnn_losses, rf_model = Preds().evaluate_models_on_dataset(z_train, y_L, z_test, y_test_scaled)

    # --- 4️⃣ Train + Evaluate regressor ---
    embedding_dim   = z_train.shape[1]
    regression_head = make_regression_head(embedding_dim, layer1_dim, layer2_dim, layer3_dim, y_L, dropout, device)
    test_loss       = evaluate_regressor(regression_head, z_train, z_test, y_L, y_test_scaled, regressor_epochs, lr_regressor)
    barlow_cnn_losses.append(test_loss)

    print(f"dataset: {desired_dataset}, method: barlow")
    print( "    RMSE      | LinReg | CatBoost | RForest | NN")
    print(f"& Z (barlow) & {barlow_cnn_losses[0]:.4f} & {barlow_cnn_losses[1]:.4f}  & {barlow_cnn_losses[2]:.4f} & {barlow_cnn_losses[3]:.4f} \\ ")
    print(f"R²: {rf_model.score(z_test, y_test_scaled):.3f}")



In [ ]:
"BARLOW + AE"

if params["run_console"]["barlow_ae"] == True:
    latent_dim   = params["barlow"]["ae"]["latent_dim"]
    ssl_weight   = params["barlow"]["ae"]["ssl_weight"]
    augment_const_ae= params["barlow"]["ae"]["augment_const"]

    input_dim = X_train.shape[1] * X_train.shape[2] if X_train.ndim == 3 else X_train.shape[1]
    X_tensor  = torch.tensor(X_train, dtype=torch.float32, device=device).reshape(len(X_train), -1)

    # ae_model  = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, latent_dim], pred_dim=0).to(device)
    ae_model  = ae.FlexibleAutoencoder(layer_dims=[input_dim, input_dim//4, input_dim//12, latent_dim], pred_dim=0).to(device)
    optimizer = torch.optim.AdamW(ae_model.parameters(), lr=AE_lr)

    # ===== TRAIN AE WITH Barlow Twins SSL =====
    start   = time.time()
    ae_model.train()
    for epoch in tqdm(range(train_epochs)):
        optimizer.zero_grad()
        
        X_recon    = ae_model(X_tensor)
        recon_loss = F.mse_loss(X_recon, X_tensor)
        X3d     = X_train if X_train.ndim == 3 else X_train.reshape(len(X_train), -1, 1)
        X3d     = torch.tensor(X3d, dtype=torch.float32, device=device)
        v1, v2  = make_two_views_augmentation(X3d, device, augment_const_ae)
        v1_flat = v1.reshape(len(v1), -1)
        v2_flat = v2.reshape(len(v2), -1)
        assert v1_flat.shape[1] == input_dim, f"Expected {input_dim}, got {v1_flat.shape[1]}"

        z1 = ae_model.encode(v1_flat)
        z2 = ae_model.encode(v2_flat)

        # Barlow Twins
        B, D     = z1.shape
        z1_norm  = (z1 - z1.mean(0)) / (z1.std(0) + 1e-12)
        z2_norm  = (z2 - z2.mean(0)) / (z2.std(0) + 1e-12)
        xcorr    = (z1_norm.T @ z2_norm) / B
        on_diag  = torch.diagonal(xcorr).add_(-1).pow(2).sum()
        off_diag = (xcorr - torch.diag(torch.diagonal(xcorr))).pow(2).sum()
        ssl_loss = on_diag + SSL_LAMBDA * off_diag
        loss     = recon_loss + ssl_weight * ssl_loss
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}/{train_epochs} - Recon: {recon_loss.item():.4f} SSL: {ssl_loss.item():.4f}")

    end   = time.time()
    print(f"Training time for {train_epochs} epochs: {(end - start):.2f} s")

    ae_model.eval()
    z_train = get_latent_from_encoder(ae_model, X_L, device=device)
    z_test  = get_latent_from_encoder(ae_model, X_test, device=device)

    # ===== TRAIN CATBOOST & EVALUATE =====
    _, rmse = train_and_eval_catboost(z_train, y_L, z_test, y_test_scaled)
    print(f"AE+Barlow latent CatBoost RMSE: {rmse:.4f}")

    _, y_pred_barlow, _, _ = Preds.predict_catboost_multioutput(z_train, y_L, z_test, y_test_scaled)

    barlow_ae_losses, _ = Preds().evaluate_models_on_dataset(z_train, y_L, z_test, y_test_scaled)
    print(f"Results:\nLinReg: {barlow_ae_losses[0]:.4f} | CatBoost: {barlow_ae_losses[1]:.4f} | RForest: {barlow_ae_losses[2]:.4f}")
    print(f" & \\val{{{barlow_ae_losses[0]:.3f}}}{{}}"
        f" & \\val{{{barlow_ae_losses[1]:.3f}}}{{}}"
        f" & \\val{{{barlow_ae_losses[2]:.3f}}}{{}} \\\\")


In [ ]:
"BARLOW + AE (SSL)"

if params["run_console"]["barlow_ae_ssl"] == True:
    # ==== BARLOW TWINS + AE (Semi-Supervised) ====
    latent_dim = 32
    ssl_weight = 1.0
    sup_weight = 0.5  # weight for supervised fine-tuning
    SSL_LAMBDA = 0.005

    # ===== Tensors =====
    # Use only labeled subset
    X_L_tensor = torch.tensor(X_L, dtype=torch.float32, device=device).reshape(len(X_L), -1)
    y_L_tensor = torch.tensor(y_L, dtype=torch.float32, device=device)
    if y_L_tensor.ndim == 1:
        y_L_tensor = y_L_tensor.view(-1, 1)

    # ===== Model =====
    ae_model = ae.FlexibleAutoencoder(
        layer_dims=[input_dim, input_dim//4, input_dim//12, latent_dim],pred_dim=1).to(device)
    optimizer = torch.optim.AdamW(ae_model.parameters(), lr=AE_lr)

    # 1️⃣ Stage 1 — Self-Supervised Pretraining (Barlow Twins)
    print("\n=== Stage 1: Self-Supervised Pretraining (Barlow Twins) ===")
    ae_model.train()
    start = time.time()
    for epoch in range(train_epochs):
        optimizer.zero_grad()

        # reconstruction loss
        X_recon = ae_model(X_tensor, mode="reconstruct")
        recon_loss = F.mse_loss(X_recon, X_tensor)

        # augmentations for Barlow Twins
        X3d     = X_train if X_train.ndim == 3 else X_train.reshape(len(X_train), -1, 1)
        X3d     = torch.tensor(X3d, dtype=torch.float32, device=device)
        v1, v2  = make_two_views_augmentation(X3d, device, 0.1)
        v1_flat = v1.reshape(len(v1), -1)
        v2_flat = v2.reshape(len(v2), -1)

        # encodings
        z1 = ae_model.encode(v1_flat)
        z2 = ae_model.encode(v2_flat)

        # Barlow Twins loss
        B, D     = z1.shape
        z1_norm  = (z1 - z1.mean(0)) / (z1.std(0) + 1e-12)
        z2_norm  = (z2 - z2.mean(0)) / (z2.std(0) + 1e-12)
        xcorr    = (z1_norm.T @ z2_norm) / B
        on_diag  = torch.diagonal(xcorr).add_(-1).pow(2).sum()
        off_diag = (xcorr - torch.diag(torch.diagonal(xcorr))).pow(2).sum()
        ssl_loss = on_diag + SSL_LAMBDA * off_diag

        # combined SSL + AE loss
        loss = recon_loss + ssl_weight * ssl_loss
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}/{train_epochs} - Recon: {recon_loss.item():.4f} SSL: {ssl_loss.item():.4f}")
    end = time.time()
    print(f"Training time for {train_epochs} epochs: {(end - start):.2f} s")

    # 2️⃣ Stage 2 — Supervised Fine-Tuning (on labeled subset)
    print("\n=== Stage 2: Supervised Fine-Tuning ===")
    for epoch in range(max(5, train_epochs // 2)):
        optimizer.zero_grad()
        y_pred = ae_model(X_L_tensor, mode="predict")  # use forward(mode="predict")
        sup_loss = F.mse_loss(y_pred, y_L_tensor)      # shapes now match
        sup_loss.backward()
        optimizer.step()
        print(f"Fine-tune {epoch+1} - Supervised Loss: {sup_loss.item():.4f}")

    # 3️⃣ Evaluation
    ae_model.eval()
    z_train = get_latent_from_encoder(ae_model, X_L, device=device)
    z_test  = get_latent_from_encoder(ae_model, X_test, device=device)

    barlow_ae_ssl_loss, _ = Preds().evaluate_models_on_dataset(z_train, y_L, z_test, y_test_scaled)

    print(f"\n=== Results ===\nLinReg: {barlow_ae_ssl_loss[0]:.4f} | CatBoost: {barlow_ae_ssl_loss[1]:.4f} | "
        f"Cluster: {barlow_ae_ssl_loss[2]:.4f} | RForest: {barlow_ae_ssl_loss[2]:.4f}")
    print(f" & \\val{{{barlow_ae_ssl_loss[0]:.3f}}}{{}}"
        f" & \\val{{{barlow_ae_ssl_loss[1]:.3f}}}{{}}"
        f" & \\val{{{barlow_ae_ssl_loss[2]:.3f}}}{{}} \\\\")


In [ ]:
"Direct pred. (X > y) on dataset with missing labels"
# Train on X_L, predict on X_test (no mention of X_U)

if params["run_console"]["mean_X"] == True:
    X_L_2d     = X_L.mean(axis=1).astype(np.float32)
    X_test_2d  = X_test.mean(axis=1).astype(np.float32)
    start = time.time()
    mean_losses, rf_model_mean= Preds().evaluate_models_on_dataset(X_L_2d, y_L, X_test_2d, y_test_scaled)
    end   = time.time()
    print(f"Pred time for mean: {(end - start):.2f} s")

    # _, _, nn_rmse = preds.predict_mlp_multioutput(X_L_2d, y_L, X_test_2d, y_test_scaled,
    #                                           hidden_layer_sizes=(128, 64), max_iter=500)
    embedding_dim   = X_L_2d.shape[1]
    regression_head = make_regression_head(embedding_dim, layer1_dim, layer2_dim, layer3_dim,
                                           y_L, dropout, device)
    test_loss = evaluate_regressor(regression_head, X_L_2d, X_test_2d,
                                   y_L, y_test_scaled, regressor_epochs, lr_regressor)
    mean_losses.append(test_loss)

    # ====== X last ======
    X_train_last = X_L[:, -1, :].astype(np.float32)
    X_test_last  = X_test[:, -1, :].astype(np.float32)
    last_losses, rf_model_last  = Preds().evaluate_models_on_dataset(X_train_last, y_L, X_test_last, y_test_scaled)

    print(f"Results on raw features ({int(label_frac*100)}% labeled):")
    print("    RMSE   | LinReg | CatBoost | Cluster | RForest | NN")
    print(f"dataset: {desired_dataset}, method: mean")
    print(f"& X (mean) & {mean_losses[0]:.4f} & {mean_losses[1]:.4f}  & {mean_losses[2]:.4f} & {mean_losses[3]:.4f} \\ ")
    print(f"& X (last) & {last_losses[0]:.4f} & {last_losses[1]:.4f}   & {last_losses[2]:.4f}")
    print(f"R²: {rf_model_mean.score(X_test_2d, y_test_scaled):.3f}")
    print(f"R²: {rf_model_last.score(X_test_last, y_test_scaled):.3f}")

#     # ====== X first ======
#     X_train_first = X_L[:, 0, :].astype(np.float32)
#     X_test_first  = X_test[:, 0, :].astype(np.float32)
#     first_losses, _ = Preds().evaluate_models_on_dataset(X_train_first, y_L, X_test_first, y_test_scaled)
#     print(f"& X (first) & {first_losses[0]:.4f} & {first_losses[1]:.4f}   & {first_losses[2]:.4f}")

#     # ====== X random ======
#     n_train, rows, _ = X_L.shape
#     n_test       = X_test.shape[0]
#     train_idx    = np.random.randint(0, rows, size=n_train)
#     test_idx     = np.random.randint(0, rows, size=n_test)
#     X_train_rand = X_train[np.arange(n_train), train_idx, :].astype(np.float32)
#     X_test_rand  = X_test[np.arange(n_test), test_idx, :].astype(np.float32)
#     rand_losses, _ = Preds().evaluate_models_on_dataset(X_train_rand, y_L, X_test_rand, y_test_scaled)
#     print(f"& X (rand)& {rand_losses[0]:.4f} & {rand_losses[1]:.4f}   & {rand_losses[2]:.4f}")

if 1:#params["run_console"]["flatten_X"] == True:
    X_train_flat = X_L.reshape(X_L.shape[0], -1)
    X_test_flat  = X_test.reshape(X_test.shape[0], -1)
    print(X_train_flat.shape, X_test_flat.shape)

    start = time.time()
    flat_mean_losses, rf_model = Preds().evaluate_models_on_dataset(X_train_flat, y_L, X_test_flat, y_test_scaled)
    end   = time.time()
    print(f"Pred time for mean: {(end - start):.2f} s")
    print(f"& flatten(X)& {flat_mean_losses[0]:.4f} & {flat_mean_losses[1]:.4f}   & {flat_mean_losses[2]:.4f}")#
    print(f"R²: {rf_model.score(X_test_flat, y_test_scaled):.3f}")

    # # ====== define FNN ======
    # X_train_t = torch.tensor(X_flat_L, dtype=torch.float32)
    # y_train_t = torch.tensor(y_flat_L, dtype=torch.float32)
    # X_test_t  = torch.tensor(X_test_flat, dtype=torch.float32)
    # y_test_t  = torch.tensor(y_test_scaled, dtype=torch.float32)

    # input_dim  = X_train_t.shape[1]
    # output_dim = y_train_t.shape[1] if y_train_t.ndim > 1 else 1
    # predictor_hidden_dims = [128, 64]
    # predictor_lr      = 1e-3
    # predictor_epochs  = 500
    # predictor_dropout = 0.0
    # nn_predictor = MLPHead(input_dim  = input_dim,
    #                        output_dim = output_dim,
    #                        hidden_sizes = predictor_hidden_dims,
    #                        lr = predictor_lr,
    #                        epochs  = predictor_epochs,
    #                        dropout = predictor_dropout,
    #                        early_stop_patience = 30,
    #                        device  = device)
    # nn_predictor.train(X_train_t, y_train_t, X_test_t, y_test_t)
    # nn_rmse = nn_predictor.evaluate(X_test_t, y_test_t)
    # print(f"NN RMSE (MLPHead, label_frac={label_frac}): {nn_rmse:.4f}")


In [ ]:
"CNN and LSTM solvers"

def X_extract_cnn_features(X, latent_dim=8, channels_1=64, channels_2=64, kernel_size=3, pool_kernel=2,
                         device=device):
    B, T, n_features = X.shape
    X_t = torch.tensor(X, dtype=torch.float32).to(device)

    class CnnEnc(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv1d(n_features, channels_1, kernel_size).to(device)
            self.pool1 = nn.MaxPool1d(pool_kernel)
            self.conv2 = nn.Conv1d(channels_1, channels_2, kernel_size).to(device)
            self.pool2 = nn.MaxPool1d(pool_kernel)
            self.enc_linear = nn.Linear(1, latent_dim)  # placeholder

            # compute flat size with dummy
            with torch.no_grad():
                dummy = torch.zeros(1, n_features, T, device=device)
                x = self.pool1(F.relu(self.conv1(dummy)))
                x = self.pool2(F.relu(self.conv2(x)))
                self.flat_size = x.numel()
                self.flat_shape = x.shape[1:]
            self.enc_linear = nn.Linear(self.flat_size, latent_dim).to(device)

        def encode(self, x):
            x = x.permute(0, 2, 1)
            x = self.pool1(F.relu(self.conv1(x)))
            x = self.pool2(F.relu(self.conv2(x)))
            z = self.enc_linear(x.view(-1, self.flat_size))
            return z

    cnn = CnnEnc().to(device)
    cnn.eval()
    with torch.no_grad():
        features = cnn.encode(X_t).cpu().numpy()
    return cnn, features

def extract_cnn_features(X, latent_dim=8, channels=[64,64], kernel_size=3, pool_kernel=2,
                         device=device):
    _, T, n_features = X.shape
    X_t = torch.tensor(X, dtype=torch.float32).to(device)
    cnn = CnnAutoencoder(n_features=n_features, n_timesteps=T, latent_dim=latent_dim,
                         channels=channels, kernel_size=kernel_size, pool_kernel=pool_kernel).to(device)
    cnn.eval()
    with torch.no_grad():
        features = cnn.encode(X_t).cpu().numpy()
    return cnn, features

def train_cnn_head(features_train, y_train, features_test, y_test, lr=1e-3, epochs=100, batch_size=32, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_t = torch.tensor(features_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test_t  = torch.tensor(features_test, dtype=torch.float32).to(device)
    y_test_t  = torch.tensor(y_test, dtype=torch.float32).to(device)

    head      = nn.Linear(features_train.shape[1], y_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()

    loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
    head.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            y_pred = head(xb)
            loss   = loss_fn(y_pred, yb)
            loss.backward()
            optimizer.step()
        if epoch % 10 == 0:
            print(f"CNN Head Epoch {epoch}, Loss: {loss.item():.4f}")

    head.eval()
    with torch.no_grad():
        y_pred_test = head(X_test_t).cpu().numpy()
    rmse = np.sqrt(np.mean((y_test - y_pred_test)**2))
    return head, rmse

def extract_lstm_features(X, hidden_size=64, num_layers=1, device=device):
    B, T, n_features = X.shape
    X_t  = torch.tensor(X, dtype=torch.float32).to(device)
    lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, num_layers=num_layers,
                   batch_first=True).to(device)
    lstm.eval()
    with torch.no_grad():
        _, (hn, _) = lstm(X_t)
        features = hn[-1].cpu().numpy()
    return lstm, features

def train_lstm_head(features_train, y_train, features_test, y_test, lr=1e-3, epochs=50, batch_size=32, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_t = torch.tensor(features_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test_t  = torch.tensor(features_test, dtype=torch.float32).to(device)
    y_test_t  = torch.tensor(y_test, dtype=torch.float32).to(device)

    head = nn.Linear(features_train.shape[1], y_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
    head.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            y_pred = head(xb)
            loss = loss_fn(y_pred, yb)
            loss.backward()
            optimizer.step()
        if epoch % 10 == 0:
            print(f"LSTM Head Epoch {epoch}, Loss: {loss.item():.4f}")

    head.eval()
    with torch.no_grad():
        y_pred_test = head(X_test_t).cpu().numpy()
    rmse = np.sqrt(np.mean((y_test - y_pred_test)**2))
    return head, rmse

if params["run_console"]["cnn_lstm"] == True:
    latent_dim     = 32
    latent_dim_lstm= 32
    channels_list = [64, 128]
    kernel_size   = 6#3
    pool_kernel   = 2
    epochs        = 100
    # lr            = 1e-3
    # batch_size = 16

    # ===== CNN =====
    cnn_model, X_train_cnn = extract_cnn_features(X_train, latent_dim=latent_dim, channels=channels_list,
                                                  kernel_size=kernel_size, pool_kernel=pool_kernel, device=device)
    _, X_test_cnn = extract_cnn_features(X_test, latent_dim=latent_dim, channels=channels_list,
                                         kernel_size=kernel_size, pool_kernel=pool_kernel, device=device)
    start = time.time()
    cnn_head, cnn_rmse = train_cnn_head(X_train_cnn, y_L, X_test_cnn, y_test_scaled, epochs=epochs)#, lr=lr)
    print("CNN RMSE:", cnn_rmse)
    cnn_mean_losses, rf_model_cnn = Preds().evaluate_models_on_dataset(X_train_cnn, y_L, X_test_cnn, y_test_scaled)
    end   = time.time()
    print(f"Pred time for mean: {(end - start):.2f} s")

    # ===== LSTM =====
    lstm_model, X_train_lstm = extract_lstm_features(X_train, hidden_size=latent_dim_lstm)
    _, X_test_lstm           = extract_lstm_features(X_test, hidden_size=latent_dim_lstm, device=lstm_model.weight_ih_l0.device)
    lstm_head, lstm_rmse     = train_lstm_head(X_train_lstm, y_L, X_test_lstm, y_test_scaled, epochs=epochs)#, lr=lr)

    start = time.time()
    print("LSTM RMSE:", lstm_rmse)
    lstm_losses, rf_model_lstm = Preds().evaluate_models_on_dataset(X_train_lstm, y_L, X_test_lstm, y_test_scaled)
    end   = time.time()
    print(f"Pred time for mean: {(end - start):.2f} s")

    print(f"& CNN(X)& {cnn_mean_losses[0]:.4f} & {cnn_mean_losses[1]:.4f} & {cnn_mean_losses[2]:.4f}")
    print(f"& LSTM (X) & {lstm_losses[0]:.4f} & {lstm_losses[1]:.4f}   & {lstm_losses[2]:.4f}")
    print(f"R² (CNN): {rf_model_cnn.score(X_test_cnn, y_test_scaled):.3f}")
    print(f"R² (LSTM): {rf_model_lstm.score(X_test_lstm, y_test_scaled):.3f}")


In [ ]:
"Saving to file"
RESULTS_FILE       = "results/results_numbers.json"
LATEX_RESULTS_FILE = "results/latex_results.txt"

if params["run_console"]["mean_X"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "mean(X)", mean_losses, RESULTS_FILE, result_type="rmse")
if params["run_console"]["flatten_X"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "flattened(X)", flat_mean_losses, RESULTS_FILE, result_type="rmse")
if params["run_console"]["timevae"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "TimeVAE", timevae_losses, RESULTS_FILE, result_type="rmse")
    JSONLogger.log_result_to_json(desired_dataset, "TimeVAE", [timevae_recon_loss], RESULTS_FILE, result_type="l_recons")
    JSONLogger.log_result_to_json(desired_dataset, "TimeVAE", [timevae_profiling_metrics], RESULTS_FILE, result_type="profiling")

if params["run_console"]["ts2vec"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "TS2Vec", ts2vec_losses, RESULTS_FILE, result_type="rmse")
if params["run_console"]["ts2vec_fed"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "TS2Vec (fed)", ts2vec_fed_losses, RESULTS_FILE, result_type="rmse")
if params["run_console"]["moment"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "Moment (cent)", moment_losses, RESULTS_FILE, result_type="rmse")
# if params["run_console"]["moment_fed"] == True:
#     log_result_to_json(desired_dataset, f"Moment (fed, dim={dim_splitting})", moment_fed_losses)
if params["run_console"]["cellsup"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "Cellsup", cellsup_losses, RESULTS_FILE, result_type="rmse")
if params["run_console"]["barlow_cnn"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "Barlow (CNN)", barlow_cnn_losses, RESULTS_FILE, result_type="rmse")
if params["run_console"]["cnn_lstm"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "LSTM (X)", lstm_losses, RESULTS_FILE, result_type="rmse")
    JSONLogger.log_result_to_json(desired_dataset, "CNN (X)", cnn_mean_losses, RESULTS_FILE, result_type="rmse")

# ---- Read JSON ----
# data    = JSONLogger.load_json_file_safely(RESULTS_FILE)
# methods = data.get(desired_dataset, {})
# ready_methods = {}

# for method, runs in methods.items():
#     if len(runs) >= num_runs and len(runs) % num_runs == 0:
#         ready_methods[method] = runs
#         print(f"Added {desired_dataset} / {method} to latex results")
#     else:
#         print(f"Not enough runs for {desired_dataset} / {method} yet.")

# if ready_methods:
#     timestamp = datetime.now().strftime("%H:%M")
#     with open(LATEX_RESULTS_FILE, "a") as f:
#         f.write(f"-- {desired_dataset} {timestamp} {data_splitting=} {label_frac=} {window_size=} --\n")
#         for method, runs in ready_methods.items():
#             arr = np.array(runs[-num_runs:])
#             means, stds = arr.mean(axis=0), arr.std(axis=0)
#             line = f"{method} & " + " & ".join(f"\\val{{{m:.3f}}}{{{std:.3f}}}" for m, std in zip(means, stds)) + "\n"
#             f.write(line)

# === Read JSON2
data = JSONLogger.load_json_file_safely(RESULTS_FILE)
methods_by_type = data.get(desired_dataset, {})

for result_type, methods in methods_by_type.items():
    ready_methods = {}
    for method, runs in methods.items():
        if len(runs) >= num_runs and len(runs) % num_runs == 0:
            ready_methods[method] = runs
            print(f"Added {desired_dataset} / {method} / {result_type} to latex results")
    
    if ready_methods:
        timestamp = datetime.now().strftime("%H:%M")
        with open(LATEX_RESULTS_FILE, "a") as f:
            f.write(f"-- {desired_dataset} {timestamp} {data_splitting=} {label_frac=} {window_size=} {result_type=} --\n")
            for method, runs in ready_methods.items():
                arr = np.array(runs[-num_runs:])
                means, stds = arr.mean(axis=0), arr.std(axis=0)
                line = f"{method} & " + " & ".join(f"\\val{{{m:.3f}}}{{{std:.3f}}}" for m, std in zip(means, stds)) + "\n"
                f.write(line)

Notifiers.send_discord_message(WEBHOOK_URL, "Run finished")
# Notifiers.make_beep_sound(times=3, delay=0.2)


In [ ]:
"======== code breaker ========"
1 > f


In [ ]:
import numexpr as ne

a = np.random.rand(1_00_000_000)
b = np.random.rand(1_00_000_000)

# Normal NumPy
start = time.time()
c     = a * np.sin(b) + b**2
end   = time.time()
print(f"NumPy time: {end - start:.4f} s")

# Faster NumExpr
start = time.time()
c2    = ne.evaluate("a * sin(b) + b**2")
end   = time.time()
print(f"NumExpr time: {end - start:.4f} s")


In [ ]:
"KAN net"
if params["run_console"]["kan"] == True:
    from kan import KAN

    # X_train_flat = X_train.reshape(X_train.shape[0], -1)
    # X_test_flat  = X_test.reshape(X_test.shape[0], -1)
    X_train_mean = X_train.mean(axis=1)
    X_test_mean  = X_test.mean(axis=1)

    X_train_t = torch.tensor(X_train_mean, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
    X_test_t  = torch.tensor(X_test_mean, dtype=torch.float32)
    y_test_t  = torch.tensor(y_test_scaled, dtype=torch.float32)

    dataset = {
        'train_input': X_train_t,
        'train_label': y_train_t,
        'test_input': X_test_t,
        'test_label': y_test_t,
        'train_ratio': 0.8}

    model = KAN(width=[X_train_t.shape[1], 128, y_train_t.shape[1]], grid=5, k=3)

    print("Starting training...")
    results = model.fit(
        dataset, 
        opt="LBFGS",
        steps=5,
        lamb=0.01)

    print("\nTraining Complete.")
    print(f"Final training loss: {results['train_loss'][-1]:.4e}")
    print(f"Final test loss: {results['test_loss'][-1]:.4e}")
    with torch.no_grad():
        y_pred_t = model(X_test_t)
    y_pred = y_pred_t.cpu().numpy()

    rmse = root_mean_squared_error(y_test_t.cpu().numpy(), y_pred)
    print(f"Test RMSE: {rmse:.4f}")

    X_train_mean = X_train.mean(axis=1)
    X_test_mean  = X_test.mean(axis=1)

    X_train_t = torch.tensor(X_train_mean, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
    X_test_t  = torch.tensor(X_test_mean, dtype=torch.float32)
    y_test_t  = torch.tensor(y_test_scaled, dtype=torch.float32)

    model = KAN(width=[X_train_t.shape[1], 64, 32, y_train_t.shape[1]], grid=5, k=3)

    results = model.fit(
        {'train_input': X_train_t, 'train_label': y_train_t,
        'test_input': X_test_t,  'test_label': y_test_t,
        'train_ratio': 0.8},
        opt="LBFGS", steps=30, lamb=0.01, update_grid=True)

    with torch.no_grad():
        y_pred_t = model(X_test_t)
    y_pred = y_pred_t.cpu().numpy()

    rmse = root_mean_squared_error(y_test_t.cpu().numpy(), y_pred)
    print(f"[KAN] Test RMSE: {rmse:.4f}")
    model.plot()


In [ ]:
"autocorr"
from statsmodels.tsa.stattools import acf

X = X_milling
P, R, C = X.shape

LAG_MAX             = int(R / 4)  # Maximum lag: R/4 rule of thumb
CONFIDENCE_INTERVAL = 2 / np.sqrt(R) # Significance threshold: 2/sqrt(R)
DOWNSAMPLE_FACTOR   = 2 # Factor for decimation

for p in range(P):
    for c in range(C):
        # Generate an AR(1) series (time dimension is rows)
        series = np.zeros(R)
        series[0] = np.random.randn()
        for r in range(1, R):
            # Strong positive correlation (phi=0.8)
            series[r] = 0.8 * series[r-1] + np.random.randn() * 0.5
        X[p, :, c] = series

print(f"Time Series Length (R): {R}")
print(f"Maximum Lag (LAG_MAX): {LAG_MAX}")
print(f"Confidence Threshold: +/- {CONFIDENCE_INTERVAL:.3f}\n")

# --- 2. Function to Compute ACF for the 3D Array ---
def compute_3d_acf(data, lag_max):
    """Computes ACF for every (page, col) series."""
    P, _, C = data.shape
    # Initialize the result array: (lag_max + 1, P, C)
    # +1 because lag 0 (autocorr=1) is included
    acf_matrix = np.zeros((lag_max + 1, P, C))

    for p in range(P):
        for c in range(C):
            series = data[p, :, c]            
            acf_values = acf(series, nlags=lag_max, fft=False, adjusted=False)
            acf_matrix[:, p, c] = acf_values
    return acf_matrix

ACF_X = compute_3d_acf(X, LAG_MAX)

# --- 4. Downsample X to get X_prime (Decimation) ---
X_prime = X[:, ::DOWNSAMPLE_FACTOR, :] 

R_prime       = X_prime.shape[1]
LAG_MAX_prime = int(R_prime / 4) # Recalculate based on new R
print(f"Downsampled Length (R'): {R_prime}")
print(f"New Maximum Lag (LAG_MAX'): {LAG_MAX_prime}\n")

# --- 5. Compute ACF for Downsampled Data (X_prime) ---
ACF_X_prime = compute_3d_acf(X_prime, LAG_MAX_prime)

# --- 6. Compact Presentation and Change Assessment ---
# A. Compact Presentation: Average ACF
ACF_AVG_X = np.mean(ACF_X, axis=(1, 2))
ACF_AVG_X_prime = np.mean(ACF_X_prime, axis=(1, 2))

# B. Single-Value Metric: Change in Average Lag-1 Correlation
# Lag 1 is the second element (index 1) in the ACF array
AVG_RHO_1_X       = ACF_AVG_X[1]
AVG_RHO_1_X_prime = ACF_AVG_X_prime[1]
DIFF_RHO_1        = np.abs(AVG_RHO_1_X - AVG_RHO_1_X_prime)

# C. Quantify Change: RMSD of Average ACF Curves (truncated to shorter length)
common_lags = min(len(ACF_AVG_X), len(ACF_AVG_X_prime))
RMSD_ACF    = np.sqrt(np.mean((ACF_AVG_X[:common_lags] - ACF_AVG_X_prime[:common_lags])**2))

print("--- RESULTS ---")
print(f"Original Average Lag-1 Autocorr: {AVG_RHO_1_X:.3f}")
print(f"Downsampled Average Lag-1 Autocorr: {AVG_RHO_1_X_prime:.3f}")
print(f"Absolute Change in Avg Lag-1 Autocorr: {DIFF_RHO_1:.3f}")
print(f"RMSD of Average ACF Curves (up to lag {common_lags-1}): {RMSD_ACF:.3f}")
print("Average ACF (Original vs. Downsampled):")
results_df = pd.DataFrame({
    'Lag': np.arange(common_lags),
    'ACF_X_Avg': ACF_AVG_X[:common_lags],
    'ACF_X_prime_Avg': ACF_AVG_X_prime[:common_lags]}).round(3)
print(results_df.head(10)) # Print first 10 lags for brevity


In [ ]:
if 'y_pred' not in locals() and 'y_pred' not in globals():
    _, y_pred, _, _ = Preds().predict_catboost_multioutput(X_L_2d, y_L, X_test_2d, y_test_scaled)


_, y_pred_ts2vec, _, _ = Preds().predict_catboost_multioutput(z_train, y_train_scaled, z_test, y_test_scaled)



# if 'y_pred_rand' not in locals() and 'y_pred_rand' not in globals():
#     _, y_pred_rand, _, _ = Preds().predict_catboost_multioutput(X_train_rand, y_L, X_test_rand, y_test_scaled)
# if 'y_pred_last' not in locals() and 'y_pred_last' not in globals():
#     _, y_pred_last, _, _ = Preds().predict_catboost_multioutput(X_train_last, y_L, X_test_last, y_test_scaled)
# if 'y_pred_first' not in locals() and 'y_pred_first' not in globals():
#     _, y_pred_first, _, _ = Preds().predict_catboost_multioutput(X_train_first, y_L, X_test_first, y_test_scaled)
# # if 'y_pred_barlow_cnn' not in locals() and 'y_pred_barlow_cnn' not in globals():
#     # _, y_pred_barlow_cnn, _, _ = Preds().predict_catboost_multioutput(z_train, y_L, z_test, y_test_scaled)
# if 'y_pred_cellsup' not in locals() and 'y_pred_cellsup' not in globals():
#     _, y_pred_cellsup, _, _ = Preds().predict_catboost_multioutput(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)


wafer_idx     = 21
scatter_points= range(len(y_test_scaled[wafer_idx]))
plt.figure(figsize=(11, 5))
plt.gca().set_axisbelow(True)
# plt.scatter(scatter_points, y_train_scaled[wafer_idx], color='black', label='Train', marker='o')
plt.scatter(scatter_points, y_test_scaled[wafer_idx], color='blue', label='Test', marker='o', s=10)
# plt.scatter(scatter_points, y_pred[wafer_idx], color='red', label='Pred (mean)', marker='o', s=10)

plt.scatter(scatter_points, y_pred_ts2vec[wafer_idx], color='red', label='Pred (mean)', marker='o', s=10)

# plt.scatter(scatter_points, y_pred_rand[wafer_idx], color='yellow', label='Pred (rand)', marker='o')
# plt.scatter(scatter_points, y_pred_last[wafer_idx], color='purple', label='Pred (last)', marker='o', s=10)
# plt.scatter(scatter_points, y_pred_first[wafer_idx], color='cyan', label='Pred (first)', marker='o', s=10)
# plt.scatter(scatter_points, y_pred_cellsup[wafer_idx], color='green', label='Pred (cellsup)', marker='o', s=10)
# plt.scatter(scatter_points, y_pred_barlow_cnn[wafer_idx], color='orange', label='Pred (barlow)', marker='o')
plt.legend(fontsize=8)
plt.minorticks_on()
plt.grid(True, which='both', axis='both', color='gray', alpha=0.3, linewidth=0.5)
plt.xlabel('Site ID')
plt.ylabel('Spatial measur.')
plt.tight_layout()

print(f"Mean | y_train | y_test | y_pred | y_pred_rand | y_cellsup | y_pred_barlow_cnn | y_pred_last | y_pred_first")
print(f"        {y_train_scaled[wafer_idx].mean():.2f} | {y_test_scaled[wafer_idx].mean():.2f} | \
{y_pred[wafer_idx].mean():.2f} |   {y_pred_rand[wafer_idx].mean():.2f}    | \
  {y_pred_cellsup[wafer_idx].mean():.2f}  | {y_pred_barlow_cnn[wafer_idx].mean():.2f} | {y_pred_last[wafer_idx].mean():.2f} | {y_pred_first[wafer_idx].mean():.2f}")


In [ ]:
"""[almost CORRECT, small issue] Headsup (no predictor q)"""

# once the missing fields are correct, then rely on the moved class in another file, import like so:
# from headsup import Headsup

# ===== init encoder model =====
if os.path.exists(TS2VEC_ENCODER_FILE):
    print("Loading cached TS2Vec encoder...")
    with open(TS2VEC_ENCODER_FILE, "rb") as f:
        ts2vec_encoder = pickle.load(f)
else:
    print("Training TS2Vec encoder...")
    ts2vec_encoder = TS2VecEncoder(z_pooling=z_pooling_method, device=device, patience=ts2vec_patience)
    if ts2vec_encoder._stop_early:
        print("Training stopped early due to no improvement.")
    ts2vec_encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
                       depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=ts2vec_epochs)
    with open(TS2VEC_ENCODER_FILE, "wb") as f:
        pickle.dump(ts2vec_encoder, f)

encoder_torch  = TorchWrapper(ts2vec_encoder.ts_model).to(device)
proj_head      = ProjectionHead(input_dim=ts2vec_latent_dims, proj_dim=projection_dim).to(device)
decoder        = Decoder(latent_dim=ts2vec_latent_dims, output_shape=(X_train.shape[1], X_train.shape[2]),
                         hidden_sizes=decoder_hidden_dims).to(device) # doesnt belong to encoder
supervised_head= MLPHead(input_dim=ts2vec_latent_dims, output_dim=y_train_scaled.shape[1],
                         hidden_sizes=predictor_hidden_sizes, dropout=predictor_dropout, device=device)
# ===== run Headsup =====
class Headsup:
    def __init__(self, encoder, proj_head, decoder, supervised_head, device, contrast_temp: float = 0.5,
                 aug1: str = "jitter", aug2: str = "mag_warp", aug1_strength: float = 0.1, aug2_strength: float = 0.1,
                 last_block_lr: float = 1e-3, default_lr: float = 1e-4):
        """Wrapper for pretraining + fine-tuning an encoder with projection, decoder, and supervised head
            encoder: nn.Module
            proj_head: nn.Module
            decoder: nn.Module
            supervised_head: wrapper with .model attribute
            device: torch.device
            contrast_temp: float, temperature for NT-Xent loss
            jitter_strength: float, strength of jitter augmentation
            mag_warp_strength: float, strength of mag_warp augmentation
            last_block_lr: float, learning rate for last block when frac < 0.5
            default_lr: float, default learning rate for other params"""
        self.MIN_BATCH_SIZE   = 2 # for contrastive loss equation
        self.PRINT_EVERY      = 3
        self.lr_min           = 1e-5

        self.encoder          = encoder
        self.proj_head        = proj_head
        self.decoder          = decoder
        self.supervised_head  = supervised_head
        self.device           = device

        self.contrast_temp      = contrast_temp
        self.aug1               = aug1
        self.aug2               = aug2
        self.aug1_strength      = aug1_strength
        self.aug2_strength      = aug2_strength
        self.last_block_lr      = last_block_lr
        self.default_lr         = default_lr

    def _augment(self, X):
        X1 = make_augmentations(X, self.aug1, self.device, self.aug1_strength)
        X2 = make_augmentations(X, self.aug2, self.device, self.aug2_strength)
        return X1, X2

    def _early_stop_check(self, loss_total, best_loss, wait, patience):
        """Stop when loss isnt getting better. Returns updated best_loss, wait counter, and a boolean flag indicating whether to stop."""
        if loss_total < best_loss:
            best_loss = loss_total
            wait = 0
            stop = False
        else:
            wait += 1
            stop = wait >= patience
        return best_loss, wait, stop

    def _make_lr_cos_scheduler(self, optimizer, warmup_steps: int, total_steps: int, min_lr: float):
        """Cosine LR scheduler with linear warmup.
            - optimizer: torch optimizer
            - warmup_steps: steps to linearly ramp up LR
            - total_steps: total training steps
            - min_lr: minimum LR at the end of cosine decay"""
        schedulers = []
        for group in optimizer.param_groups:
            base_lr = group["lr"]

            def lr_lambda(step, base_lr=base_lr):
                if step < warmup_steps:
                    return step / float(max(1, warmup_steps))
                progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
                return (min_lr / base_lr) + (1 - min_lr / base_lr) * 0.5 * (1 + math.cos(math.pi * progress))
            schedulers.append(lr_lambda)
        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=schedulers)

    def _pretrain_single_epoch(self, X_train, batch_size, weights, optimizer, scheduler):
        """Runs one epoch of pretraining on the encoder."""
        loss_recon, loss_contrast, loss_total = 0, 0, 0
        for i in range(0, len(X_train), batch_size):
            X_batch = torch.tensor(X_train[i:i+batch_size], dtype=torch.float32, device=self.device)
            if X_batch.size(0) < self.MIN_BATCH_SIZE:
                continue

            X1, X2    = self._augment(X_batch)
            z1, z2    = self.encoder(X1), self.encoder(X2)
            h1, h2    = self.proj_head(z1).mean(dim=1), self.proj_head(z2).mean(dim=1)
            z1_pooled = z1.mean(dim=1)
            x_recon   = self.decoder(z1_pooled)

            loss_contrast = nt_xent_loss(h1, h2, temperature=self.contrast_temp)
            loss_recon    = F.mse_loss(x_recon, X_batch)
            loss_total    = weights["recon"] * loss_recon + weights["contrast"] * loss_contrast

            optimizer.zero_grad()
            loss_total.backward()
            optimizer.step()
            scheduler.step()
        return loss_recon.item(), loss_contrast.item(), loss_total.item()

    def pretrain(self, X_train, batch_size, epochs, warmup_frac_pretrain, weights=weights_pretrain, lr=lr_pretrain, patience = None):
        """Pretrains encoder with contrastive + reconstruction loss. Pretrain usually has lots of steps, and finetuning has few"""
        optimizer_pretrain = torch.optim.AdamW(list(self.encoder.parameters()) +
                                               list(self.proj_head.parameters()) +
                                               list(self.decoder.parameters()), lr=lr)
        max_steps          = train_epochs_pretrain * (len(X_train) // batch_size)
        warmup_steps       = int(warmup_frac_pretrain * max_steps)
        scheduler_pretrain = self._make_lr_cos_scheduler(optimizer_pretrain, warmup_steps=warmup_steps, total_steps=max_steps, min_lr=self.lr_min)

        best_loss, wait = float("inf"), 0
        for epoch in range(epochs):
            loss_recon, loss_contrast, loss_total = self._pretrain_single_epoch(X_train, batch_size, weights, optimizer_pretrain, scheduler_pretrain)
            if epoch % self.PRINT_EVERY == 0:
                print(f"[Pretrain] Epoch {epoch+1}/{epochs}: "
                      f"recon={loss_recon:.4f}, contrast={loss_contrast:.4f}, total={loss_total:.4f}")

            if patience is not None:
                best_loss, wait, stop_flag = self._early_stop_check(loss_total, best_loss, wait, patience)
                if stop_flag:
                    print(f"Early stopping triggered @ epoch {epoch+1}")
                    break
        return self.encoder, self.proj_head, self.decoder

    def _setup_encoder_optimizer(self, frac: float):
        """Sets encoder layers' requires_grad according to labeled fraction.
        Returns weights for the loss components and optimizer"""
        if frac == 1.0:
            for p in self.encoder.parameters():
                p.requires_grad = True # unfrozen encoder
            weights_train   = weights_train_100 #{"pred": 1.0, "recon": 0.5, "contrast": 0.5}
            params_to_opt   = list(self.encoder.parameters()) + list(self.proj_head.parameters()) + \
                              list(self.decoder.parameters()) + list(self.supervised_head.model.parameters())
            optimizer_train = torch.optim.AdamW(params_to_opt, lr=lr_train)
        elif frac >= 0.5:
            for p in self.encoder.parameters():
                p.requires_grad = False # frozen encoder
            weights_train   = weights_train_50 #{"pred": 1.0, "recon": 0.1, "contrast": 0.1}
            params_to_opt   = list(self.encoder.parameters()) + list(self.proj_head.parameters()) + \
                              list(self.decoder.parameters()) + list(self.supervised_head.model.parameters())
            optimizer_train = torch.optim.AdamW(params_to_opt, lr=lr_train)
        else:
            for name, p in self.encoder.named_parameters():
                p.requires_grad = False # frozen encoder
                if name.startswith("encoder_layers") and "last_block" in name:
                    p.requires_grad = True # unfreeze last block only
            weights_train = weights_train_other #{"pred": 2.0, "recon": 0.0, "contrast": 0.0}

            last_block_params = [p for n, p in self.encoder.named_parameters()
                                 if n.startswith("encoder_layers") and "last_block" in n]
            params_to_opt     = list(self.proj_head.parameters()) + list(self.decoder.parameters()) + \
                                list(self.supervised_head.model.parameters())
            optimizer_train   = torch.optim.AdamW([{"params": last_block_params, "lr": self.last_block_lr}, {"params": params_to_opt, "lr": self.default_lr}])
        return weights_train, optimizer_train

    def _train_single_epoch(self, X_L, y_L, X_train, optimizer_train, scheduler_train, batch_size, weights_train):
        """Runs one epoch of fine-tuning on labeled + unlabeled data (tensor conversion done once)."""
        X_L     = X_L.to(self.device) if not isinstance(X_L, torch.Tensor) else X_L
        y_L     = y_L.to(self.device) if not isinstance(y_L, torch.Tensor) else y_L
        X_train = torch.tensor(X_train, dtype=torch.float32, device=self.device) if not isinstance(X_train, torch.Tensor) else X_train

        loss_pred, loss_recon, loss_contrast, loss_total = 0, 0, 0, 0

        num_batches = (len(X_train) + batch_size - 1) // batch_size
        for i in range(num_batches):
            start = i * batch_size
            end_l = min(start + batch_size, len(X_L))
            end_u = min(start + batch_size, len(X_train))

            X_batch_l = X_L[start:end_l]
            y_batch_l = y_L[start:end_l]
            X_batch_u = X_train[start:end_u]

            if X_batch_l.size(0) < self.MIN_BATCH_SIZE or X_batch_u.size(0) < self.MIN_BATCH_SIZE:
                continue

            # encode
            z_l, z_u = self.encoder(X_batch_l), self.encoder(X_batch_u)
            z_l_pooled, z_u_pooled = z_l.mean(dim=1), z_u.mean(dim=1)

            # supervised loss
            y_hat     = self.supervised_head.model(z_l_pooled)
            loss_pred = F.mse_loss(y_hat, y_batch_l)

            # reconstruction loss
            x_recon_l, x_recon_u = self.decoder(z_l_pooled), self.decoder(z_u_pooled)
            loss_recon = (F.mse_loss(x_recon_l, X_batch_l) + F.mse_loss(x_recon_u, X_batch_u)) / 2

            # contrastive loss
            X1_L, X2_L = self._augment(X_batch_l)
            X1_U, X2_U = self._augment(X_batch_u)
            z1_L, z2_L = self.encoder(X1_L), self.encoder(X2_L)
            z1_U, z2_U = self.encoder(X1_U), self.encoder(X2_U)
            h1_L, h2_L = self.proj_head(z1_L).mean(dim=1), self.proj_head(z2_L).mean(dim=1)
            h1_U, h2_U = self.proj_head(z1_U).mean(dim=1), self.proj_head(z2_U).mean(dim=1)
            loss_contrast = (nt_xent_loss(h1_L, h2_L, self.contrast_temp) + nt_xent_loss(h1_U, h2_U, self.contrast_temp)) / 2

            # backward
            loss_total = weights_train["pred"] * loss_pred + weights_train["recon"] * loss_recon + weights_train["contrast"] * loss_contrast
            optimizer_train.zero_grad()
            loss_total.backward()
            optimizer_train.step()
            scheduler_train.step()
        return loss_pred.item(), loss_recon.item(), loss_contrast.item(), loss_total.item()

    def training_loop(self, X_train, y_train_scaled, X_test, y_test_scaled, batch_size, train_epochs_finetune, warmup_frac_train, label_fractions, patience = None):
        """Fine-tunes encoder + heads over all labeled fractions. Pretrain usually has lots of steps, and finetuning has few"""
        results_dict = {}
        z_train_dict = {}
        z_test_dict  = {}
        y_L_dict     = {}
        for frac in label_fractions:
            n_samples = int(len(X_train) * frac)
            X_L       = torch.tensor(X_train[:n_samples], dtype=torch.float32, device=self.device)
            y_L       = torch.tensor(y_train_scaled[:n_samples], dtype=torch.float32, device=self.device)

            weights_train, optimizer_train = self._setup_encoder_optimizer(frac)
            # scheduler_train = CosineAnnealingLR(optimizer_train, T_max=train_epochs_finetune * (len(X_train)//batch_size), eta_min=self.lr_min)
            max_steps       = train_epochs_finetune * (len(X_train) // batch_size)
            warmup_steps    = int(warmup_frac_train * max_steps)
            scheduler_train = self._make_lr_cos_scheduler(optimizer_train, warmup_steps=warmup_steps, total_steps=max_steps, min_lr=self.lr_min)

            best_loss, wait = float("inf"), 0
            for epoch in range(train_epochs_finetune):
                loss_pred, loss_recon, loss_contrast, loss_total = self._train_single_epoch(
                    X_L, y_L, X_train, optimizer_train, scheduler_train, batch_size, weights_train)
                if epoch % self.PRINT_EVERY == 0:
                    print(f"[Finetune {frac*100:.0f}%] Epoch {epoch+1}/{train_epochs_finetune}: "
                          f"pred={loss_pred:.4f}, recon={loss_recon:.4f}, "
                          f"contrast={loss_contrast:.4f}, total={loss_total:.4f}")
                if patience is not None:
                    best_loss, wait, stop_flag = self._early_stop_check(loss_total, best_loss, wait, patience)
                    if stop_flag:
                        print(f"Early stopping triggered @ epoch {epoch+1} (label frac={frac*100:.0f}%)")
                        break

            #  ====== internal evaluation ======
            self.encoder.eval()
            self.proj_head.eval()
            self.decoder.eval()
            self.supervised_head.model.eval()
            with torch.no_grad():
                z_test             = self.encoder(torch.tensor(X_test, dtype=torch.float32, device=self.device))
                z_test_pooled      = z_test.mean(dim=1)
                y_pred             = self.supervised_head.model(z_test_pooled).cpu().numpy()
                results_dict[frac] = root_mean_squared_error(y_test_scaled, y_pred)

                z_train            = self.encoder(X_L).mean(dim=1).cpu().numpy()
                z_train_dict[frac] = z_train
                z_test_dict[frac]  = z_test_pooled.cpu().numpy()
                y_L_dict[frac]     = y_L.cpu().numpy()
                print(f"Sup. head RMSE ({frac*100:.0f}% labels): {results_dict[frac]:.4f}")
        return results_dict, z_train_dict, z_test_dict, y_L_dict

headsup_model = Headsup(encoder_torch, proj_head, decoder, supervised_head, device,aug1=aug1,
                        aug1_strength=aug1_strength, aug2=aug2, aug2_strength=aug2_strength)
# ===== pretrain ======
if os.path.exists(PRETRAIN_ENCODER_FILE):
    checkpoint = torch.load(PRETRAIN_ENCODER_FILE)
    headsup_model.encoder.load_state_dict(checkpoint["encoder"])
    headsup_model.proj_head.load_state_dict(checkpoint["proj_head"])
    headsup_model.decoder.load_state_dict(checkpoint["decoder"])
    print("Loaded pretrained model.")
else:
    headsup_model.pretrain(X_train, batch_size=batch_size_pretrain, epochs=train_epochs_pretrain,
                           warmup_frac_pretrain=warmup_frac_pretrain, patience=patience_pretrain, weights = weights_pretrain)
    torch.save({"encoder": headsup_model.encoder.state_dict(), "proj_head": headsup_model.proj_head.state_dict(),
                "decoder": headsup_model.decoder.state_dict()}, PRETRAIN_ENCODER_FILE)
    print("Saved pretrained model.")

# ======== train =========
results_dict = {}

if os.path.exists(EMBEDDING_CACHE_FILE):
    with open(EMBEDDING_CACHE_FILE, "rb") as f:
        checkpoint = pickle.load(f)
    results_dict = checkpoint["results_dict"]
    z_train_dict = checkpoint["z_train_dict"]
    z_test_dict  = checkpoint["z_test_dict"]
    y_L_dict     = checkpoint["y_L_dict"]
    print("Loaded finetuned cached embeddings and results.")
else:
    results_dict, z_train_dict, z_test_dict, y_L_dict = headsup_model.training_loop(X_train, y_train_scaled, X_test, y_test_scaled,
                                                                                   batch_size=batch_size_train, patience=patience_train,
                                                                                   train_epochs_finetune=train_epochs_finetune,
                                                                                   warmup_frac_train=warmup_frac_train, label_fractions=label_fractions)
    with open(EMBEDDING_CACHE_FILE, "wb") as f:
        pickle.dump({"results_dict": results_dict, "z_train_dict": z_train_dict, "z_test_dict": z_test_dict, "y_L_dict": y_L_dict}, f)
    print("Saved embeddings and results.")

# ==== downstream / external inference ====
z_train   = z_train_dict[label_frac]
z_test_np = z_test_dict[label_frac]
y_L       = y_L_dict[label_frac]

headsup_loss = Preds.evaluate_models_on_dataset(z_train, y_L, z_test_np, y_test_scaled)

print(f"dataset: {desired_dataset}, method: ts2vec, label_frac: {label_frac}")
print("    RMSE     | LinReg | CatBoost | Cluster | RForest | ElasticNet | NN")
print(f"& Z (ts2vec) & {headsup_loss[0]:.4f} & {headsup_loss[1]:.4f}   & {headsup_loss[2]:.4f} \\\\")


In [ ]:
"[shortcut] MOMENT"

def predict_catboost_multioutput2(X_train: np.ndarray, y_train: np.ndarray,X_test: np.ndarray,
                                  y_test: np.ndarray) -> Tuple[Optional[MultiOutputRegressor], np.ndarray, float]:
    """Train multi-output CatBoost with random projection for speed and predict test set.
    Handles constant targets correctly.
    Returns:
        model: trained MultiOutputRegressor (or None if all targets constant)
        y_pred: predictions on test set
        rmse: RMSE across all targets"""
    y_pred = np.zeros_like(y_test, dtype=float)
    
    # Step 0: Identify non-constant targets
    non_constant_idx = [i for i in range(y_train.shape[1])
                        if not np.all(y_train[:, i] == y_train[0, i])]
    
    if non_constant_idx:
        # Step 1: Fast dimensionality reduction
        rp = GaussianRandomProjection(n_components=2048, random_state=42)
        X_train_rp = rp.fit_transform(X_train)
        X_test_rp  = rp.transform(X_test)
        
        # Step 2: Train minimal CatBoost
        model = MultiOutputRegressor(
            CatBoostRegressor(
                iterations=100,        # minimal for speed
                depth=4,             # shallow tree
                learning_rate=0.1,
                verbose=0,
                thread_count=-1      # all CPU cores
            ),
            n_jobs=-1               # parallel outputs
        )
        # iterations=500, learning_rate=0.1, depth=4

        model.fit(X_train_rp, y_train[:, non_constant_idx])
        y_pred[:, non_constant_idx] = model.predict(X_test_rp)
        
        # Step 3: Fill constant outputs
        for i in range(y_train.shape[1]):
            if i not in non_constant_idx:
                y_pred[:, i] = y_train[0, i]
    else:
        model = None
    
    rmse = root_mean_squared_error(y_test, y_pred)
    return model, y_pred, rmse

if params["run_console"]["moment"] == True:
    from sklearn.decomposition import PCA
    from sklearn.random_projection import GaussianRandomProjection

    linreg_loss       = Preds.predict_linreg(z_train, y_train_scaled, z_test, y_test_scaled)
    print("linreg", linreg_loss)
    model, y_pred, rmse = predict_catboost_multioutput2(z_train, y_train_scaled, z_test, y_test_scaled)
    print("RMSE:", rmse)

    unsupervised_rmse = Preds.cluster_and_label(z_train, y_train_scaled, z_test, y_test_scaled, n_clusters=5)
    print("cluster",unsupervised_rmse)

    # rf_rmse           = Preds.predict_rf_multioutput(z_train, y_train_scaled, z_test, y_test_scaled)
    # print("rf",rf_rmse)
    model  = MultiOutputRegressor(RandomForestRegressor(n_estimators=30, random_state=42, n_jobs=-1))
    model = MultiOutputRegressor(RandomForestRegressor(
            n_estimators=10,    # reduced from 100 to speed up
            max_depth=15,       # limit tree depth to speed up
            min_samples_leaf=2, # prevents overfitting / speeds up slightly
            random_state=42, n_jobs=-1))
    model.fit(z_train, y_train_scaled)
    y_pred = model.predict(z_test)
    rf_rmse = root_mean_squared_error(y_test_scaled, y_pred)
    print("rf",rf_rmse)

    # _, _, el_rmse     = Preds.predict_elasticnet_multioutput(z_train, y_train_scaled, z_test, y_test_scaled, alpha=0.1, l1_ratio=0.5)
    # print("el", el_rmse)

    # predictor_lr      = 0.008
    # predictor_epochs  = 200
    # predictor_dropout = 0.05
    # predictor_hidden_sizes = [256, 128] # latent to y output

    # nn_predictor = MLPHead(input_dim=z_train.shape[1], output_dim=y_train_scaled.shape[1],
    #                            hidden_sizes=predictor_hidden_sizes, lr=predictor_lr,
    #                            epochs=predictor_epochs, dropout=predictor_dropout, device=device)
    # nn_predictor.train(z_train, y_train_scaled, z_test, y_test_scaled)
    # nn_rmse = nn_predictor.evaluate(z_test, y_test_scaled)
    # print("NN:", nn_rmse)


In [ ]:
"4 'ablation' scenarios"
# ===== Prepare 2D / embeddings =====
# # Scenario #1 & #4: direct X→y
X_train_mean = X_train.mean(axis=1).astype(np.float32)
X_test_mean  = X_test.mean(axis=1).astype(np.float32)

n_label = int(label_frac * len(X_train_mean))
X_L, y_L = X_train_mean[:n_label], y_train_scaled[:n_label]

# ===== Scenario #1: Direct supervised on 10% labels =====
linreg_1 = LinearRegression().fit(X_L, y_L)
y_pred_1 = linreg_1.predict(X_test_mean)
rmse_1 = np.sqrt(mean_squared_error(y_test_scaled, y_pred_1))

# ===== Scenario #4: Oracle supervised on 100% labels =====
linreg_4 = LinearRegression().fit(X_train_mean, y_train_scaled)
y_pred_4 = linreg_4.predict(X_test_mean)
rmse_4 = np.sqrt(mean_squared_error(y_test_scaled, y_pred_4))

# ===== Scenario #2: Linear probe on pretrained encoder =====
# Freeze encoder, extract embeddings
with torch.no_grad():
    z_train = custom_model.encoder(torch.tensor(X_train[:n_label], dtype=torch.float32, device=device)).mean(dim=1).cpu().numpy()
    z_test  = custom_model.encoder(torch.tensor(X_test, dtype=torch.float32, device=device)).mean(dim=1).cpu().numpy()

# Train simple predictor on embeddings
linreg_2 = LinearRegression().fit(z_train, y_L)
y_pred_2 = linreg_2.predict(z_test)
rmse_2 = np.sqrt(mean_squared_error(y_test_scaled, y_pred_2))

# ===== Scenario #3: Fine-tune pretrained encoder on 10% labels =====
for p in custom_model.encoder.parameters():
    p.requires_grad = True  # unfreeze encoder

results_dict, _, _, _ = custom_model.training_loop(
    X_train, y_train_scaled, X_test, y_test_scaled,
    batch_size=batch_size_train,
    train_epochs_finetune=train_epochs_finetune,
    warmup_frac_train=warmup_frac_train,
    label_fractions=[label_frac])
rmse_3 = results_dict[label_frac]

# ===== Print RMSE table =====
print(f"Scenario | RMSE")
# print(f"#1 Direct X→y (10% labels): {rmse_1:.4f}")
print(f"#2 Linear Probe (encoder frozen): {rmse_2:.4f}")
print(f"#3 Fine-tune (encoder trainable): {rmse_3:.4f}")
# print(f"#4 Oracle X→y (100% labels): {rmse_4:.4f}")


In [ ]:
"SHAP feature importance cell"
from catboost import CatBoostRegressor, Pool
from sklearn.multioutput import MultiOutputRegressor
from sklearn.feature_selection import SelectKBest, mutual_info_regression
import shap

selector = SelectKBest(mutual_info_regression, k=15, random_state=42)
X_selected = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()]

print(f"Kept {len(selected_features)}/48 features:")
print(selected_features.tolist())

# ------------------------------
# 0️⃣ Handle constant targets
constant_targets = np.where(np.std(y_train_scaled, axis=0) == 0)[0]
print("Targets with zero variance:", constant_targets)
y_train_nonconst = np.delete(y_train_scaled, constant_targets, axis=1)
y_test_nonconst  = np.delete(y_test_scaled, constant_targets, axis=1)

# ------------------------------
# 1️⃣ Aggregate timesteps to reduce dimensionality
X_train_flat = X_train.mean(axis=1)  # stations × sensors
X_test_flat  = X_test.mean(axis=1)

# ------------------------------
# 2️⃣ Train initial multi-output model for feature selection
base_model = CatBoostRegressor(
    iterations=100,
    learning_rate=0.1,
    depth=4,
    task_type="GPU",
    verbose=0,
    random_seed=42
)
multi_model = MultiOutputRegressor(base_model)
multi_model.fit(X_train_flat, y_train_nonconst)

# ------------------------------
# 3️⃣ Feature selection based on importance (average across outputs)
importances_list = [
    estimator.get_feature_importance(Pool(X_train_flat, y_train_nonconst[:, i]))
    for i, estimator in enumerate(multi_model.estimators_)
]
importances = np.mean(importances_list, axis=0)
threshold = np.median(importances)
selected_idx = np.where(importances >= threshold)[0]

X_train_sel = X_train_flat[:, selected_idx]
X_test_sel  = X_test_flat[:, selected_idx]
print("Selected features (sensor indices):", selected_idx)

# ------------------------------
# 4️⃣ Train final multi-output model on selected features
final_base = CatBoostRegressor(
    iterations=100,
    learning_rate=0.1,
    depth=4,
    task_type="GPU",
    verbose=0,
    random_seed=42
)
cat_final = MultiOutputRegressor(final_base)
cat_final.fit(X_train_sel, y_train_nonconst)

# ------------------------------
# 5️⃣ Compute SHAP values per target
shap_values_list = []
for estimator in cat_final.estimators_:
    explainer = shap.TreeExplainer(estimator)
    shap_values_list.append(explainer.shap_values(X_test_sel))

# ------------------------------
# 6️⃣ Aggregate SHAP across targets for one summary plot
shap_values_array = np.array(shap_values_list)  # shape: (n_targets, n_samples, n_features)
mean_abs_shap = np.mean(np.abs(shap_values_array), axis=(0,1))  # mean |SHAP| across targets and samples

# ------------------------------
# 7️⃣ Plot aggregated SHAP
plt.figure(figsize=(10,6))
plt.bar([f"f{i}" for i in selected_idx], mean_abs_shap)
plt.xticks(rotation=90)
plt.ylabel("Mean |SHAP value|")
plt.title("Aggregated feature importance across all targets")
plt.show()


In [ ]:
# ===== heterogeneous encoders =====

class MLPEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim))

    def encode(self, x): return self.net(x)

class SmallConv1DEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        # For flattened input, reshape to (N, C=1, L=input_dim)
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Linear(16, latent_dim)

    def encode(self, x):
        if x.ndim == 2:
            x = x.unsqueeze(1)            # (N,1,L)
        c = self.conv(x)                 # (N,16,1)
        c = c.view(len(c), -1)
        return self.fc(c)

# Example create heterogeneous set
input_dim_flat = input_dim
encoders_dict  = {}

# 1) original AE (kept)
encoders_dict['AE_32']   = ae.FlexibleAutoencoder(layer_dims=[input_dim_flat, 64, 32], pred_dim=0).to(device)
# 2) MLP encoder (different inductive bias)
encoders_dict['MLP_32']  = MLPEncoder(input_dim_flat, 32).to(device)
# 3) small conv encoder
encoders_dict['Conv_16'] = SmallConv1DEncoder(input_dim_flat, 16).to(device)
# 4) denoising AE (keep if useful)
encoders_dict['denoiseAE'] = denoise_ae


In [ ]:
from scipy.stats import spearmanr

def latent_target_corr_multi(z: np.ndarray, y: np.ndarray, max_n: int = 3000) -> List[float]:
    """Compute Spearman correlation between pairwise distances in z
    and absolute differences in each y-dimension.
    z: (n, d) latent array
    y: (n, m) targets
    max_n: subsample size for speed
    
    Returns: list of correlations, length m"""
    n = len(y)
    if n > max_n:
        idx = np.random.choice(n, size=max_n, replace=False)
        z, y = z[idx], y[idx]
    # latent distances
    pdist = np.sqrt(((z[:, None, :] - z[None, :, :])**2).sum(-1)).ravel()
    corrs = []
    for j in range(y.shape[1]):
        ydist = np.abs(y[:, None, j] - y[None, :, j]).ravel()
        corr, _ = spearmanr(pdist, ydist)
        corrs.append(float(corr))
    return corrs

corrs = latent_target_corr_multi(z_train, y_L)
print(f"Median corr: {np.nanmedian(corrs):.4f}")
print(f"First 10 corrs: {corrs[:10]}")
# print(z_train.shape, y_L.shape, z_test.shape, y_test_scaled.shape)


In [ ]:
"[OLD] General Prediction"
label_frac = 1  # fraction of training data

print(f"dataset: {desired_dataset}_{row_frac}, label_frac: {label_frac}")
print(f" RMSE ({label_frac*100}%) | LinReg | CatBoost | Cluster | RForest | ElasticNet")

# ====== X mean ======
X_train_2d = X_train.mean(axis=1).astype(np.float32)
X_test_2d  = X_test.mean(axis=1).astype(np.float32)
linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse = Preds.evaluate_models_on_dataset(X_train_2d, y_train_scaled,
                                                                                          X_test_2d, y_test_scaled)
print(f"& mean(X)   & {linreg_loss:.4f} & {catboost_loss:.4f}   & {unsupervised_rmse:.4f}  & {rf_rmse:.4f} & - \\\\")

# ====== X last ======
X_train_2d = X_train[:, -1:, :].mean(axis=1).astype(np.float32)
X_test_2d  = X_test[:, -1:, :].mean(axis=1).astype(np.float32)
linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse = Preds.evaluate_models_on_dataset(X_train_2d, y_train_scaled,
                                                                                          X_test_2d, y_test_scaled)
print(f"& last(X)   & {linreg_loss:.4f} & {catboost_loss:.4f}   & {unsupervised_rmse:.4f}  & {rf_rmse:.4f}  & - \\\\")

# ====== X random ======
n_train, rows, _ = X_train.shape
n_test     = X_test.shape[0]
train_idx  = np.random.randint(0, rows, size=n_train)
test_idx   = np.random.randint(0, rows, size=n_test)
X_train_2d = X_train[np.arange(n_train), train_idx, :].astype(np.float32)
X_test_2d  = X_test[np.arange(n_test), test_idx, :].astype(np.float32)
linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse = Preds.evaluate_models_on_dataset(X_train_2d, y_train_scaled,
                                                                                          X_test_2d, y_test_scaled)
print(f"& random(X) & {linreg_loss:.4f} & {catboost_loss:.4f}   & {unsupervised_rmse:.4f}  & {rf_rmse:.4f}  & - \\\\")



In [ ]:
"CELLSUP: NO CLUSTERS"

# ===== params + prepare data =====
label_frac  = 1  # labelled fraction of training data
num_epochs  = 1
AE_lr       = 1e-3
weight_decay= 1e-5
dropout     = 0.1
n_samples   = int(label_frac * len(X_train))
X_L = X_train[:n_samples]  # labelled X
X_U = X_train[n_samples:]  # unlabelled X
y_L = y_train_scaled[:n_samples]  # labelled y

X_all     = np.concatenate([X_L, X_U], axis=0)
input_dim = X_all.shape[1] * X_all.shape[2] if X_all.ndim == 3 else X_all.shape[1]
X_tensor  = torch.tensor(X_train, dtype=torch.float32, device=device).reshape(len(X_train), -1)

# ===== pretrain section =====
# Pretraining step: each encoder learns X > z > X_recon. After pretraining, encoder is frozen for downstream tasks

# ===== pretrain AE variants =====
ae_encoders = {}
for latent_dim in [8, 16, 32]:
    ae_model = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, latent_dim], pred_dim=0).to(device)
    ae_model = Bootstrapping.train_ae_with_bootstraps(ae_model, X_train, num_epochs=num_epochs, lr=AE_lr, sample_frac=0.8, weight_decay=weight_decay, device=device)
    ae_encoders[f"AE_{latent_dim}"] = ae_model

# --- pretrain Denoising AE ---
denoise_ae    = ae.DenoisingAE(input_size=input_dim, hidden_dims=[64,16], latent_dim=8,
                               dropout_prob=0.05, noise_std=0.1).to(device)
optimizer_dae = torch.optim.AdamW(denoise_ae.parameters(), lr=AE_lr, weight_decay=weight_decay)
for epoch in range(num_epochs):
    optimizer_dae.zero_grad()
    X_recon = denoise_ae(X_tensor)
    loss    = F.mse_loss(X_recon, X_tensor)
    loss.backward()
    optimizer_dae.step()

# ===== assemble encoders =====
encoders_dict = {**ae_encoders,
                 "denoiseAE": denoise_ae,
                 }

# oooooooooo Add a suphead for each encoder oooooooooooo
sup_head_rmse = {}

for name, encoder in encoders_dict.items():
    rmse = train_sup_head_per_encoder(encoder, X_L, y_L, X_test, y_test_scaled, dropout, train_encoder=True, device=device, epochs=num_epochs)
    sup_head_rmse[name] = rmse
    print(f"{name}: RMSE = {rmse:.4f}")
# ooooooooooooooooooooooooooooooooooooooooooooooooooooooo

# rrrrrrrrrrr train/evaluate each encoder individually rrrrrrrrrrr
# For each encoder:
# 1. Encode X_L and X_test to z_train / z_test
# 2. Fit a predictor (CatBoost) from z_train -> y_L
# 3. Predict y_test from z_test using the frozen encoder
# This evaluates the predictive power of each encoder individually
print("Per-encoder CatBoost RMSE:")
for name, encoder in encoders_dict.items():
    z_train = get_latent_tensor(encoder, X_L, train_encoder=False, device=device).cpu().numpy()
    z_test  = get_latent_tensor(encoder, X_test, train_encoder=False, device=device).cpu().numpy()
    _, rmse = train_and_eval_catboost(z_train, y_L, z_test, y_test_scaled)
    print(f"   {name}: {rmse:.4f}")

# ===== Encoder weights =====
weight_encoding_method = "inverse_rmse"  # "uniform", "inverse_rmse", "softmax"
if weight_encoding_method == "uniform":
    encoder_weights = {name: 1.0 for name in encoders_dict.keys()}
    total           = sum(encoder_weights.values())
    encoder_weights = {k: v / total for k, v in encoder_weights.items()}
elif weight_encoding_method == "inverse_rmse": # RMSE-based weights: better encoders get higher weight
    encoder_weights = {name: 1/rmse for name, rmse in sup_head_rmse.items()}
    total           = sum(encoder_weights.values())
    encoder_weights = {k: v/total for k,v in encoder_weights.items()}
elif weight_encoding_method == "softmax": # softmax-based weights
    inv_rmse        = np.array([1/r for r in sup_head_rmse.values()])
    weights_softmax = np.exp(inv_rmse) / np.sum(np.exp(inv_rmse))
    encoder_weights = {name: w for name, w in zip(sup_head_rmse.keys(), weights_softmax)}

# ===== ensemble clustering =====
# After pretraining, we encode X_all with all encoders
# Each encoder’s latent z is clustered via KMeans → produces soft assignment vector q_i
# Soft assignments (probabilities) from all encoders are combined (weighted average) 
# → this is the ensemble cluster_prob_matrix (shape: n_samples x n_clusters)
# This cluster_prob_matrix is the central piece for downstream prediction (frozen; no backprop)

# ===== train predictor + evaluate =====
# Use the ensemble cluster probability matrix as features X -> predict y
# Encoders are frozen; predictor (CatBoost or Linear) is trained on top of ensemble features
# Inference for one sample X1:
#   1. Encode X1 via each encoder → z_i
#   2. Map z_i → cluster probabilities q_i
#   3. Fuse q_i across encoders → prob_vector
#   4. Predict y1 = predictor(prob_vector)

# uuuuuuuuuuuuuuuuuuuuuuuuuuuuuuu
print("-------- predict on latents")
def get_concat_latents(encoders_dict, X, device="cpu"):
    """Return concatenated latent vectors from all encoders for X."""
    latents = []
    for name, encoder in encoders_dict.items():
        z = get_latent_tensor(encoder, X, train_encoder=False, device=device).cpu().numpy()
        latents.append(z)
    return np.concatenate(latents, axis=1)  # shape (N, sum(latent_dims))

z_train_concat = get_concat_latents(encoders_dict, X_L, device=device)
z_test_concat  = get_concat_latents(encoders_dict, X_test, device=device)

_, rmse = train_and_eval_catboost(z_train_concat, y_L, z_test_concat, y_test_scaled)
print(f"latent CatBoost RMSE: {rmse:.4f}")

linreg_loss, catboost_loss, rf_rmse = \
    Preds.evaluate_models_on_dataset(z_train_concat, y_train_scaled, z_test_concat, y_test_scaled, label_frac=label_frac)
print(f"Ensemble clustering results: dataset: {desired_dataset}")
print("    RMSE       | LinReg | CatBoost | Cluster | RForest | ElasticNet")
print(f"& Z (concat z) & {linreg_loss:.4f} & {catboost_loss:.4f}   & {unsupervised_rmse:.4f}  & {rf_rmse:.4f}  & {el_rmse:.4f} \\\\")


In [ ]:
"popeye"
"""Cellsup: clustering encoder ensemble"""

def get_latent_from_encoder(encoder, X, device="cpu"):
    X_tensor = torch.tensor(X, dtype=torch.float32, device=device)
    if type(encoder).__name__ in ["FlexibleAutoencoder", "AE", "VAE", "DenoisingAE"]:
        if X_tensor.ndim > 2:
            X_tensor = X_tensor.reshape(X_tensor.shape[0], -1)
        if type(encoder).__name__ == "VAE":
            mu, _ = encoder.encode(X_tensor)
            z = mu
        else:
            z = encoder.encode(X_tensor)
        z = z.detach().cpu().numpy()
    elif type(encoder).__name__ == "TS2VecEncoder":
        z = encoder.encode(X)
    else:
        raise ValueError(f"Unknown encoder type: {type(encoder).__name__}")
    return z

def bootstrap_sample(X, sample_frac=0.8):
    """Draw a bootstrap sample from X with replacement. Used to approximate sampling variability when the
    true population dist is unknown
        - X: Input array of shape (n_samples, ...).
        - sample_frac: Fraction of samples to draw (default=0.8).
        - returns: bootstrap sample array of shape (int(n_samples * sample_frac), ...)"""
    idx = np.random.choice(len(X), size=int(len(X)*sample_frac), replace=True)
    return X[idx]

class Cellsup:
    """Ensemble clustering across multiple encoders."""
    def __init__(self, encoders_dict: dict, n_clusters: int = 5, device: str = "cpu"):
        self.cluster_prob_matrix   = None
        self.encoders_dict = encoders_dict
        self.n_clusters    = n_clusters
        self.device        = device
        self.clusterers    = {}  # stores KMeans per encoder

    def fit_kmeans_on_encoder_latents(self, X: np.ndarray, encoder_weights: dict = None):
        probs_list = []
        for name, encoder in self.encoders_dict.items():
            z = get_latent_from_encoder(encoder, X, device=self.device)
            kmeans = KMeans(n_clusters=self.n_clusters, random_state=42).fit(z)
            labels = kmeans.labels_
            prob   = np.eye(self.n_clusters)[labels]
            if encoder_weights is not None:
                prob = prob * encoder_weights.get(name, 1.0)
            probs_list.append(prob)
            self.clusterers[name] = kmeans
        # combine soft assignments
        # self.cluster_prob_matrix = np.mean(probs_list, axis=0) if encoder_weights is None else np.sum(probs_list, axis=0)
        self.cluster_prob_matrix = np.concatenate(probs_list, axis=1)
        return self

    def apply_kmeans_to_new_data(self, X: np.ndarray) -> np.ndarray:
        probs_list = []
        for name, encoder in self.encoders_dict.items():
            z = get_latent_from_encoder(encoder, X, device=self.device)
            kmeans = self.clusterers[name]
            labels = kmeans.predict(z)
            prob = np.eye(self.n_clusters)[labels]
            probs_list.append(prob)
        # return np.mean(probs_list, axis=0)
        return np.concatenate(probs_list, axis=1)

# ===== params + prepare data =====
label_frac = 1  # labelled fraction of training data
n_samples  = int(label_frac * len(X_train))
X_L = X_train[:n_samples]  # labelled X
X_U = X_train[n_samples:]  # unlabelled X
y_L = y_train_scaled[:n_samples]  # labelled y

X_all     = np.concatenate([X_L, X_U], axis=0)
input_dim = X_all.shape[1] * X_all.shape[2] if X_all.ndim == 3 else X_all.shape[1]

# ===== pretrain section =====
# Pretraining step: each encoder learns X -> z -> X_recon
# Only this encoder's weights are updated (no ensemble info yet)
# After pretraining, encoder is frozen for downstream tasks

# ===== pretrain AE variants =====
num_AE_runs = 5
ae_encoders = {}
for latent_dim in [8, 16, 32]:
    ae_model      = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, latent_dim], pred_dim=0).to(device)
    optimizer_ae  = torch.optim.Adam(ae_model.parameters(), lr=1e-3)
    X_boot        = bootstrap_sample(X_train, 0.8)
    X_tensor_boot = torch.tensor(X_boot, dtype=torch.float32, device=device).reshape(len(X_boot), -1)
    for epoch in range(num_AE_runs):
        optimizer_ae.zero_grad()
        X_recon = ae_model(X_tensor_boot)
        loss    = F.mse_loss(X_recon, X_tensor_boot)
        loss.backward()
        optimizer_ae.step()
    ae_encoders[f"AE_{latent_dim}"] = ae_model

# --- pretrain AE1 ---
num_AE_runs = 5
AE_lr       = 1e-3
ae_model1    = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, 16], pred_dim=0).to(device)
optimizer_ae = torch.optim.Adam(ae_model1.parameters(), lr=AE_lr)
X_tensor     = torch.tensor(X_train, dtype=torch.float32, device=device).reshape(len(X_train), -1)
for epoch in range(num_AE_runs):
    optimizer_ae.zero_grad()
    X_recon = ae_model1(X_tensor)
    loss    = F.mse_loss(X_recon, X_tensor)
    loss.backward()
    optimizer_ae.step()

# --- pretrain Denoising AE ---
num_DAE_runs = 5
denoise_ae    = ae.DenoisingAE(input_size=input_dim, hidden_dims=[64,16], latent_dim=8,
                               dropout_prob=0.05, noise_std=0.1).to(device)
optimizer_dae = torch.optim.AdamW(denoise_ae.parameters(), lr=AE_lr)
for epoch in range(num_DAE_runs):
    optimizer_dae.zero_grad()
    X_recon = denoise_ae(X_tensor)
    loss    = F.mse_loss(X_recon, X_tensor)
    loss.backward()
    optimizer_dae.step()

# --- pretrain VAE ---
num_VAE_runs   = 5
kl_loss_weight = 1e-3
vae_model      = ae.VAE(input_size=input_dim, hidden_dim=64, latent_dim=16).to(device)
optimizer_vae  = torch.optim.Adam(vae_model.parameters(), lr=1e-3)
for epoch in range(num_VAE_runs):
    optimizer_vae.zero_grad()
    mu, logvar = vae_model.encode(X_tensor)
    z          = vae_model.reparameterize(mu, logvar)
    X_recon    = vae_model.decode(z)
    recon_loss = F.mse_loss(X_recon, X_tensor)
    kl_loss    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    loss       = recon_loss + kl_loss_weight * kl_loss
    loss.backward()
    optimizer_vae.step()

# --- train/load ts2vec ---
ts2vec_encoder = TS2VecEncoder(z_pooling=z_pooling_method, device=device, patience=ts2vec_patience)
ts2vec_encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
                   depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=5)

# ===== assemble encoders =====
encoders_dict = {**ae_encoders,
                 "flexibleAE": ae_model1,
                 "vae": vae_model,
                #  "ts2vec": ts2vec_encoder,
                 "denoiseAE": denoise_ae,
                 }
# ===== optional: encoder weights =====
encoder_weights = {name: 1.0 for name in encoders_dict.keys()}
total           = sum(encoder_weights.values())
encoder_weights = {k: v / total for k, v in encoder_weights.items()}

# zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz
n_clusters = 8
# Make a copy of the original encoders dict
encoders_test = encoders_dict.copy()

# Replace AE1 with a completely untrained AE
untrained_ae = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, 16], pred_dim=0).to(device)
encoders_test["flexibleAE_untrained"] = untrained_ae

# Remove the pretrained AE1 if you want a clean comparison
# encoders_test.pop("flexibleAE")

# Re-run ensemble clustering with the modified encoders
ensemble_cluster_test = Cellsup(encoders_dict=encoders_test, n_clusters=n_clusters, device=device)
train_probs_test = ensemble_cluster_test.fit_kmeans_on_encoder_latents(X_all, encoder_weights=encoder_weights).cluster_prob_matrix
test_probs_test  = ensemble_cluster_test.apply_kmeans_to_new_data(X_test)

train_probs_L_test = train_probs_test[:len(y_L)]
# Train CatBoost on latent probs of labeled data
catboost_model_test, y_test_pred_test, catboost_rmse_test, _ = \
    Preds.predict_catboost_multioutput(train_probs_L_test, y_L, test_probs_test, y_test_scaled)

print(f"CatBoost RMSE with untrained AE in ensemble: {catboost_rmse_test:.4f}")
# zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz

# ===== train/evaluate each encoder individually =====
# For each encoder:
# 1. Encode X_L and X_test to z_train / z_test
# 2. Fit a predictor (CatBoost) from z_train -> y_L
# 3. Predict y_test from z_test using the frozen encoder
# This evaluates the predictive power of each encoder individually
print("Per-encoder RMSE on X_L vs X_test:")
for name, encoder in encoders_dict.items():
    z_train = get_latent_from_encoder(encoder, X_L, device=device)
    z_test  = get_latent_from_encoder(encoder, X_test, device=device)

    # identify non-constant columns in y_train (on labeled fraction)
    mask         = [i for i in range(y_L.shape[1]) if not np.all(y_L[:, i] == y_L[0, i])]
    const_values = {i: y_L[0, i] for i in range(y_L.shape[1]) if i not in mask}

    y_pred = np.zeros_like(y_test_scaled, dtype=float)  # shape = test labels
    model  = None
    if mask:
        model = MultiOutputRegressor(CatBoostRegressor(iterations=500, learning_rate=0.1, depth=4,random_seed=42, verbose=0))
        model.fit(z_train, y_L[:, mask])
        y_pred[:, mask] = model.predict(z_test)  # now matches y_test_scaled shape

    # fill constant columns
    for i, v in const_values.items():
        y_pred[:, i] = v

    rmse = root_mean_squared_error(y_test_scaled, y_pred)
    print(f"  {name}: {rmse:.4f}")

# ===== ensemble clustering =====
# After pretraining, we encode X_all with all encoders
# Each encoder’s latent z is clustered via KMeans → produces soft assignment vector q_i
# Soft assignments (probabilities) from all encoders are combined (weighted average) 
# → this is the ensemble cluster_prob_matrix (shape: n_samples x n_clusters)
# This cluster_prob_matrix is the central piece for downstream prediction (frozen; no backprop)
n_clusters = 8
ensemble_cluster = Cellsup(encoders_dict=encoders_dict, n_clusters=n_clusters, device=device)
# train_probs_all  = ensemble_cluster.fit_kmeans_on_encoder_latents(X_all, encoder_weights=encoder_weights).cluster_prob_matrix
train_probs_all  = ensemble_cluster.fit_kmeans_on_encoder_latents(X_train, encoder_weights=encoder_weights).cluster_prob_matrix
test_probs       = ensemble_cluster.apply_kmeans_to_new_data(X_test)

train_probs_L = train_probs_all[:len(y_L)]
train_probs_U = train_probs_all[len(y_L):]

# ===== train predictor + evaluate =====
# Use the ensemble cluster probability matrix as features X -> predict y
# Encoders are frozen; predictor (CatBoost or Linear) is trained on top of ensemble features
# Inference for one sample X1:
#   1. Encode X1 via each encoder → z_i
#   2. Map z_i → cluster probabilities q_i
#   3. Fuse q_i across encoders → prob_vector
#   4. Predict y1 = predictor(prob_vector)
catboost_model, y_test_pred, catboost_rmse, non_const_idx = \
    Preds.predict_catboost_multioutput(train_probs_L, y_L, test_probs, y_test_scaled)
print(f"label_frac: {label_frac}, CatBoost RMSE (on X_L only): {catboost_rmse:.4f}")

linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse, el_rmse = \
    Preds.evaluate_models_on_dataset(train_probs_all, y_train_scaled, test_probs, y_test_scaled, label_frac=1.0)

print(f"Ensemble clustering results:")
print("    RMSE       | LinReg | CatBoost | Cluster | RForest | ElasticNet")
print(f"& Z (ensemble) & {linreg_loss:.4f} & {catboost_loss:.4f}   & {unsupervised_rmse:.4f}  & {rf_rmse:.4f}  & {el_rmse:.4f} \\\\")


In [ ]:
"""[old] Loop Headsup"""
# params
train_epochs = 150
batch_size   = 16
warmup_steps = 1000
max_steps    = train_epochs * (len(X_train) // batch_size)
step         = 0

decoder_hidden_dims = [16, 64, 128]
optimizer_lr   = 1e-3
projection_dim = 16

# init methods
encoder = TS2VecEncoder(z_pooling=z_pooling_method, device=device)
encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
            depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=ts2vec_epochs)

proj_head   = ProjectionHead(input_dim=ts2vec_latent_dims, proj_dim=projection_dim).to(device)
predictor   = ProjectionHead(input_dim=projection_dim, proj_dim=projection_dim).to(device)
decoder     = Decoder(latent_dim=ts2vec_latent_dims, output_shape=(X_train.shape[1], X_train.shape[2]),
                      hidden_sizes=decoder_hidden_dims).to(device)
supervised_head = MLPHead(input_dim=ts2vec_latent_dims, output_dim=y_train_scaled.shape[1], hidden_sizes=predictor_hidden_sizes,
                          lr=predictor_lr, epochs=1, dropout=predictor_dropout, device=device)#.to(device)
# joint optimizer
params = (list(proj_head.parameters()) +
          list(predictor.parameters()) +
          list(decoder.parameters()) +
          list(supervised_head.model.parameters()))
optimizer = torch.optim.AdamW(params, lr=optimizer_lr)

for epoch in range(train_epochs):
    for i in range(0, len(X_train), batch_size):
        step   += 1
        X_batch = X_train[i:i+batch_size]
        y_batch = y_train_scaled[i:i+batch_size]

        # 1. Augment
        X1 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "jitter", device, 0.1)
        X2 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "mag_warp", device, 0.1)

        # 2. Encode
        z1 = torch.tensor(encoder.encode(X1.cpu().numpy()), dtype=torch.float32, device=device)
        z2 = torch.tensor(encoder.encode(X2.cpu().numpy()), dtype=torch.float32, device=device)

        # 3. Projections
        h1, h2 = proj_head(z1), proj_head(z2)
        p1     = predictor(h1)

        # 4. Losses
        loss_contrast = Losses.compute_byol_loss(p1, h2.detach())
        x_recon       = decoder(z1)
        loss_recon    = F.mse_loss(x_recon, torch.tensor(X_batch, dtype=torch.float32, device=device))

        y_batch_t = torch.tensor(y_batch, dtype=torch.float32, device=device)
        y_hat     = supervised_head.model(z1)
        loss_pred = F.mse_loss(y_hat, y_batch_t)

        # 5. Weighted total loss
        weights = get_loss_weights(step, warmup_steps, max_steps)
        # weights = {"pred": 1.0, "recon": 0.0, "contrast": 0.0}
        loss    = (weights["recon"] * loss_recon + weights["contrast"] * loss_contrast + weights["pred"] * loss_pred)

        # 6. Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{train_epochs}: "
          f"pred={loss_pred.item():.4f}, recon={loss_recon.item():.4f}, "
          f"contrast={loss_contrast.item():.4f}, total loss={loss.item():.4f}")

# ===== Inference =====
proj_head.eval()
predictor.eval()
decoder.eval()
supervised_head.model.eval()

with torch.no_grad():
    # encode test set once with frozen encoder
    z_test = torch.tensor(ts2vec_encoder.encode(X_test), dtype=torch.float32, device=device)
    y_pred = supervised_head.model(z_test).cpu().numpy()

rmse = root_mean_squared_error(y_test_scaled, y_pred)
print(f"Headsup RMSE: {rmse:.4f}")



In [ ]:
"[to remove] Run Headsup with TS2Vec encoder"
# 0. params
w_pred, w_recon, w_contrast = 0.5, 0.2, 0.3
decoder_hidden_dims = [16, 64, 128]
optimizer_lr   = 9e-3
projection_dim = 16
proj_input_dim = projection_dim

# 1. augment X
X_1 = make_augmentations(torch.tensor(X_train, dtype=torch.float32, device=device), "jitter", device, 0.1)
X_2 = make_augmentations(torch.tensor(X_train, dtype=torch.float32, device=device), "mag_warp", device, 0.1)

# 2. fit TS2Vec on train data
ts2vec_encoder = TS2VecEncoder(z_pooling=z_pooling_method, device=device)
ts2vec_encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
                   depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=ts2vec_epochs)

# encode augmented views (X → z)
z_1 = torch.tensor(ts2vec_encoder.encode(X_1.cpu().numpy()), dtype=torch.float32, device=device)
z_2 = torch.tensor(ts2vec_encoder.encode(X_2.cpu().numpy()), dtype=torch.float32, device=device)

# encode full train/test set for supervised loss
z_train   = torch.tensor(ts2vec_encoder.encode(X_train), dtype=torch.float32, device=device)
z_test    = torch.tensor(ts2vec_encoder.encode(X_test), dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32, device=device)
y_test_t  = torch.tensor(y_test_scaled, dtype=torch.float32, device=device)

# 3. projection head
proj_head = ProjectionHead(input_dim=z_1.shape[1], proj_dim=projection_dim).to(device)
h_1 = proj_head(z_1)
h_2 = proj_head(z_2)

# 4. predictor for BYOL contrast
predictor = ProjectionHead(input_dim=proj_input_dim, proj_dim=projection_dim).to(device)
p_1 = predictor(h_1)

# 5. BYOL contrastive loss
loss_contrast = Losses.compute_byol_loss(p_1, h_2.detach())

# 6. optional decoder
decoder = Decoder(latent_dim=z_1.shape[1], output_shape=(X_train.shape[1], X_train.shape[2]),
                  hidden_sizes= decoder_hidden_dims).to(device)
x_recon = decoder(z_1)
loss_recon = F.mse_loss(x_recon, torch.tensor(X_train, dtype=torch.float32, device=device))

# 7. supervised predictor head
supervised_head = MLPHead(input_dim=z_1.shape[1], output_dim=y_train_scaled.shape[1],
                          hidden_sizes=predictor_hidden_sizes, lr=predictor_lr,
                          epochs=predictor_epochs, dropout=predictor_dropout, device=device)

# forward pass for supervised prediction
y_hat_train = supervised_head.model(z_train)
loss_pred   = F.mse_loss(y_hat_train, y_train_t)

# 8. combine losses
# total_loss = w_pred * loss_pred + w_recon * loss_recon + w_contrast * loss_contrast
weights = get_loss_weights(step, warmup_steps, max_steps)
loss    = (weights["recon"] * loss_recon + weights["contrast"] * loss_contrast + weights["pred"] * loss_pred)
print(f"Loss: pred={loss_pred.item():.4f}, recon={loss_recon.item():.4f}, contrast={loss_contrast.item():.4f}, total={loss.item():.4f}")

# 9. backprop and joint optimization
optimizer = torch.optim.AdamW(list(proj_head.parameters()) +
                              list(predictor.parameters()) +
                              list(decoder.parameters()) +
                              list(supervised_head.model.parameters()),
                              lr=optimizer_lr)
optimizer.zero_grad()
loss.backward()
optimizer.step()

"Predictions"
# 1️⃣ Encode test data with TS2Vec (frozen encoder)
z_test = torch.tensor(ts2vec_encoder.encode(X_test), dtype=torch.float32, device=device)

# 2️⃣ Predict with supervised MLPHead
y_pred = supervised_head.predict(z_test) # changes done to supervised_head

rmse  = root_mean_squared_error(y_test_scaled, y_pred)
print(f"Headsup RMSE: {rmse:.4f}")


In [ ]:
"old cellsup"

"""Cellsup: clustering encoder ensemble"""
from scipy.special import softmax

# old
def XXX_get_latent_from_encoder(encoder, X, device="cpu"):
    X_tensor = torch.tensor(X, dtype=torch.float32, device=device)
    if type(encoder).__name__ in ["FlexibleAutoencoder", "AE", "VAE", "DenoisingAE"]:
        if X_tensor.ndim > 2:
            X_tensor = X_tensor.reshape(X_tensor.shape[0], -1)
        if type(encoder).__name__ == "VAE":
            mu, _ = encoder.encode(X_tensor)
            z = mu
        else:
            z = encoder.encode(X_tensor)
        z = z.detach().cpu().numpy()
    elif type(encoder).__name__ == "TS2VecEncoder":
        z = encoder.encode(X)
    else:
        raise ValueError(f"Unknown encoder type: {type(encoder).__name__}")
    return z

def get_latent_from_encoder(encoder, X, device="cpu") -> np.ndarray:
    """Return latent z for any encoder type with shape (N, latent_dim)."""
    if isinstance(encoder, nn.Module):
        X_tensor = torch.tensor(X, dtype=torch.float32, device=device)
        if X_tensor.ndim > 2:
            X_tensor = X_tensor.reshape(len(X_tensor), -1)
        if type(encoder).__name__ == "VAE":
            mu, _ = encoder.encode(X_tensor)
            z     = mu.detach().cpu().numpy()
        else: # AE / DAE: return latent layer instead of reconstruction
            z = encoder.encode(X_tensor).detach().cpu().numpy()
    elif type(encoder).__name__ == "TS2VecEncoder":
        z = encoder.encode(X)  # returns (N, T, latent_dim)
        z = z.mean(axis=1)      # temporal pooling
    else:
        raise ValueError(f"Unknown encoder type: {type(encoder).__name__}")
    return z

def bootstrap_sample(X, sample_frac=0.8):
    """Draw a bootstrap sample from X with replacement. Used to approximate sampling variability when the
    true population dist is unknown
        - X: Input array of shape (n_samples, ...).
        - sample_frac: Fraction of samples to draw (default=0.8).
        - returns: bootstrap sample array of shape (int(n_samples * sample_frac), ...)"""
    idx = np.random.choice(len(X), size=int(len(X)*sample_frac), replace=True)
    return X[idx]

def build_Q_concat(clusterers: Dict[str, KMeans], zs: Dict[str, np.ndarray], weights: Dict[str, float] | None = None) -> np.ndarray:
    """Convert encoder latents into concatenated cluster one-hots.
    Args:
        clusterers: dict[name] -> fitted KMeans (K clusters).
        zs: dict[name] -> latent array for that encoder, shape (N, D_z).
        weights: optional dict[name] -> scalar weight for that encoder.
    Returns:
        Q_concat: np.ndarray of shape (N, E*K), concatenated per-encoder one-hots."""
    blocks: List[np.ndarray] = []
    for name, kmeans in clusterers.items():
        z = zs[name]                        # (N, D_z)
        labels = kmeans.predict(z)          # (N,)
        K = kmeans.n_clusters
        q = np.eye(K)[labels]               # (N, K) hard one-hot
        if weights is not None:
            q = q * float(weights.get(name, 1.0))
        blocks.append(q)
    return np.concatenate(blocks, axis=1)   # (N, E*K)


class SupHead(nn.Module):
    """Small supervised head: maps latent z -> target y"""
    def __init__(self, input_dim: int, output_dim: int, hidden_sizes=[64, 32]):
        super().__init__()
        layers, prev_dim = [], input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            prev_dim = h
        layers.append(nn.Linear(prev_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)

class Cellsup:
    """Ensemble clustering across multiple encoders."""
    def __init__(self, encoders_dict: dict, n_clusters: int = 5, device: str = "cpu", cluster_assignment: str = "soft"):
        self.encoders_dict      = encoders_dict
        self.n_clusters         = n_clusters
        self.device             = device
        self.cluster_assignment = cluster_assignment
        self.cluster_prob_matrix= None
        self.clusterers         = {}  # stores KMeans per encoder

    def fit_kmeans_on_encoder_latents(self, X: np.ndarray, encoder_weights: dict = None):
        probs_list = []
        for name, encoder in self.encoders_dict.items():
            z      = get_latent_from_encoder(encoder, X, device=self.device)
            kmeans = KMeans(n_clusters=self.n_clusters, random_state=42).fit(z)
            labels = kmeans.labels_
            if self.cluster_assignment == "soft":
                distances = kmeans.transform(z)  # shape (n_samples, n_clusters)
                prob      = softmax(-distances, axis=1)  # convert distance → probability
            elif self.cluster_assignment == "hard":
                prob = np.eye(self.n_clusters)[labels]
            if encoder_weights is not None:
                prob *= encoder_weights.get(name, 1.0)
            probs_list.append(prob)
            self.clusterers[name] = kmeans
        # # combine soft assignments
        # self.cluster_prob_matrix = np.mean(probs_list, axis=0) if encoder_weights is None else np.sum(probs_list, axis=0)
        # return self

        # FIX: concatenate horizontally instead of averaging
        self.cluster_prob_matrix = np.concatenate(probs_list, axis=1)  # (N, E*K)
        return self

    def apply_kmeans_to_new_data(self, X: np.ndarray) -> np.ndarray:
        probs_list = []
        for name, encoder in self.encoders_dict.items():
            z      = get_latent_from_encoder(encoder, X, device=self.device)
            kmeans = self.clusterers[name]
            labels = kmeans.predict(z)
            if self.cluster_assignment == "soft":
                distances = kmeans.transform(z)  # shape (n_samples, n_clusters)
                prob      = softmax(-distances, axis=1)  # convert distance → probability
            elif self.cluster_assignment == "hard":
                prob = np.eye(self.n_clusters)[labels]
            probs_list.append(prob)
        # return np.mean(probs_list, axis=0)

        # fix
        return np.concatenate(probs_list, axis=1)  # (N, E*K)

# ===== params + prepare data =====
label_frac = 0.5  # labelled fraction of training data
n_samples  = int(label_frac * len(X_train))
X_L = X_train[:n_samples]  # labelled X
X_U = X_train[n_samples:]  # unlabelled X
y_L = y_train_scaled[:n_samples]  # labelled y

X_all     = np.concatenate([X_L, X_U], axis=0)
input_dim = X_all.shape[1] * X_all.shape[2] if X_all.ndim == 3 else X_all.shape[1]

# ===== pretrain section =====
# Pretraining step: each encoder learns X -> z -> X_recon
# Only this encoder's weights are updated (no ensemble info yet)
# After pretraining, encoder is frozen for downstream tasks

# ===== pretrain AE variants =====
num_AE_runs = 5
ae_encoders = {}
for latent_dim in [8, 16, 32]:
    ae_model      = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, latent_dim], pred_dim=0).to(device)
    optimizer_ae  = torch.optim.AdamW(ae_model.parameters(), lr=1e-3)
    X_boot        = bootstrap_sample(X_train, 0.8)
    X_tensor_boot = torch.tensor(X_boot, dtype=torch.float32, device=device).reshape(len(X_boot), -1)
    for epoch in range(num_AE_runs):
        optimizer_ae.zero_grad()
        X_recon = ae_model(X_tensor_boot)
        loss    = F.mse_loss(X_recon, X_tensor_boot)
        loss.backward()
        optimizer_ae.step()
    ae_encoders[f"AE_{latent_dim}"] = ae_model

# --- pretrain AE1 ---
num_AE_runs = 5
AE_lr       = 1e-3
ae_model1    = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, 16], pred_dim=0).to(device)
optimizer_ae = torch.optim.AdamW(ae_model1.parameters(), lr=AE_lr)
X_tensor     = torch.tensor(X_train, dtype=torch.float32, device=device).reshape(len(X_train), -1)
for epoch in range(num_AE_runs):
    optimizer_ae.zero_grad()
    X_recon = ae_model1(X_tensor)
    loss    = F.mse_loss(X_recon, X_tensor)
    loss.backward()
    optimizer_ae.step()

# --- pretrain Denoising AE ---
num_DAE_runs = 5
denoise_ae    = ae.DenoisingAE(input_size=input_dim, hidden_dims=[64,16], latent_dim=8,
                               dropout_prob=0.05, noise_std=0.1).to(device)
optimizer_dae = torch.optim.AdamW(denoise_ae.parameters(), lr=AE_lr)
for epoch in range(num_DAE_runs):
    optimizer_dae.zero_grad()
    X_recon = denoise_ae(X_tensor)
    loss    = F.mse_loss(X_recon, X_tensor)
    loss.backward()
    optimizer_dae.step()

# --- pretrain VAE ---
num_VAE_runs   = 5
kl_loss_weight = 1e-3
vae_model      = ae.VAE(input_size=input_dim, hidden_dim=64, latent_dim=16).to(device)
optimizer_vae  = torch.optim.AdamW(vae_model.parameters(), lr=1e-3)
for epoch in range(num_VAE_runs):
    optimizer_vae.zero_grad()
    mu, logvar = vae_model.encode(X_tensor)
    z          = vae_model.reparameterize(mu, logvar)
    X_recon    = vae_model.decode(z)
    recon_loss = F.mse_loss(X_recon, X_tensor)
    kl_loss    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    loss       = recon_loss + kl_loss_weight * kl_loss
    loss.backward()
    optimizer_vae.step()

# --- train/load ts2vec ---
# n_epochs = 1 # change back to 5 when done debugging
# ts2vec_encoder = TS2VecEncoder(z_pooling=z_pooling_method, device=device, patience=ts2vec_patience)
# ts2vec_encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
#                    depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=n_epochs)

# ===== assemble encoders =====
encoders_dict = {**ae_encoders,
                 "flexibleAE": ae_model1,
                 "vae": vae_model,
                #  "ts2vec": ts2vec_encoder,
                 "denoiseAE": denoise_ae,
                 }
# ===== optional: encoder weights =====
encoder_weights = {name: 1.0 for name in encoders_dict.keys()}
total           = sum(encoder_weights.values())
encoder_weights = {k: v / total for k, v in encoder_weights.items()}

# encoder_weights = {name: 1/rmse for name, rmse in sup_head_rmse.items()}
# total = sum(encoder_weights.values())
# encoder_weights = {k: v/total for k,v in encoder_weights.items()}


# ooooooooooooooooooooooooooooooooooooooooooooooooooooooo
trainable_encoders = ["FlexibleAutoencoder", "AE", "VAE", "DenoisingAE"]
sup_head_rmse = {}

for name, encoder in encoders_dict.items():
    train_encoder = isinstance(encoder, nn.Module) and type(encoder).__name__ in trainable_encoders

    # ----- sample latent to get input_dim -----
    # Get the latent representation of a sample to determine the input dimension for SupHead.
    # Put the encoder in eval mode to handle BatchNorm layers with batch_size=1
    if train_encoder:
        encoder.eval()
    
    z_sample_L = get_latent_from_encoder(encoder, X_L[:2], device=device) # Use a batch of size 2
    if isinstance(z_sample_L, np.ndarray):
        z_sample_L = torch.tensor(z_sample_L, dtype=torch.float32, device=device)
    
    # Switch back to train mode if needed
    if train_encoder:
        encoder.train()
    
    latent_dim = z_sample_L.shape[-1]
    sup_head = SupHead(input_dim=latent_dim, output_dim=y_L.shape[1]).to(device)

    # ----- optimizer -----
    params = list(sup_head.parameters())
    if train_encoder:
        params += list(encoder.parameters())
    optimizer = torch.optim.AdamW(params, lr=1e-3)

    # ----- training -----
    for epoch in range(5):
        sup_head.train()
        if train_encoder:
            encoder.train()
            # The .encode() method must be called to get the latent space
            X_tensor = torch.tensor(X_L, dtype=torch.float32, device=device).reshape(len(X_L), -1)
            z = encoder.encode(X_tensor)
            if isinstance(z, tuple):
                z = z[0]
            # Ensure z is 2D
            if z.ndim > 2:
                z = z.mean(axis=1)
        else:
            with torch.no_grad():
                z = get_latent_from_encoder(encoder, X_L, device=device)
                if z.ndim > 2:
                    z = z.mean(axis=1)
                if isinstance(z, np.ndarray):
                    z = torch.tensor(z, dtype=torch.float32, device=device)

        if z.ndim == 1:
            z = z[:, None]

        print(f"{name} SupHead input: {z.shape}")
        y_pred = sup_head(z)
        y_tensor = torch.tensor(y_L, dtype=torch.float32, device=device)
        loss = F.mse_loss(y_pred, y_tensor)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"Epoch #{epoch}: loss={loss.item():.4f}")

    # ----- evaluation -----
    sup_head.eval()
    with torch.no_grad():
        if train_encoder:
            encoder.eval()
            X_test_tensor = torch.tensor(X_test, dtype=torch.float32, device=device)
            if X_test_tensor.ndim > 2:
                 X_test_tensor = X_test_tensor.reshape(len(X_test_tensor), -1)
            z_test = encoder.encode(X_test_tensor)
            if isinstance(z_test, tuple):
                z_test = z_test[0]
        else:
            z_test = get_latent_from_encoder(encoder, X_test, device=device)
            
        if z_test.ndim > 2:
            z_test = z_test.mean(axis=1)
        if isinstance(z_test, np.ndarray):
            z_test = torch.tensor(z_test, dtype=torch.float32, device=device)
        if z_test.ndim == 1:
            z_test = z_test[:, None]

        y_pred_test = sup_head(z_test).cpu().numpy()
        rmse = root_mean_squared_error(y_test_scaled, y_pred_test)
        sup_head_rmse[name] = rmse
        print(f"{name}: RMSE = {rmse:.4f}")
# ooooooooooooooooooooooooooooooooooooooooooooooooooooooo

# ===== train/evaluate each encoder individually =====
# For each encoder:
# 1. Encode X_L and X_test to z_train / z_test
# 2. Fit a predictor (CatBoost) from z_train -> y_L
# 3. Predict y_test from z_test using the frozen encoder
# This evaluates the predictive power of each encoder individually
print("Per-encoder RMSE on X_L vs X_test:")
for name, encoder in encoders_dict.items():
    z_train = get_latent_from_encoder(encoder, X_L, device=device)
    z_test  = get_latent_from_encoder(encoder, X_test, device=device)

    # identify non-constant columns in y_train (on labeled fraction)
    mask         = [i for i in range(y_L.shape[1]) if not np.all(y_L[:, i] == y_L[0, i])]
    const_values = {i: y_L[0, i] for i in range(y_L.shape[1]) if i not in mask}

    y_pred = np.zeros_like(y_test_scaled, dtype=float)  # shape = test labels
    model  = None
    if mask:
        model = MultiOutputRegressor(CatBoostRegressor(iterations=500, learning_rate=0.1, depth=4, verbose=0))
        model.fit(z_train, y_L[:, mask])
        y_pred[:, mask] = model.predict(z_test)  # now matches y_test_scaled shape

    # fill constant columns
    for i, v in const_values.items():
        y_pred[:, i] = v

    rmse = root_mean_squared_error(y_test_scaled, y_pred)
    print(f"  {name}: {rmse:.4f}")

# ===== ensemble clustering =====
# After pretraining, we encode X_all with all encoders
# Each encoder’s latent z is clustered via KMeans → produces soft assignment vector q_i
# Soft assignments (probabilities) from all encoders are combined (weighted average) 
# → this is the ensemble cluster_prob_matrix (shape: n_samples x n_clusters)
# This cluster_prob_matrix is the central piece for downstream prediction (frozen; no backprop)
n_clusters = 8
ensemble_cluster = Cellsup(encoders_dict=encoders_dict, n_clusters=n_clusters, device=device)
train_probs_all  = ensemble_cluster.fit_kmeans_on_encoder_latents(X_all, encoder_weights=encoder_weights).cluster_prob_matrix
test_probs       = ensemble_cluster.apply_kmeans_to_new_data(X_test)

train_probs_L = train_probs_all[:len(y_L)]
train_probs_U = train_probs_all[len(y_L):]

# ===== train predictor + evaluate =====
# Use the ensemble cluster probability matrix as features X -> predict y
# Encoders are frozen; predictor (CatBoost or Linear) is trained on top of ensemble features
# Inference for one sample X1:
#   1. Encode X1 via each encoder → z_i
#   2. Map z_i → cluster probabilities q_i
#   3. Fuse q_i across encoders → prob_vector
#   4. Predict y1 = predictor(prob_vector)
catboost_model, y_test_pred, catboost_rmse, non_const_idx = \
    Preds.predict_catboost_multioutput(train_probs_L, y_L, test_probs, y_test_scaled)
print(f"label_frac: {label_frac}, CatBoost RMSE (on X_L only): {catboost_rmse:.4f}")

linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse, el_rmse = \
    Preds.evaluate_models_on_dataset(train_probs_all, y_train_scaled, test_probs, y_test_scaled, label_frac=1.0)

print(f"Ensemble clustering results:")
print("    RMSE       | LinReg | CatBoost | Cluster | RForest | ElasticNet")
print(f"& Z (ensemble) & {linreg_loss:.4f} & {catboost_loss:.4f}   & {unsupervised_rmse:.4f}  & {rf_rmse:.4f}  & {el_rmse:.4f} \\\\")


In [ ]:
# pages, rows, cols = X_train.shape
# plt.figure(figsize=(15, 6))
# for i in range(cols):
#     plt.plot(X_train[4,:,i])
#     break
# plt.show()

rows, cols = y_train_scaled.shape
plt.figure(figsize=(15, 6))
for i in range(rows):
    plt.plot(y_train_scaled[i,:])
    if i >10:
        break
plt.show()


In [ ]:
"Logging results"
CSV_LOG_NAME  = f"runs_log.csv"
JSON_LOG_NAME = f"runs_log.jsonl"
CSV_LOG_FILE  = Path(f"results/{CSV_LOG_NAME}")
JSON_LOG_FILE = Path(f"results/{JSON_LOG_NAME}")

params  = {"pretrain_epochs": train_epochs_pretrain, "train_epochs": train_epochs_finetune,
           "batch_size_pretrain": batch_size_pretrain, "batch_size_train": batch_size_train,
           "decoder_hidden_dims": decoder_hidden_dims, "projection_dim": projection_dim,
           "lr_pretrain": lr_pretrain, "lr_train": lr_train}

formatted_results_dict = {k: f"{v:.4f}" for k, v in results_dict.items()}
results = {"results": formatted_results_dict,
           "linreg_rmse": f"{linreg_loss:.4f}","catboost_rmse": f"{catboost_loss:.4f}",
           "unsupervised_rmse": f"{unsupervised_rmse:.4f}", "rf_rmse": f"{rf_rmse:.4f}",
           "el_rmse": f"{el_rmse:.4f}"}

top_metrics = {"linreg_rmse": linreg_loss,
               "catboost_rmse": catboost_loss,
               "unsupervised_rmse": unsupervised_rmse,
               "rf_rmse": rf_rmse,
               "el_rmse": el_rmse}

LogRunResults.log_run_in_csv(params, results, CSV_LOG_FILE)
LogRunResults.log_run_json(params, results_dict, top_metrics, JSON_LOG_FILE)


In [ ]:
# ===== Headsup for CatBoost =====
# doesnt work well, catboost doesnt backprop

class HeadsCatboost(Headsup):
    def __init__(self, encoder, proj_head, decoder, device, catboost_params=None):
        super().__init__(encoder, proj_head, decoder, supervised_head=None, device=device)
        self.catboost_params = catboost_params or dict(iterations=500, learning_rate=0.05, depth=6, verbose=0)
        self.catboost_model = None
        self.non_constant_idx = []

    def pretrain_encoder(self, X_train, batch_size, epochs):
        """Run only the pretraining stage."""
        self.pretrain(X_train, batch_size=batch_size, epochs=epochs)

    def predict_catboost_multioutput(self, X_train: np.ndarray, y_train: np.ndarray, 
                                     X_test: np.ndarray, y_test: np.ndarray) -> Tuple[Optional[MultiOutputRegressor], np.ndarray, float, list]:
        """Train multi-output CatBoost and return predictions, RMSE, and non-constant indices."""
        y_pred = np.zeros_like(y_test, dtype=float)
        non_constant_idx = [i for i in range(y_train.shape[1])
                            if not np.all(y_train[:, i] == y_train[0, i])]
        if non_constant_idx:
            model = MultiOutputRegressor(CatBoostRegressor(**self.catboost_params))
            model.fit(X_train, y_train[:, non_constant_idx])
            y_pred[:, non_constant_idx] = model.predict(X_test)
            for i in range(y_train.shape[1]):
                if i not in non_constant_idx:
                    y_pred[:, i] = y_train[0, i]
        else:
            model = None
        rmse = root_mean_squared_error(y_test, y_pred)
        return model, y_pred, rmse, non_constant_idx

    def train_supervised(self, X_train, y_train_scaled, X_test, y_test_scaled, label_fractions):
        """Train CatBoost on frozen encoder features for multiple label fractions."""
        results_dict, z_train_dict, z_test_dict, y_L_dict = {}, {}, {}, {}

        self.encoder.eval()
        self.proj_head.eval()
        self.decoder.eval()

        for frac in label_fractions:
            n_samples = int(len(X_train) * frac)
            X_L, y_L = X_train[:n_samples], y_train_scaled[:n_samples]

            # Frozen encoder features
            with torch.no_grad():
                z_train = self.encoder(torch.tensor(X_L, dtype=torch.float32, device=self.device)).mean(dim=1).cpu().numpy()
                z_test  = self.encoder(torch.tensor(X_test, dtype=torch.float32, device=self.device)).mean(dim=1).cpu().numpy()

            # Train CatBoost
            model, y_pred, rmse, non_constant_idx = self.predict_catboost_multioutput(z_train, y_L, z_test, y_test_scaled)
            self.catboost_model = model
            self.non_constant_idx = non_constant_idx

            results_dict[frac]  = rmse
            z_train_dict[frac]  = z_train
            z_test_dict[frac]   = z_test
            y_L_dict[frac]      = y_L
            print(f"RMSE with {frac*100:.0f}% labeled (CatBoost): {rmse:.4f}")

        return results_dict, z_train_dict, z_test_dict, y_L_dict

    def predict(self, X, y_ref=None):
        """Predict using trained CatBoost on pooled latent features."""
        self.encoder.eval()
        with torch.no_grad():
            z = self.encoder(torch.tensor(X, dtype=torch.float32, device=self.device)).mean(dim=1).cpu().numpy()

        if self.catboost_model is None:
            raise ValueError("CatBoost model not trained yet.")

        n_targets = y_ref.shape[1] if y_ref is not None else max(self.non_constant_idx)+1
        y_pred_full = np.zeros((len(X), n_targets))

        if self.non_constant_idx:
            y_pred_full[:, self.non_constant_idx] = self.catboost_model.predict(z)

        if y_ref is not None:
            for i in range(n_targets):
                if i not in self.non_constant_idx:
                    y_pred_full[:, i] = y_ref[0, i]

        return y_pred_full

# ===== Example usage =====
custom_model_cb = HeadsCatboost(encoder_torch, proj_head, decoder, device)
custom_model_cb.pretrain_encoder(X_train, batch_size=batch_size, epochs=1)
results_dict_cb, z_train_dict_cb, z_test_dict_cb, y_L_dict_cb = custom_model_cb.train_supervised(X_train, y_train_scaled, X_test,
                                                                                                 y_test_scaled, label_fractions)
# # inference
# frac      = 1.0
# y_pred_cb = custom_model_cb.predict(X_test, y_ref=y_test_scaled)
# rmse_cb   = root_mean_squared_error(y_test_scaled, y_pred_cb)
# print(f"CatBoost RMSE on {frac*100:.0f}% labeled: {rmse_cb:.4f}")


In [ ]:
"""Headsup for AE"""
class HeadsupAE(Headsup):
    """Wrapper for FlexibleAutoencoder with projection + supervised head.
    - Encoder/decoder work on sequences [B, T, F].
    - Decoder reconstructs full sequence.
    - Supervised head works on pooled latent (mean over time).
    """
    def __init__(self, ae_model, device: torch.device,
                 contrast_temp: float = 0.5, jitter_strength: float = 0.1,
                 mag_warp_strength: float = 0.1, last_block_lr: float = 5e-4,
                 default_lr: float = 1e-4, supervised_head=None):

        if supervised_head is None:
            supervised_head = TorchWrapper(ae_model.prediction_head)

        super().__init__(
            encoder=None,  # we override
            proj_head=ae_model.projection_head,
            decoder=ae_model.decoder,
            supervised_head=supervised_head,
            device=device,
            contrast_temp=contrast_temp,
            jitter_strength=jitter_strength,
            mag_warp_strength=mag_warp_strength,
            last_block_lr=last_block_lr,
            default_lr=default_lr)
        self.ae_model = ae_model
        self.encoder  = ae_model.encoder

    def _encode(self, X: torch.Tensor) -> torch.Tensor:
        """Encode each timestep independently to satisfy BN layers"""
        B, T, F = X.shape
        Z_list = [self.ae_model.encoder(X[:, t, :]) for t in range(T)]  # list of [B, H]
        Z = torch.stack(Z_list, dim=1)  # [B, T, H]
        return Z

    def _pool_latents(self, Z: torch.Tensor) -> torch.Tensor:
        """Pool latent sequence for supervised head (mean over time)."""
        return Z.mean(dim=1)  # [B, H]

    def _pretrain_single_epoch(self, X_train, batch_size, weights, optimizer, scheduler):
        loss_recon, loss_contrast, loss_total = 0, 0, 0
        for i in range(0, len(X_train), batch_size):
            X_batch = torch.tensor(X_train[i:i+batch_size], dtype=torch.float32, device=self.device)
            if X_batch.size(0) < self.MIN_BATCH_SIZE:
                continue

            # augmentations
            X1, X2 = self._augment(X_batch)

            # encode sequences
            z1, z2 = self._encode(X1), self._encode(X2)

            # project pooled
            h1, h2 = self.proj_head(self._pool_latents(z1)), self.proj_head(self._pool_latents(z2))

            # reconstruct sequence
            x_recon = self.decoder(z1)

            # losses
            loss_contrast = nt_xent_loss(h1, h2, temperature=self.contrast_temp)
            loss_recon    = F.mse_loss(x_recon, X_batch)
            loss_total    = weights["recon"]*loss_recon + weights["contrast"]*loss_contrast

            optimizer.zero_grad()
            loss_total.backward()
            optimizer.step()
            scheduler.step()
        return loss_recon.item(), loss_contrast.item(), loss_total.item()

    def pretrain(self, X_train, batch_size, epochs,
                 weights={"recon":0.1,"contrast":1.0}, lr=1e-3):
        optimizer_pretrain = torch.optim.AdamW(
            list(self.ae_model.encoder.parameters()) +
            list(self.proj_head.parameters()) +
            list(self.decoder.parameters()),
            lr=lr
        )
        max_steps = epochs * (len(X_train)//batch_size)
        scheduler_pretrain = CosineAnnealingLR(
            optimizer_pretrain, T_max=max_steps, eta_min=self.lr_min
        )

        for epoch in range(epochs):
            loss_recon, loss_contrast, loss_total = self._pretrain_single_epoch(
                X_train, batch_size, weights, optimizer_pretrain, scheduler_pretrain
            )
            if epoch % self.PRINT_EVERY == 0:
                print(f"[Pretrain] Epoch {epoch+1}/{epochs}: "
                      f"recon={loss_recon:.4f}, contrast={loss_contrast:.4f}, total={loss_total:.4f}")
        return self.ae_model.encoder, self.proj_head, self.decoder

    def _setup_encoder_optimizer(self, frac: float, lr_train: float = 1e-4):
        """Freeze/unfreeze encoder depending on labeled fraction."""
        if frac == 1.0:
            for p in self.ae_model.encoder.parameters(): p.requires_grad = True
            weights_train = {"pred": 1.0, "recon": 0.5, "contrast": 0.5}
            params = list(self.ae_model.encoder.parameters()) + list(self.proj_head.parameters()) + list(self.decoder.parameters())
            if self.supervised_head: params += list(self.supervised_head.parameters())
            opt = torch.optim.AdamW(params, lr=lr_train)
        elif frac >= 0.5:
            for p in self.ae_model.encoder.parameters(): p.requires_grad = False
            weights_train = {"pred": 1.0, "recon": 0.1, "contrast": 0.1}
            params = list(self.proj_head.parameters()) + list(self.decoder.parameters())
            if self.supervised_head: params += list(self.supervised_head.parameters())
            opt = torch.optim.AdamW(params, lr=lr_train)
        else:
            for p in self.ae_model.encoder.parameters(): p.requires_grad = False
            weights_train = {"pred": 2.0, "recon": 0.0, "contrast": 0.0}
            params = list(self.proj_head.parameters())
            if self.supervised_head: params += list(self.supervised_head.parameters())
            opt = torch.optim.AdamW(params, lr=self.default_lr)
        return weights_train, opt

    def _train_single_epoch(self, X_L, y_L, X_train, optimizer_train, scheduler_train, batch_size, weights_train):
        loss_pred, loss_recon, loss_contrast, loss_total = 0, 0, 0, 0
        for i in range(0, len(X_train), batch_size):
            X_batch_l = X_L[i:i+batch_size]
            y_batch_l = y_L[i:i+batch_size]
            if X_batch_l.size(0) < self.MIN_BATCH_SIZE: continue

            start_unlab = i % len(X_train)
            X_batch_u = torch.tensor(X_train[start_unlab:start_unlab+batch_size], dtype=torch.float32, device=self.device)
            if X_batch_u.size(0) < self.MIN_BATCH_SIZE: continue

            # encode sequences
            z_l, z_u = self._encode(X_batch_l), self._encode(X_batch_u)

            # supervised prediction
            if self.supervised_head:
                y_hat = self.supervised_head(self._pool_latents(z_l))
                loss_pred = F.mse_loss(y_hat, y_batch_l)

            # reconstruction
            x_recon_l, x_recon_u = self.decoder(z_l), self.decoder(z_u)
            loss_recon = (F.mse_loss(x_recon_l, X_batch_l) + F.mse_loss(x_recon_u, X_batch_u)) / 2

            # contrastive
            X1_l, X2_l = self._augment(X_batch_l)
            X1_u, X2_u = self._augment(X_batch_u)
            z1_l, z2_l = self._encode(X1_l), self._encode(X2_l)
            z1_u, z2_u = self._encode(X1_u), self._encode(X2_u)
            h1_l, h2_l = self.proj_head(self._pool_latents(z1_l)), self.proj_head(self._pool_latents(z2_l))
            h1_u, h2_u = self.proj_head(self._pool_latents(z1_u)), self.proj_head(self._pool_latents(z2_u))
            loss_contrast = (nt_xent_loss(h1_l, h2_l, self.contrast_temp) +
                             nt_xent_loss(h1_u, h2_u, self.contrast_temp)) / 2

            # total loss
            loss_total = (weights_train["pred"]*loss_pred +
                          weights_train["recon"]*loss_recon +
                          weights_train["contrast"]*loss_contrast)

            optimizer_train.zero_grad()
            loss_total.backward()
            optimizer_train.step()
            scheduler_train.step()
        return loss_pred.item(), loss_recon.item(), loss_contrast.item(), loss_total.item()

# --- model setup ---
input_dim = X_train.shape[2]
layer_dims = [input_dim, 16]  # encoder dims per timestep
ae_model = ae.FlexibleAutoencoder(layer_dims=layer_dims, pred_dim=y_train_scaled.shape[1],
                                  dropout_prob=0.05, projection_dim=16)
# --- run ---
headsup_model = HeadsupAE(ae_model, device=device, supervised_head=TorchWrapper(ae_model.prediction_head))
headsup_model.pretrain(X_train, batch_size=32, epochs=300)
results_dict, z_train_dict, z_test_dict, y_L_dict = headsup_model.training_loop(X_train, y_train_scaled, X_test, y_test_scaled,
                                                                               batch_size=32, train_epochs_finetune=300,
                                                                               label_fractions=[1.0, 0.5, 0.25, 0.1])
# ==== downstream / external inference ====
frac      = 1.0  # choose fraction to evaluate
z_train   = z_train_dict[frac]
z_test_np = z_test_dict[frac]
y_L       = y_L_dict[frac]

linreg_loss, catboost_loss, unsupervised_rmse, \
    rf_rmse, el_rmse = Preds.evaluate_models_on_dataset(z_train, y_L, z_test_np, y_test_scaled, label_frac=frac)

print(f"dataset: {desired_dataset}, method: ts2vec")
print("  RMSE   | LinReg | CatBoost | Cluster | RForest | ElasticNet | NN")
print(f"& Z (AE) & {linreg_loss:.4f} & {catboost_loss:.4f} & {unsupervised_rmse:.4f} & {rf_rmse:.4f} & {el_rmse:.4f} \\")



In [ ]:
"best results stored here!!!"

"option 1: Loop2 Headsup (no predictor q)"

# ===== params =====
set_seed(42)
train_epochs_pretrain= 150
train_epochs_finetune= 150
batch_size           = 16
decoder_hidden_dims  = [16, 64, 128]
optimizer_lr         = 1e-3
projection_dim       = 16

label_fractions = [1.0, 0.5, 0.25, 0.01]
results_dict    = {}
cb_results_dict = {}

# ===== init models =====
encoder   = TS2VecEncoder(z_pooling=z_pooling_method, device=device)
encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
            depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=ts2vec_epochs)

proj_head = ProjectionHead(input_dim=ts2vec_latent_dims, proj_dim=projection_dim).to(device)
decoder   = Decoder(latent_dim=ts2vec_latent_dims, output_shape=(X_train.shape[1], X_train.shape[2]),
                    hidden_sizes=decoder_hidden_dims).to(device)
sup_head  = MLPHead(input_dim=ts2vec_latent_dims, output_dim=y_train_scaled.shape[1],
                    hidden_sizes=predictor_hidden_sizes, dropout=predictor_dropout, device=device)

# ===== optimizer =====
params    = list(proj_head.parameters()) + list(decoder.parameters()) + list(sup_head.model.parameters())
optimizer = torch.optim.AdamW(params, lr=optimizer_lr)
max_steps = train_epochs_pretrain * (len(X_train) // batch_size)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps, eta_min=1e-5)

# ===== 1) Pretraining loop (contrastive + optional recon) =====
weights = {"pred": 0.0, "recon": 0.1, "contrast": 1.0}
for epoch in range(train_epochs_pretrain):
    for i in range(0, len(X_train), batch_size):
        X_batch = X_train[i:i+batch_size]

        # augment
        X1 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "jitter", device, 0.1)
        X2 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "mag_warp", device, 0.1)

        # encode
        z1 = torch.tensor(encoder.encode(X1.cpu().numpy()), dtype=torch.float32, device=device)
        z2 = torch.tensor(encoder.encode(X2.cpu().numpy()), dtype=torch.float32, device=device)

        # projection
        h1, h2 = proj_head(z1), proj_head(z2)

        # losses
        loss_contrast = nt_xent_loss(h1, h2, temperature=0.5)
        x_recon       = decoder(z1)
        loss_recon    = F.mse_loss(x_recon, torch.tensor(X_batch, dtype=torch.float32, device=device))

        # total weighted loss (no pred loss in pretrain)
        loss = weights["recon"]*loss_recon + weights["contrast"]*loss_contrast

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

    if epoch % 5 == 0:
        print(f"[Pretrain] Epoch {epoch+1}/{train_epochs_pretrain}: recon={loss_recon.item():.4f}, "
              f"contrast={loss_contrast.item():.4f}, total={loss.item():.4f}")

# ===== 2) Fine-tuning loop (combined labeled + unlabeled) =====
weights = {"pred": 1.0, "recon": 0.5, "contrast": 0.5}

for frac in label_fractions:
    n_samples = int(len(X_train) * frac)
    X_L = X_train[:n_samples]
    y_L = y_train_scaled[:n_samples]

    for epoch in range(train_epochs_finetune):
        for i in range(0, len(X_train), batch_size):
            # --- Labeled subset in batch ---
            X_batch_l = X_L[i:i+batch_size]
            y_batch_l = y_L[i:i+batch_size]
            if len(X_batch_l) == 0: #skip empty batches
                continue

            # --- Unlabeled subset in batch (fill to batch_size) ---
            start_unlab = i % len(X_train)  # rotate over full dataset
            X_batch_u   = X_train[start_unlab:start_unlab + batch_size]
            if len(X_batch_u) == 0: #skip empty batches
                continue

            # Encode
            z_l = torch.tensor(encoder.encode(X_batch_l), dtype=torch.float32, device=device)
            z_u = torch.tensor(encoder.encode(X_batch_u), dtype=torch.float32, device=device)

            # Supervised loss on labeled
            y_hat     = sup_head.model(z_l)
            loss_pred = F.mse_loss(y_hat, torch.tensor(y_batch_l, dtype=torch.float32, device=device))

            # Reconstruction + contrastive on labeled + unlabeled
            x_recon_l  = decoder(z_l)
            x_recon_u  = decoder(z_u)
            loss_recon = (F.mse_loss(x_recon_l, torch.tensor(X_batch_l, dtype=torch.float32, device=device)) +
                          F.mse_loss(x_recon_u, torch.tensor(X_batch_u, dtype=torch.float32, device=device))) / 2

            # Contrastive
            X1_l = make_augmentations(torch.tensor(X_batch_l, dtype=torch.float32, device=device), "jitter", device, 0.1)
            X2_l = make_augmentations(torch.tensor(X_batch_l, dtype=torch.float32, device=device), "mag_warp", device, 0.1)
            X1_u = make_augmentations(torch.tensor(X_batch_u, dtype=torch.float32, device=device), "jitter", device, 0.1)
            X2_u = make_augmentations(torch.tensor(X_batch_u, dtype=torch.float32, device=device), "mag_warp", device, 0.1)

            z1_l = torch.tensor(encoder.encode(X1_l.cpu().numpy()), dtype=torch.float32, device=device)
            z2_l = torch.tensor(encoder.encode(X2_l.cpu().numpy()), dtype=torch.float32, device=device)
            z1_u = torch.tensor(encoder.encode(X1_u.cpu().numpy()), dtype=torch.float32, device=device)
            z2_u = torch.tensor(encoder.encode(X2_u.cpu().numpy()), dtype=torch.float32, device=device)

            h1_l, h2_l = proj_head(z1_l), proj_head(z2_l)
            h1_u, h2_u = proj_head(z1_u), proj_head(z2_u)
            loss_contrast = (nt_xent_loss(h1_l, h2_l, temperature=0.5) +
                             nt_xent_loss(h1_u, h2_u, temperature=0.5)) / 2

            # Weighted total
            loss = weights["pred"]*loss_pred + weights["recon"]*loss_recon + weights["contrast"]*loss_contrast

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

        if epoch % 5 == 0:
            print(f"[Finetune {frac*100:.0f}%] Epoch {epoch+1}/{train_epochs_finetune}: "
                  f"pred={loss_pred.item():.4f}, recon={loss_recon.item():.4f}, "
                  f"contrast={loss_contrast.item():.4f}, total={loss.item():.4f}")

    # ===== inference =====
    proj_head.eval()
    decoder.eval()
    sup_head.model.eval()
    with torch.no_grad():
        z_test = torch.tensor(encoder.encode(X_test), dtype=torch.float32, device=device)
        y_pred = sup_head.model(z_test).cpu().numpy()
        results_dict[frac] = root_mean_squared_error(y_test_scaled, y_pred)
        print(f"RMSE with {frac*100:.0f}% labeled: {results_dict[frac]:.4f}")

        # ==========
        # Encode labeled data for CatBoost training
        z_train = torch.tensor(encoder.encode(X_L), dtype=torch.float32, device=device).cpu().numpy()
        z_test  = z_test.cpu().numpy()  # move to numpy for CatBoost

        # Train & predict with CatBoost
        cb_model, y_pred_cb, rmse_cb, non_constant_idx = Preds().predict_catboost_multioutput(z_train, y_L, z_test, y_test_scaled)
        print(f"RMSE with {frac*100:.0f}% labeled (CatBoost): {rmse_cb:.4f}")
        cb_results_dict[frac] = rmse_cb
        # =======

# --- Display nicely ---
print("\nRMSE for different labeled fractions:")
for frac, rmse_val in results_dict.items():
    print(f"  {int(frac*100):>3}% label: RMSE = {rmse_val:.4f}")

for frac, rmse_val in cb_results_dict.items():
    print(f"  {int(frac*100):>3}% label (CB): RMSE = {rmse_val:.4f}")



# """option 2 (old): Loop2 Headsup (no predictor q)"""

# set_seed(42)

# # ===== params =====
# train_epochs = 50
# batch_size   = 16
# warmup_steps = 1000
# max_steps    = train_epochs * (len(X_train) // batch_size)
# step         = 0

# decoder_hidden_dims = [16, 64, 128]
# optimizer_lr        = 1e-3
# projection_dim      = 16  # smaller than z

# # ===== init models =====
# encoder   = TS2VecEncoder(z_pooling=z_pooling_method, device=device)
# encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
#             depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=ts2vec_epochs)

# proj_head = ProjectionHead(input_dim=ts2vec_latent_dims, proj_dim=projection_dim).to(device)
# decoder   = Decoder(latent_dim=ts2vec_latent_dims, output_shape=(X_train.shape[1], X_train.shape[2]),
#                     hidden_sizes=decoder_hidden_dims).to(device)
# sup_head  = MLPHead(input_dim=ts2vec_latent_dims, output_dim=y_train_scaled.shape[1],
#                     hidden_sizes=predictor_hidden_sizes,
#                     lr=predictor_lr, epochs=1, dropout=predictor_dropout, device=device)

# # ===== optimizer =====
# params    = list(proj_head.parameters()) + list(decoder.parameters()) + list(sup_head.model.parameters())
# optimizer = torch.optim.AdamW(params, lr=optimizer_lr, weight_decay=1e-1) #1e-1 = 0.8586
# # scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps, eta_min=1e-5)

# # ===== training loop =====
# for epoch in range(train_epochs):
#     # predictor_lr = schedule_learning_rate(step, max_steps, lr_0=1e-3, lr_end=1e-5, schedule_type="linear")
#     for i in range(0, len(X_train), batch_size):
#         step += 1
#         X_batch, y_batch = X_train[i:i+batch_size], y_train_scaled[i:i+batch_size]

#         # augment
#         X1 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "jitter", device, 0.1)
#         X2 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "mag_warp", device, 0.1)

#         # encode
#         z1 = torch.tensor(encoder.encode(X1.cpu().numpy()), dtype=torch.float32, device=device)
#         z2 = torch.tensor(encoder.encode(X2.cpu().numpy()), dtype=torch.float32, device=device)

#         # projections
#         h1, h2 = proj_head(z1), proj_head(z2)

#         # losses
#         loss_contrast = Losses.compute_byol_loss(h1, h2.detach())  # no predictor q
#         x_recon       = decoder(z1)
#         loss_recon    = F.mse_loss(x_recon, torch.tensor(X_batch, dtype=torch.float32, device=device))
#         y_hat         = sup_head.model(z1)
#         loss_pred     = F.mse_loss(y_hat, torch.tensor(y_batch, dtype=torch.float32, device=device))

#         # weighted total loss
#         weights = {"pred": 1.0, "recon": 0.0, "contrast": 0.0}
#         loss = weights["pred"]*loss_pred + weights["recon"]*loss_recon + weights["contrast"]*loss_contrast

#         # backprop
#         optimizer.zero_grad()
#         loss.backward()
#         # torch.nn.utils.clip_grad_norm_(params, 1.0)
#         # torch.nn.utils.clip_grad_norm_(list(sup_head.model.parameters()), max_norm=1.0)
#         optimizer.step()
#         # scheduler.step()

#     print(f"Epoch {epoch+1}/{train_epochs}: "
#           f"pred={loss_pred.item():.4f}, recon={loss_recon.item():.4f}, "
#           f"contrast={loss_contrast.item():.4f}, total={loss.item():.4f}")

# # ===== inference =====
# proj_head.eval()
# decoder.eval()
# sup_head.model.eval()

# with torch.no_grad():
#     z_test = torch.tensor(encoder.encode(X_test), dtype=torch.float32, device=device)
#     y_pred = sup_head.model(z_test).cpu().numpy()

# rmse = root_mean_squared_error(y_test_scaled, y_pred)
# print(f"Headsup RMSE: {rmse:.4f}")


In [ ]:
"""[real data] Conditional VAE. Train on train set, inference on test set"""
should_we_include_X = True
batch_size = 32

# === Model parameters ===
hidden_dim = 64
latent_dim = 10
dropout    = 0.05

# === Training parameters ===
epochs              = 100
learning_rate       = 1e-3
weight_decay        = 1e-5
scheduler_patience  = 5
early_stop_patience = 20
save_path           = f"{encoders_folder}/best_cvae.pth"

# === Dataset from df (downsample + split) ===
df_small = df.sample(frac=0.1, random_state=42)  # keep 10%
X = df_small.drop(columns=y_cols + [time_col_name], errors="ignore").values
y = df_small[y_cols].values

# === dont use traintestsplit for timeseries (it randomly shuffles, breaking temporal order)
split_idx       = int(len(df_small) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# === Scaling ===
x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_train  = x_scaler.fit_transform(X_train)
X_test   = x_scaler.transform(X_test)
y_train  = y_scaler.fit_transform(y_train)
y_test   = y_scaler.transform(y_test)

# === Convert to tensors ===
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)
y_dim   = y_train.shape[1]
x_dim   = X_train.shape[1]

# === Datasets + Loaders ===
train_dataset = TensorDataset(y_train, X_train)
test_dataset  = TensorDataset(y_test, X_test)
train_loader  = DataLoader(train_dataset, batch_size=batch_size, drop_last=True)
val_loader    = DataLoader(test_dataset,  batch_size=batch_size, drop_last=True)

# === Model + Optimizer + Scheduler===
cond_vae  = ae.ConditionalVAE(y_dim=y_dim, x_dim=x_dim, hidden_dim=hidden_dim, latent_dim=latent_dim, dropout=dropout).to(device)
optimizer = torch.optim.AdamW(cond_vae.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=scheduler_patience)

# === Trainer ===
trainer   = train_ae.TrainConditionalVAE()
best_loss = trainer.train_cvae(device=device, cvae=cond_vae, epochs=epochs,train_loader=train_loader, optimizer=optimizer,
                               scheduler=scheduler, val_loader=val_loader, patience=early_stop_patience, save_path=save_path)
print(f"Best validation loss: {best_loss:.4f}")

# === inference ===
cond_vae.load_state_dict(torch.load(save_path))
cond_vae.eval()

# === Full test set prediction ===
batch_size_pred   = 64  # adjust based on memory
all_y_pred_scaled = []

with torch.no_grad():
    for batch in range(0, len(X_test), batch_size_pred):
        x_batch = X_test[batch : batch + batch_size_pred].to(device)
        if not should_we_include_X:
            x_batch = torch.zeros(x_batch.size(0), x_dim).to(device)
        z_batch             = torch.randn(x_batch.size(0), latent_dim).to(device)
        y_batch_pred_scaled = cond_vae.decode(z_batch, x=x_batch)
        all_y_pred_scaled.append(y_batch_pred_scaled.cpu())

# Concatenate all batches + MSE
y_pred_scaled = torch.cat(all_y_pred_scaled, dim=0).numpy()
mse = mean_squared_error(y_test, y_pred_scaled)
mae = mean_absolute_error(y_test, y_pred_scaled)
print(f"Scaled y: test MSE = {mse:.4f}, test MAE = {mae:.4f}")

y_pred      = y_scaler.inverse_transform(y_pred_scaled)
y_true_orig = y_scaler.inverse_transform(y_test.numpy())

plt.plot(y_true_orig, label="True")
plt.plot(y_pred, label="Predicted")
plt.title(f"Cond. VAE ('{desired_dataset}' dataset)")
plt.xlabel("Timestep")
plt.ylabel("y value")
plt.legend()
plt.show()


# NOTE: consider this repo for Conditional VAE (https://github.com/unnir/cVAE/blob/master/cvae.py)
# or https://freedium.cfd/https://medium.com/@sofeikov/implementing-conditional-variational-auto-encoders-cvae-from-scratch-29fcbb8cb08f

In [ ]:
"""Multistep LSTM predictor"""
import torch.optim as optim
from forecasting_module import BaseForecaster

class FlexibleLSTM(nn.Module):
    def __init__(self, input_dim: int, hidden_size: int, output_dim: int):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])  # last step only

class FlexibleLSTMTrainer:
    def __init__(self, model, lr=1e-3, epochs=30, device="cpu"):
        self.model = model.to(device)
        self.lr = lr
        self.epochs = epochs
        self.device = device
        self.loss_fn = nn.MSELoss()

    def _make_batches(self, y, X, seq_len, horizon):
        """Convert timeseries to (X_seq, y_future) windows."""
        data = y if X is None else np.concatenate([y, X], axis=1)
        X_batches, y_batches = [], []
        for i in range(len(data) - seq_len - horizon):
            X_batches.append(data[i:i+seq_len])
            y_batches.append(y[i+seq_len:i+seq_len+horizon])
        return (torch.tensor(np.stack(X_batches), dtype=torch.float32),
                torch.tensor(np.stack(y_batches), dtype=torch.float32),)

    def fit(self, y_train, X_train, seq_len, horizon, y_val=None, X_val=None):
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        Xb, yb = self._make_batches(y_train, X_train, seq_len, horizon)
        Xb, yb = Xb.to(self.device), yb.to(self.device)

        for epoch in range(self.epochs):
            self.model.train()
            optimizer.zero_grad()
            preds = self.model(Xb)
            loss  = self.loss_fn(preds, yb[:, -1, :])  # predict horizon last-step
            loss.backward()
            optimizer.step()
            if (epoch+1) % 10 == 0:
                print(f"Epoch {epoch+1}, loss={loss.item():.4f}")

    def predict_seq2seq(self, y_test, X_test, seq_len, horizon):
        Xb, yb = self._make_batches(y_test, X_test, seq_len, horizon)
        self.model.eval()
        with torch.no_grad():
            preds = self.model(Xb.to(self.device)).cpu().numpy()
        return preds

    def predict_autoreg(self, y_test, X_test, seq_len, horizon):
        """Step-by-step forecasting with feedback of predictions."""
        data  = y_test if X_test is None else np.concatenate([y_test, X_test], axis=1)
        preds = []
        self.model.eval()
        with torch.no_grad():
            for i in range(len(data) - seq_len - horizon):
                window = torch.tensor(data[i:i+seq_len], dtype=torch.float32).unsqueeze(0).to(self.device)
                pred   = self.model(window).cpu().numpy()
                preds.append(pred)
                # feed prediction back only into y part (not exogenous)
                data[i+seq_len] = np.concatenate([pred[0], data[i+seq_len, y_test.shape[1]:]])
        return np.array(preds)


class LSTMForecaster(BaseForecaster):
    def __init__(self, df, time_col: str, y_cols: list[str],
                 use_exogenous: bool = False, seq_len: int = 30, horizon: int = 10,
                 hidden_size: int = 64, epochs: int = 30, lr: float = 1e-3):
        super().__init__(df, time_col, y_cols)
        self.use_exogenous = use_exogenous
        self.seq_len = seq_len
        self.horizon = horizon
        self.hidden_size = hidden_size
        self.epochs = epochs
        self.lr = lr

        # Exogenous columns = all columns minus time + y
        if self.use_exogenous:
            self.X_cols = [c for c in df.columns if c not in [time_col] + y_cols]
        else:
            self.X_cols = []

    def _prepare_data(self, df):
        """Return tensors for y (and X if exogenous)."""
        y = df[self.y_cols].values.astype(np.float32)
        if self.use_exogenous and self.X_cols:
            X = df[self.X_cols].values.astype(np.float32)
        else:
            X = None
        return y, X

    def fit(self, df_train, df_val=None):
        y_train, X_train = self._prepare_data(df_train)
        y_val, X_val = (None, None) if df_val is None else self._prepare_data(df_val)

        self.model = FlexibleLSTM(
            input_dim=len(self.y_cols) + (X_train.shape[1] if X_train is not None else 0),
            hidden_size=self.hidden_size,
            output_dim=len(self.y_cols))

        trainer = FlexibleLSTMTrainer(self.model, lr=self.lr, epochs=self.epochs)
        trainer.fit(y_train, X_train, self.seq_len, self.horizon,
                    y_val=y_val, X_val=X_val)
        self.trainer = trainer

    def predict(self, df_test, autoregressive: bool = False):
        y_test, X_test = self._prepare_data(df_test)
        if autoregressive:
            return self.trainer.predict_autoreg(y_test, X_test, self.seq_len, self.horizon)
        else:
            return self.trainer.predict_seq2seq(y_test, X_test, self.seq_len, self.horizon)

    def forecast_lstm(self, train_df, test_df, horizon: int,
                      use_exogenous: bool = True, stepwise: bool = False):
        """Train + forecast like SARIMAX.
        Args:
            train_df, test_df: pandas DataFrames
            horizon: forecast horizon
            use_exogenous: toggle exogenous features
            stepwise: if True → autoregressive, else → direct multi-step
        Returns:
            forecast_dict: {y_col: np.array predictions}
            y_true: true target values (scaled)"""
        # override exogenous toggle for this run
        self.use_exogenous = use_exogenous
        if use_exogenous:
            self.X_cols = [c for c in train_df.columns if c not in [self.time_col] + self.y_cols]
        else:
            self.X_cols = []

        self.fit(train_df)
        preds = self.predict(test_df, autoregressive=stepwise)
        y_true, _ = self._prepare_data(test_df)

        if preds.ndim == 1:
            preds = preds.reshape(-1, 1)

        forecast_dict = {col: preds[:, i] for i, col in enumerate(self.y_cols)}
        self.forecast_dfs = {col: forecast_dict[col] for col in self.y_cols}
        return forecast_dict, y_true


"""LSTM run"""
n_windows = 5
horizon = 10       # prediction steps
seq_len = 30       # past steps
# min_window_len = seq_len + horizon  # ensure each window is long enough

logging.basicConfig(level=logging.INFO)

forecaster = LSTMForecaster(df_uniform, time_col=time_col_name, y_cols=y_cols, seq_len=seq_len)

print("==== single window evaluation ====")
# forecast_dict, y_true_scaled = forecaster.forecast_lstm(
#     df_train, df_test, horizon,
#     use_exogenous=True,   # or False
#     stepwise=False)        # sequence prediction (False) or step-by-step (True)

# forecast_df_temp = forecaster.forecast_dfs[y_cols[0]]

print("==== multi window evaluation ====")
all_forecasts, mae_list, sizes = {}, [], []
windows_list = ForecastUtils.make_windows(
    df_uniform, n_windows=n_windows, horizon_len=horizon,
    horizon_frac=horizon_frac, min_window_len=min_window_len, start_point=0)

for i, (train_df, test_df) in enumerate(windows_list):
    print(f"Window {i}: train {train_df.shape}, test {test_df.shape}")
    forecast_dict, y_true_scaled = forecaster.forecast_lstm(
        train_df, test_df, horizon,
        use_exogenous=True,
        stepwise=True)
    all_forecasts[f"window_{i}"] = forecast_dict

    for j, target in enumerate(y_cols):
        y_true = y_true_scaled[:, j]
        y_pred = forecast_dict[target]
        mae = mean_absolute_error(y_true, y_pred)
        mae_list.append(mae)
        sizes.append(len(y_true))

# ---- Weighted MAE ----
weighted_mae = sum(MAE * n for MAE, n in zip(mae_list, sizes)) / sum(sizes)
std_mae      = math.sqrt(sum((MAE - weighted_mae) ** 2 * n for MAE, n in zip(mae_list, sizes)) / sum(sizes))
print(f"Final MAE: mean ± std= {{{weighted_mae:.4f}}}{{{std_mae:.4f}}}")


In [ ]:
"""TimesFM"""
from timesfm import TimesFmHparams, TimesFm, TimesFmCheckpoint

# Hyperparameters
hparams = TimesFmHparams(
    backend="jax",
    per_core_batch_size=32,
    horizon_len=128,
    num_layers=20,
    context_len=512,
    use_positional_embedding=True,)

# Local checkpoint folder containing 'checkpoint'
checkpoint = TimesFmCheckpoint(local_dir="interim_data")

# Initialize model
model = TimesFm(hparams=hparams, checkpoint=checkpoint)

# Load manually (if needed)
# model.load_from_checkpoint("interim_data/checkpoint", checkpoint_type=CheckpointType.FLAX)
model.load_from_checkpoint(repo_id="google/timesfm-1.0-200m")#, checkpoint_type=CheckpointType.FLAX)

# Forecast example
y = np.arange(100)
forecast = model.forecast(y, horizon=10)
print(forecast)


In [ ]:
"""Conditional VAE. Train on train set, inference on test set"""

# === Data parameters ===
n_samples  = 1000
y_dim      = 1
x_dim      = 10  # optional
batch_size = 32

# === Model parameters ===
hidden_dim = 64
latent_dim = 10
dropout    = 0.05

# === Training parameters ===
epochs              = 100
learning_rate       = 1e-3
weight_decay        = 1e-5
scheduler_patience  = 5
early_stop_patience = 10
save_path           = f"{encoders_folder}/best_cvae.pth"

# === Dataset ===
y_data  = torch.randn(n_samples, y_dim)
x_data  = torch.randn(n_samples, x_dim)
dataset = TensorDataset(y_data, x_data)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(dataset, batch_size=batch_size)

cond_vae  = ae.ConditionalVAE(y_dim=y_dim, x_dim=x_dim, hidden_dim=hidden_dim,
                              latent_dim=latent_dim, dropout=dropout).to(device)

# === Optimizer + Scheduler ===
optimizer = torch.optim.AdamW(cond_vae.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=scheduler_patience)

# === Trainer ===
trainer   = train_ae.TrainConditionalVAE()
best_loss = trainer.train_cvae(device=device, cvae=cond_vae, epochs=epochs, train_loader=train_loader, optimizer=optimizer,
                               scheduler=scheduler, val_loader=val_loader, patience=early_stop_patience, save_path=save_path)
print(f"Best validation loss: {best_loss:.4f}")

# === Load best model for inference ===
cond_vae.load_state_dict(torch.load(save_path))
cond_vae.eval()

# === Example inference ===
n_rows_gen   = 5
z_sample     = torch.randn(n_rows_gen, latent_dim).to(device)
is_x_present = True
if is_x_present:
    x_sample = torch.randn(n_rows_gen, x_dim).to(device)
else:
    x_sample = torch.zeros(n_rows_gen, x_dim).to(device)

y_sample = cond_vae.decode(z_sample, x=x_sample)
print(f"Generated y sample: {y_sample}")


In [ ]:
"""Applying metrics on X vs z"""

# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# # Split your original data
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Encode train/test via VAE
# z_train = vae.encoder.predict(X_train)  # shape (num_samples, latent_dim)
# z_test  = vae.encoder.predict(X_test)

# # --- Predictor on raw X ---
# model_X = LinearRegression()
# model_X.fit(X_train.reshape(X_train.shape[0], -1), y_train)  # flatten if needed
# y_pred_X = model_X.predict(X_test.reshape(X_test.shape[0], -1))

# # --- Predictor on latent z ---
# model_z = LinearRegression()
# model_z.fit(z_train, y_train)
# y_pred_z = model_z.predict(z_test)

# # --- Metrics ---
# def print_metrics(y_true, y_pred, name):
#     mse = mean_squared_error(y_true, y_pred)
#     mae = mean_absolute_error(y_true, y_pred)
#     r2  = r2_score(y_true, y_pred)
#     print(f"{name} -> MSE: {mse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")

# print_metrics(y_test, y_pred_X, "Predictor on X")
# print_metrics(y_test, y_pred_z, "Predictor on z")


In [ ]:
from downsampling import LTTBDownsampler, OtherDownsamplers

def df_plotter(df, df_downsampled, col_to_plot: int, time_col: str) -> None:
    plt.figure(figsize=(7, 4))
    plt.plot(df[time_col], df.iloc[:, col_to_plot], alpha=0.8, linewidth=2)
    plt.plot(df_downsampled[time_col], df_downsampled.iloc[:, col_to_plot], alpha=0.8, linewidth=1)
    plt.legend(['Original', 'Downsampled'])
    plt.tight_layout()
    plt.show()

def _add_time_features(df: pd.DataFrame, y_col: str, time_col: str = 'Datetime', lag_amount: int = 1, rolling_window: int = 3) -> pd.DataFrame:
    df = df.copy()
    df.loc[:, 'hour']      = df[time_col].dt.hour
    df.loc[:, 'dayofweek'] = df[time_col].dt.dayofweek
    df.loc[:, f'lag_{lag_amount}'] = df[y_col].shift(lag_amount)
    df.loc[:, f'rolling_mean_{rolling_window}'] = df[y_col].rolling(rolling_window).mean()
    return df.dropna()


class SplitterScaler:
    @staticmethod
    def split_X_and_y(df, time_col: str, y) -> tuple:
        X = df.drop([time_col]+ y, axis=1)
        y = df[y]
        return X, y
        # y1= df[['PowerConsumption_Zone1']]
        # y2= df[['PowerConsumption_Zone2']]
        # y3= df[['PowerConsumption_Zone3']]
        # return X, y, y1, y2, y3

    @staticmethod
    def traintest_split_then_scale(X: pd.DataFrame, y: pd.Series) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, StandardScaler]:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        scaler_X       = StandardScaler()
        X_train_scaled = scaler_X.fit_transform(X_train)
        X_test_scaled  = scaler_X.transform(X_test)

        scaler_y       = StandardScaler()
        y_train_scaled = scaler_y.fit_transform(y_train)
        y_test_scaled  = scaler_y.transform(y_test)
        return X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled, scaler_y

    @staticmethod
    def scale_X_and_y(X: pd.DataFrame, y: pd.DataFrame):
        """Scale features X and target y using StandardScaler.
        Returns scaled arrays and fitted scalers"""
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        X_scaled = scaler_X.fit_transform(X)
        y_scaled = scaler_y.fit_transform(y)
        return X_scaled, y_scaled, (scaler_X, scaler_y)

    @staticmethod
    def scale_X_and_y_latents(X_train: pd.DataFrame | np.ndarray, X_valid: pd.DataFrame | np.ndarray,
                            y_train: pd.DataFrame | np.ndarray, y_valid: pd.DataFrame | np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, StandardScaler, StandardScaler]:
        """Fit scalers on X_train and y_train, transform both train and valid sets.
        Returns scaled X_train, X_valid, y_train, y_valid, scaler_X, scaler_y"""
        scaler_X  = StandardScaler().fit(X_train)
        X_train_s = scaler_X.transform(X_train)
        X_valid_s = scaler_X.transform(X_valid)

        y_train   = np.asarray(y_train).reshape(-1, 1)
        y_valid   = np.asarray(y_valid).reshape(-1, 1)
        scaler_y  = StandardScaler().fit(y_train)
        y_train_s = scaler_y.transform(y_train)
        y_valid_s = scaler_y.transform(y_valid)

        return X_train_s, X_valid_s, y_train_s, y_valid_s, scaler_X, scaler_y


class TrainerAndEvaluator:

    @staticmethod
    def train_catboost_model(X_train_scaled, y_train_scaled):
        model = CatBoostRegressor(iterations=1300, depth=8, learning_rate=0.15, l2_leaf_reg=2, loss_function='RMSE',
                                  random_seed=42, verbose=0, early_stopping_rounds=100, task_type='CPU', bagging_temperature=3)
        model.fit(X_train_scaled, y_train_scaled.ravel())
        return model

    @staticmethod
    def train_and_evaluate_model(train_df: Optional[pd.DataFrame] = None, test_df: Optional[pd.DataFrame] = None,
                                 X_train: Optional[np.ndarray] = None, X_test: Optional[np.ndarray] = None,
                                 y_train: Optional[np.ndarray] = None, y_test: Optional[np.ndarray] = None,
                                 scaler_X: Optional[StandardScaler] = None) -> Tuple[float, float]:
        """Train and evaluate CatBoost. If X/y provided, use them directly.
        Otherwise, extract features from train_df/test_df"""
        
        if X_train is None or y_train is None:
            # Mode 1: from DataFrames
            X, y = SplitterScaler.split_X_and_y(train_df, time_col_name, y_cols)
            X_train_s, X_test_s, y_train_s, y_test_s, _ = SplitterScaler.traintest_split_then_scale(X, y)
        else:
            # Mode 2: from arrays
            if scaler_X:
                X_train_s = scaler_X.transform(X_train)
                X_test_s  = scaler_X.transform(X_test)
            else:
                X_train_s, X_test_s = X_train, X_test  # already scaled
            y_train_s, y_test_s = y_train, y_test

        model  = TrainerAndEvaluator.train_catboost_model(X_train_s, y_train_s)
        y_pred = model.predict(X_test_s)

        # --- metrics ---
        if y_test_s.ndim == 1 or y_test_s.shape[1] == 1:
            rmse  = float(np.sqrt(mean_squared_error(y_test_s, y_pred)))
            nrmse = float(rmse / (y_test_s.max() - y_test_s.min()))
        else:
            rmses = [np.sqrt(mean_squared_error(y_test_s[:, i], y_pred[:, i]))
                     for i in range(y_test_s.shape[1])]
            rmse  = float(np.mean(rmses))
            nrmse = float(np.mean([
                r / (y_test_s[:, i].max() - y_test_s[:, i].min())
                for i, r in enumerate(rmses)]))
        return rmse, nrmse

def downsample_and_evaluate(df, new_points_per_col, compression_ratio_list, rmse_list, nrmse_list, method, rdp_epsilon=0.1):
    """Downsample df, evaluate model on original and compressed, store metrics."""
    if method == "union_lttb":
        df_downsampled = LTTBDownsampler.downsample_using_lttb_union(df, time_col=time_col_name, max_points_per_col=new_points_per_col)
    elif method == "intersection_lttb":
        df_downsampled = LTTBDownsampler.downsample_using_lttb_intersection(df, time_col=time_col_name, max_points_per_col=new_points_per_col)
    elif method == "union_lttb_with_spikes":
        df_downsampled = LTTBDownsampler.downsample_lttb_union_with_spikes(df, time_col=time_col_name, max_points_per_col=new_points_per_col,\
                                                                           spike_thresh=4, spike_method='mad')
    elif method == "union_rdp":
        df_downsampled = OtherDownsamplers.downsample_using_rdp_union(df, time_col=time_col_name, epsilon=rdp_epsilon)
    else:
        raise ValueError(f"Unknown downsampling method: {method}")

    rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(df)
    # print(f"orig. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

    compression_ratio = len(df) / len(df_downsampled)
    # print(f"data compressed {compression_ratio:.2f}x")
    rmse_c, nrmse_c = TrainerAndEvaluator.train_and_evaluate_model(df_downsampled)
    # print(f"compress. RMSE: {rmse_c:.3f}, norm RMSE: {nrmse_c:.3%}")

    compression_ratio_list.append(compression_ratio)
    rmse_list.append(rmse_c)
    nrmse_list.append(nrmse_c)
    return compression_ratio_list, rmse_list, nrmse_list


In [ ]:
"""timeVAE preprocessing (windowing)"""

def create_windows(data: np.ndarray, window_size: int, step_size: int) -> np.ndarray:
    N      = (data.shape[0] - window_size) // step_size + 1
    windows= np.array([data[i*step_size:i*step_size+window_size] for i in range(N)])
    return windows

X_raw    = df.drop(columns=[time_col_name] + y_cols).values
y_raw    = df[y_cols].values

# scale + Transform
scaler_X = StandardScaler().fit(X_raw)
scaler_y = StandardScaler().fit(y_raw)
X_scaled = scaler_X.transform(X_raw)
y_scaled = scaler_y.transform(y_raw)

window_size = 20
step_size   = window_size // 4
X_windows   = create_windows(X_scaled, window_size, step_size)
y_windows   = create_windows(y_scaled, window_size, step_size)

print("X_win shape:", X_windows.shape, "y_win shape:", y_windows.shape)

# Save to ./data/ + timeVAE repo
should_we_save_in_timeVAE = True
if should_we_save_in_timeVAE:
    os.makedirs(interim_data_loc, exist_ok=True)
    np.savez_compressed(f"{interim_data_loc}/{desired_dataset}.npz", data=np.array(X_windows, dtype=np.float32))
    # np.savez_compressed(f"{interim_data_loc}/{desired_dataset}.npz", data=X_windows, target=y_windows)
    np.savez_compressed(f"../../timeVAE/data/{desired_dataset}.npz", data=np.array(X_windows, dtype=np.float32))
print(f"saved {desired_dataset} info to {interim_data_loc}/{desired_dataset}.npz")


In [ ]:
"""predicting y on latents z"""
z_train = np.load(f"{interim_data_loc}/z_train_{desired_dataset}.npy")
z_valid = np.load(f"{interim_data_loc}/z_valid_{desired_dataset}.npy")
print(f"z_train shape: {z_train.shape}, z_valid shape: {z_valid.shape}")

y_squeezed = y_windows[:, -1, 0].reshape(-1, 1)

# y train/valid shaped like z_train/valid
n_train = z_train.shape[0]
n_valid = z_valid.shape[0]
y_train = y_squeezed[:n_train]
y_valid = y_squeezed[n_train:n_train + n_valid]

print("y_train/y_valid shapes:", y_train.shape, y_valid.shape)

# z_train_df = pd.DataFrame(z_train, columns=[f"z{i}" for i in range(z_train.shape[1])])
# z_valid_df = pd.DataFrame(z_valid, columns=[f"z{i}" for i in range(z_valid.shape[1])])

rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(X_train=z_train, X_test=z_valid,
                                                           y_train=y_train, y_test=y_valid)
print(f"RMSE: {rmse:.4f}, NRMSE: {nrmse:.4f}")


In [ ]:
"""catch22"""

def catch22_features_from_windows(X_windows: np.ndarray, y_windows: np.ndarray, which_y: str) -> tuple[np.ndarray, np.ndarray]:
    """Apply catch22 to each window/channel and return (features, targets).
    X_windows: shape (n_windows, window_size, n_channels)
    y_windows: shape (n_windows, window_size, n_targets)"""
    
    n_windows, _, n_channels = X_windows.shape
    feats = np.empty((n_windows, n_channels * 22), dtype=float)
    
    for i in range(n_windows):
        channel_feats = [catch22_all(X_windows[i, :, ch])['values']
                         for ch in range(n_channels)]
        feats[i] = np.concatenate(channel_feats)
    
    # Take last value in each target window
    if which_y == "last":
        y_out = y_windows[:, -1, :]
    elif which_y == "mean":
        y_out = np.mean(y_windows, axis=1)
    return feats, y_out

X_c22, y_c22 = catch22_features_from_windows(X_windows, y_windows, which_y="last")
X_train, X_test, y_train, y_test = train_test_split(X_c22, y_c22, test_size=0.2, random_state=42)
print(f"shapes: X_train {X_train.shape}, X_test {X_test.shape}, y_train {y_train.shape}, y_test {y_test.shape}")

# X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled, _ = SplitterScaler.traintest_split_then_scale(X_c22, y_c22)

if y_train.shape[1] == 1:
    rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(X_train=X_train, X_test=X_test,
                                                               y_train=y_train.ravel(), y_test=y_test.ravel())
    print(f"Train RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")
else:
    rmses   = []
    nrmse_s = []
    for i in range(y_train.shape[1]):
        rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(
            X_train=X_train,
            X_test=X_test,
            y_train=y_train[:, i],
            y_test=y_test[:, i])
        rmses.append(rmse)
        nrmse_s.append(nrmse)
    print("Mean RMSE:", np.mean(rmses))
    print("Mean NRMSE:", np.mean(nrmse_s))


In [ ]:
"""(outdated) Evaluate downsampling methods"""
df_train, df_test = train_test_split(df, test_size=0.2, shuffle=False)
rmse, nrmse       = TrainerAndEvaluator.train_and_evaluate_model(df_train, df_test)
print(f"Original train RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

N_VW   = 10

N_LTTB = 2
spike_method      = 'zscore' # 'zscore' or 'mad'
spike_method_thres= 3

rdp_epsilon       = 1
subsampling_ratio = 100
N_last_rows       = 100

lttb_union_train = LTTBDownsampler.downsample_using_lttb_union(df_train, time_col=time_col_name,
                                                               max_points_per_col=len(df_train) // N_LTTB)
rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(lttb_union_train, df_test)
print(f"LTTB union ({N_LTTB=})")
print(f"data compressed {len(df_train)/len(lttb_union_train):.2f}x")
print(f"compress. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

lttb_intersect_train = LTTBDownsampler.downsample_using_lttb_intersection(df_train, time_col=time_col_name,
                                                                          max_points_per_col=len(df_train) // N_LTTB)
rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(lttb_intersect_train, df_test)
print("LTTB intersection")
print(f"data compressed {len(df_train)/len(lttb_intersect_train):.2f}x")
print(f"compress. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

lttb_union_spikes_train = LTTBDownsampler.downsample_lttb_union_with_spikes(df_train, time_col=time_col_name,
                                                                            max_points_per_col=len(df_train) // N_LTTB,
                                                                            spike_thresh=spike_method_thres, spike_method=spike_method)
rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(lttb_union_spikes_train, df_test)
print(f"LTTB union +spikes ({spike_method=})")
print(f"data compressed {len(df_train)/len(lttb_union_spikes_train):.2f}x")
print(f"compress. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

rdp_union_train = OtherDownsamplers.downsample_using_rdp_union(df_train, time_col=time_col_name, epsilon=rdp_epsilon)
rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(rdp_union_train, df_test)
print(f"RDP union ({rdp_epsilon=})")
print(f"data compressed {len(df_train)/len(rdp_union_train):.2f}x")
print(f"compress. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

vw_union_train = OtherDownsamplers.downsample_using_vw_union(df_train, time_col=time_col_name, target_points=len(df_train) // N_VW)
rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(vw_union_train, df_test)
print(f"VW union ({N_VW=})")
print(f"data compressed {len(df_train)/len(vw_union_train):.2f}x")
print(f"compress. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

subsample_train = OtherDownsamplers.subsample_timeseries(df_train, subsampling_ratio=subsampling_ratio)
rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(subsample_train, df_test)
print(f"Subsample ({subsampling_ratio=})")
print(f"data compressed {len(df_train)/len(subsample_train):.2f}x")
print(f"compress. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

last_N_rows_train = OtherDownsamplers.get_last_N_rows(df_train, N_rows=N_last_rows)
rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(last_N_rows_train, df_test)
print(f"Last N rows ({N_last_rows=})")
print(f"data compressed {len(df_train)/len(last_N_rows_train):.2f}x")
print(f"compress. RMSE: {rmse:.3f}, norm RMSE: {nrmse:.3%}")

# df_plotter(df, lttb_union_train, 1, time_col_name)

In [ ]:
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf

def plot_corr_matrix(df: pd.DataFrame, title: str = "Feature Correlation Matrix") -> None:
    """Plot heatmap of Pearson correlation matrix for the DataFrame features"""
    corr = df.corr()
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True, cbar_kws={"shrink": .8})
    plt.title(title)
    plt.tight_layout()
    plt.show()

def plot_autocorr(series: pd.Series, lags: int = 40, title: str = None) -> None:
    """Plot autocorrelation function (ACF) for a pandas Series"""
    plt.figure(figsize=(8, 4))
    plot_acf(series.dropna(), lags=lags, alpha=0.05)
    plt.title(title or f"Autocorrelation (up to {lags} lags)")
    plt.tight_layout()
    plt.show()

# plot_corr_matrix(df)
# plot_corr_matrix(df_downsampled)
# plot_corr_matrix(df_downsampled2)

plot_autocorr(df['act_still'], lags=10)
plot_autocorr(df_downsampled['act_still'], lags=10)
plot_autocorr(df_downsampled2['act_still'], lags=10)

In [ ]:
"""LSTM"""
# ===== Example usage =====
n_samples  = 100
n_rows     = 10
n_features = 15
X = torch.randn(n_samples, n_rows, n_features)
y = torch.randn(n_samples, 1)

# Train/val split
train_size = 80
train_X, val_X = X[:train_size], X[train_size:]
train_y, val_y = y[:train_size], y[train_size:]

# Model/optimizer/loss
model     = LSTMModel(input_size=n_features, hidden_size=64, num_layers=2, output_size=1)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Train
trainer = LSTMTrainer(model, optimizer, criterion)
trainer.fit(train_X, train_y, val_X, val_y, epochs=300)

# Inference
new_seq    = torch.randn(1, n_rows, n_features)  # batch=1, seq=10, features=15
prediction = trainer.predict(new_seq)
print("Prediction:", prediction)


In [ ]:
"""Seq2seq LSTM"""

import torch
import torch.nn as nn

class Seq2SeqLSTM(nn.Module):
    """
    Encoder-decoder LSTM for multi-step forecasting.
    Auto-regressive: generates one step at a time.
    """
    def __init__(self, input_size, hidden_size, num_layers, output_size, device='cpu'):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.device = device

        # Encoder LSTM
        self.encoder = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        # Decoder LSTM (one step at a time)
        self.decoder = nn.LSTM(output_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_input, horizon):
        """
        encoder_input: (batch, seq_len, input_size)
        horizon: number of future steps to predict
        returns: (batch, horizon, output_size)
        """
        batch_size = encoder_input.size(0)
        # Encode
        _, (h, c) = self.encoder(encoder_input)

        # Initialize decoder input as last step of encoder
        decoder_input = encoder_input[:, -1:, :]  # shape: (batch, 1, input_size)
        outputs = []

        for t in range(horizon):
            out, (h, c) = self.decoder(decoder_input, (h, c))
            step_pred = self.fc(out)  # (batch, 1, output_size)
            outputs.append(step_pred)
            decoder_input = step_pred  # feed prediction as next input

        return torch.cat(outputs, dim=1)  # (batch, horizon, output_size)

# toy data
batch_size, seq_len, n_features = 4, 10, 3
horizon = 5
X = torch.randn(batch_size, seq_len, n_features)

model = Seq2SeqLSTM(input_size=n_features, hidden_size=32, num_layers=1, output_size=n_features)
preds = model(X, horizon)  # shape: (batch, horizon, n_features)
print(preds.shape)


In [ ]:
"""Make scatterplot of compressions vs rmse"""
compression_ratio_list, rmse_list, nrmse_list = [], [], []
new_points_per_col_ratio = list(range(2, 10, 8))
new_points_per_col = [int(len(df) / ratio) for ratio in new_points_per_col_ratio]

for i, point_n in enumerate(new_points_per_col):
    compression_ratio_list, rmse_list, nrmse_list = downsample_and_evaluate(
        df, point_n, compression_ratio_list, rmse_list, nrmse_list, rdp_epsilon=0.1, method="union_rdp")

rmse, nrmse = TrainerAndEvaluator.train_and_evaluate_model(df)

plt.figure(figsize=(8, 5))
# plt.scatter(compression_ratio_list, nrmse_list, label="Norm. RMSE", marker="o")
# plt.scatter(1, nrmse, label="original")

plt.scatter(compression_ratio_list, rmse_list, label="RMSE", marker="x")
plt.scatter(1, rmse, label="original")

plt.xlabel("Compression Ratio")
plt.ylabel("RMS error")
plt.title("Error vs Compression Ratio")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# """Choose good epsilon for RDP"""
# for col in df.columns:
#     if col == time_col_name:
#         continue
#     if pd.api.types.is_datetime64_any_dtype(df[time_col_name]):
#         x_vals = df[time_col_name].astype("int64")
#     else:
#         x_vals = df[time_col_name].values

#     coords = np.column_stack((x_vals, df[col].values))
#     eps1 = OtherDownsamplers.choose_rdp_epsilon(coords, method="mad", factor=1.0)
#     eps2 = OtherDownsamplers.choose_rdp_epsilon(coords, method="std", factor=1.0)
#     eps3 = OtherDownsamplers.choose_rdp_epsilon(coords, method="fraction", factor=1e-3)

#     print(f"RDP epsilon (mad): {eps1:.3f}, (std): {eps2:.3f}, (fraction): {eps3:.3f}")

In [ ]:
def plot_time_deltas(df: pd.DataFrame, time_col: str) -> None:
    """Calculate time diff between consecutive datetime entries in the specified column,
    and plot these time deltas"""
    df[time_col]= pd.to_datetime(df[time_col])
    diffs       = df[time_col].diff().dropna()
    plt.plot(diffs.dt.total_seconds())
    plt.ylabel('Time delta (s)')
    plt.title('Time diff Between Consecutive datetimes')
    plt.show()

datasets = {"GDELT_USA_LAB":      {"path": "Time-IMM/GDELT_USA_LAB",       "time_col": "date_time", "y": ["AvgTone"]},
            # "ClusterTrace_707":   {"path": "Time-IMM/ClusterTrace_707",    "time_col": "date_time", "y": ["cpu", "memory"]},
            "EPA_air_los_angeles":{"path": "Time-IMM/EPA_air_los_angeles", "time_col": "date_time", "y": ["ozone"]},
            "FNSPID_EXPE":        {"path": "Time-IMM/FNSPID_EXPE",         "time_col": "date_time", "y": ["adj close"]},
            # "ILINet":             {"path": "Time-IMM/ILINet",              "time_col": "date_time", "y": ["TOTAL PATIENTS"]},
            "repohealth_facebook":{"path": "Time-IMM/repohealth_facebook", "time_col": "date_time", "y": ["new_issues"]},
            "studentlife_fc7337": {"path": "Time-IMM/studentlife_fc7337",  "time_col": "date_time", "y": ["sleep_duration"]},
            "power_consumption":  {"path": "power_consumption",            "time_col": "Datetime",  "y": ["PowerConsumption_Zone1", "PowerConsumption_Zone2", "PowerConsumption_Zone3"]},
            "appliances_energy":  {"path": "appliances_energy",            "time_col": "date",      "y": ["rv1", "rv2"]},
            
            "india_catchment": {"path": "india_catchments"},
            "argoverse":       {"path": "argoverse_forecasting"},
            "weather_bench2":  {"path": "?"},
            "PTB_ECG":         {"path": "ptb-xl-1.0.3"},
            }

desired_dataset= "india_catchment"

df             = pd.read_csv(f"../public_datasets/2D/{datasets[desired_dataset]['path']}.csv")
time_col_name  = datasets[desired_dataset]["time_col"]

"""Pre-processing each dataset"""
try:
    if desired_dataset == "GDELT_USA_LAB":
        df.drop(["record_id"], axis=1, inplace=True)
    elif desired_dataset == "ClusterTrace_707":
        pass
    elif desired_dataset == "EPA_air_los_angeles":
        df.drop(["record_id"], axis=1, inplace=True)
    elif desired_dataset == "FNSPID_EXPE":
        df.drop(["record_id"], axis=1, inplace=True)
    elif desired_dataset == "ILINet":
        df.drop(["record_id"], axis=1, inplace=True)
        df = df.drop_duplicates(subset=['date_time'], keep='first').reset_index(drop=True) #some rows have same time
    elif desired_dataset == "repohealth_facebook":
        df.drop(["record_id"], axis=1, inplace=True)
    elif desired_dataset == "studentlife_fc7337":
        df.drop(["record_id"], axis=1, inplace=True)
    elif desired_dataset == "power_consumption":
        df.drop(["PowerConsumption_Zone2", "PowerConsumption_Zone3"], axis=1, inplace=True)
    elif desired_dataset == "appliances_energy":
        df.drop(["rv2"], axis=1, inplace=True)
except Exception as e:
    pass

y_cols = [col for col in datasets[desired_dataset]["y"] if col in df.columns]
df = df.fillna(0)

plot_time_deltas(df, time_col_name)


In [ ]:
"""Common pre-processing steps"""

# interpolate irregular timestamps first
time_delta = df[time_col_name].diff().dropna()
if time_delta.nunique() == 1: # timestamps uniform, skip PCHIP
    df_uniform = df.copy()
else:
    freq       = ForecastUtils.infer_dominant_freq(df, time_col_name)
    df_uniform = ForecastUtils.apply_pchip_interpolation(df, time_col_name, freq)

horizon  = ForecastUtils.compute_horizon(len(df_uniform), fixed_points=20, pct=0.01)
n_train  = len(df_uniform) - horizon
df_train = df_uniform.iloc[:n_train].copy()
df_test  = df_uniform.iloc[n_train:].copy()

n_windows     = 5    # windows to create 
horizon       = 10   # prediction horizon for each window
horizon_frac  = 0.01 # frac of ENTIRE series length
min_window_len= 1008 # len of smallest window (1008 is timeGPT default)


In [ ]:
"""TimeGPT"""
NIXTLA_API_KEY = 'nixak-TnuMDCHsSM4hajkuXycqZZrNxwtAIoT9O9H7Q8ZwKl2JuJlazRqIPknwJW1AVHX2yB3yfCAwmAogugqQ'
logging.getLogger("nixtla").setLevel(logging.WARNING)

forecaster = TimeGPTForecaster(df_uniform, time_col=time_col_name, y_cols=y_cols, api_key=NIXTLA_API_KEY)

"""single window evaluation"""
print("==== single window evaluation ====")
forecast_dict, _ = forecaster.forecast_timegpt(df_train, df_test, horizon, use_exogenous_cols=True)
forecast_df_temp = forecaster.forecast_dfs[y_cols[0]]

"""Multi-window evaluation"""
print("==== multi window evaluation ====")
use_exogenous_cols=True
windows_list  = ForecastUtils.make_windows(df_uniform, n_windows=n_windows, horizon_len=horizon, 
                                           horizon_frac=horizon_frac, min_window_len=min_window_len, start_point=0)
all_forecasts, mae_list, sizes = {}, [], []

for i, (train_df, test_df) in enumerate(windows_list):
    print(f"Window {i}: train shape {train_df.shape}, test shape {test_df.shape}")
    forecast_dict, y_true_scaled = forecaster.forecast_timegpt(train_df, test_df, horizon_len=horizon, use_exogenous_cols=use_exogenous_cols)
    all_forecasts[f"window_{i}"] = forecast_dict

    # Compute MAE per window
    for j, target in enumerate(y_cols):
        y_true = y_true_scaled[:, j] # shape = H
        y_pred = forecast_dict[target]
        mae    = mean_absolute_error(y_true, y_pred)
        mae_list.append(mae)
        sizes.append(len(y_true))

# Step 3: compute weighted MAE across windows (horizon = weight)
weighted_mae = sum(MAE * n for MAE, n in zip(mae_list, sizes)) / sum(sizes)
std_mae      = math.sqrt(sum((MAE - weighted_mae) ** 2 * n for MAE, n in zip(mae_list, sizes)) / sum(sizes))
print(f"Final MAE: mean ± std= {{{weighted_mae:.4f}}}{{{std_mae:.4f}}}")

# # Plot:
# client.plot(df, forecast_df, time_col=time_col_name, target_col=y_cols[0], level=[80,90])

# show last 2% of the history + all predictions
# n = int(len(df) * 0.02)
# df_tail = df.tail(n)
# forecast_tail = forecast_df[forecast_df[time_col_name] >= df_tail[time_col_name].iloc[0]]
# client.plot(df_tail,forecast_tail,time_col=time_col_name,target_col=y_cols[0],level=[80, 90])


In [ ]:
"""SARIMAX"""
logging.basicConfig(level=logging.INFO)

forecaster = SARIMAXForecaster(df_uniform, time_col=time_col_name, y_cols=y_cols)

print("==== single window evaluation ====")
forecast_dict, y_true_scaled = forecaster.forecast_sarimax(df_train, df_test, horizon, 
                                                           order=(1, 0, 0), seasonal_order=(0, 0, 0, 0))
forecast_df_temp = forecaster.forecast_dfs[y_cols[0]]

print("==== multi window evaluation ====")
use_exogenous_cols = False
windows_list = ForecastUtils.make_windows(df_uniform, n_windows=n_windows, horizon_len=horizon, 
                                          horizon_frac=horizon_frac, min_window_len=min_window_len, start_point=0)
all_forecasts, mae_list, sizes = {}, [], []

for i, (train_df, test_df) in enumerate(windows_list):
    print(f"Window {i}: train shape {train_df.shape}, test shape {test_df.shape}")
    forecast_dict, y_true_scaled = forecaster.forecast_sarimax(train_df, test_df, horizon, order=(1, 0, 0),
                                                               seasonal_order=(0, 0, 0, 0), use_exogenous_cols=use_exogenous_cols)
    all_forecasts[f"window_{i}"] = forecast_dict

    # Compute MAE per window
    for j, target in enumerate(y_cols):
        y_true = y_true_scaled[:, j]
        y_pred = forecast_dict[target]
        mae    = mean_absolute_error(y_true, y_pred)
        mae_list.append(mae)
        sizes.append(len(y_true))

# ---- Weighted MAE ----
weighted_mae = sum(MAE * n for MAE, n in zip(mae_list, sizes)) / sum(sizes)
std_mae      = math.sqrt(sum((MAE - weighted_mae) ** 2 * n for MAE, n in zip(mae_list, sizes)) / sum(sizes))
print(f"Final MAE: mean ± std= {{{weighted_mae:.4f}}}{{{std_mae:.4f}}}")


In [ ]:
"""[maybe delete?] run AE"""

# === Example params ===
input_rows        = 1000
input_cols        = 20
layer1_dim        = 64
layer2_dim        = 32
latent_dim        = 8
dropout_prob      = 0.1
layer_dims        = [input_cols, 64, 32, latent_dim]  # for FlexibleAE
pred_dim          = 0
projection_dim    = 0
batch_size        = 32
epochs            = 100
lr                = 1e-3
weight_decay      = 1e-5
scheduler_patience= 5
# === Dummy dataset ===
x_train      = torch.randn(input_rows, input_cols)
train_loader = DataLoader(list(zip(x_train, x_train)), batch_size=batch_size, shuffle=True)

# === Simple AE ===
# simple_ae   = ae.SimpleAutoencoder(input_cols, layer1_dim, layer2_dim, latent_dim, dropout_prob).to(device)
# optimizer = torch.optim.AdamW(simple_ae.parameters(), lr=lr, weight_decay=weight_decay)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=scheduler_patience)
trainer   = train_ae.TrainAutoencoder()
# best_loss = trainer.train_autoencoder(device, simple_ae, epochs, train_loader, optimizer, scheduler)
# print("Best training loss (SimpleAE):", best_loss)

# === FlexibleAE example ===
# flexible_ae = ae.FlexibleAutoencoder(layer_dims, pred_dim, dropout_prob, projection_dim, pred_hidden_dim=32, mode="reconstruct").to(device)
# optimizer = torch.optim.AdamW(flexible_ae.parameters(), lr=lr, weight_decay=weight_decay)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=scheduler_patience)
# best_loss = trainer.train_autoencoder(device, flexible_ae, epochs, train_loader, optimizer, scheduler)
# print(f"Best training loss (FlexibleAE): {best_loss:.3f}")

# === VAE ===
val_loader= DataLoader(list(zip(x_train, x_train)), batch_size=batch_size)  # create validation loader
vae       = ae.VAE(input_cols, hidden_dim=64, latent_dim=latent_dim, dropout_prob=dropout_prob).to(device)
optimizer = torch.optim.AdamW(vae.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=scheduler_patience)
trainer   = train_ae.TrainVAE()
best_loss = trainer.train_vae(device, vae, epochs, train_loader, optimizer, scheduler, val_loader=val_loader)
print(f"Best training loss (VAE): {best_loss:.3f}")

In [ ]:
# Dataset: https://timeseriesclassification.com/description.php?Dataset=BasicMotions
from scipy.io import arff

location = "/Users/fouadabiad/Downloads/BasicMotions/BasicMotions_TRAIN.arff"
data, meta = arff.loadarff(location)
df = pd.DataFrame(data)

df.head(60)


In [ ]:
# Dataset: ?

from scipy.io import arff

dataset = "/Users/fouadabiad/Downloads/wisdm+smartphone+and+smartwatch+activity+and+biometrics+dataset/wisdm-dataset/arff_files/phone/accel/data_1602_accel_phone.arff"
data, meta = arff.loadarff(dataset)
df = pd.DataFrame(data)

df.head(60)
# df.columns

# dump: activity
# predict: resultant

